In [1]:
# ══════════════════════════════════════════════════════════════════════
# SEG CELL 1 — TRAIN Attention U-Net + Dueling head, LEAKAGE-FREE (grouped)
#   Pixel-wise. weighted CE + soft Dice. Saves attn_dueling_unet_grouped_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader

DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True

tr=pd.read_csv(os.path.join(DATA_ROOT,"train_grouped.csv"))
va=pd.read_csv(os.path.join(DATA_ROOT,"val_grouped.csv"))
tr=tr[tr["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
va=va[va["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
print("Seg train "+str(len(tr))+" | val "+str(len(va))+" (patient-grouped)")

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]

class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
        mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0)
        y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)

def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()

ce_w=torch.tensor([1.0,2.0],device=DEVICE)   # weight lesion class higher
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]
    t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); d=(2*inter+1)/(pi.sum()+ti.sum()+1)
            ds.append(d.item())
    return float(np.mean(ds))

opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_grouped_best.pth"))
print("\nSaved attn_dueling_unet_grouped_best.pth (best val-Dice "+format(best,".4f")+")")
print("Run SEG CELL 2 to get the leakage-free TEST Dice.")

Seg train 2867 | val 687 (patient-grouped)
  ep 1/40 val-Dice 0.8296 *
  ep 2/40 val-Dice 0.8493 *
  ep 3/40 val-Dice 0.8456
  ep 4/40 val-Dice 0.8492
  ep 5/40 val-Dice 0.8410
  ep 6/40 val-Dice 0.8469
  ep 7/40 val-Dice 0.8612 *
  ep 8/40 val-Dice 0.8559
  ep 9/40 val-Dice 0.8662 *
  ep 10/40 val-Dice 0.8216
  ep 11/40 val-Dice 0.8634
  ep 12/40 val-Dice 0.8619
  ep 13/40 val-Dice 0.8552
  ep 14/40 val-Dice 0.8533
  ep 15/40 val-Dice 0.8395
  ep 16/40 val-Dice 0.8680 *
  ep 17/40 val-Dice 0.8606
  ep 18/40 val-Dice 0.8464
  ep 19/40 val-Dice 0.8614
  ep 20/40 val-Dice 0.8105
  ep 21/40 val-Dice 0.8661
  ep 22/40 val-Dice 0.8682 *
  ep 23/40 val-Dice 0.8715 *
  ep 24/40 val-Dice 0.8727 *
  ep 25/40 val-Dice 0.8662
  ep 26/40 val-Dice 0.8705
  ep 27/40 val-Dice 0.8691
  ep 28/40 val-Dice 0.8723
  ep 29/40 val-Dice 0.8690
  ep 30/40 val-Dice 0.8729 *
  ep 31/40 val-Dice 0.8712
  ep 32/40 val-Dice 0.8737 *
  ep 33/40 val-Dice 0.8727
  ep 34/40 val-Dice 0.8703
  ep 35/40 val-Dice 0.8733
 

In [2]:
# ══════════════════════════════════════════════════════════════════════
# SEG CELL 2 — TEST segmentation on LEAKAGE-FREE grouped test set
#   Reports per-image Dice + IoU. Loads attn_dueling_unet_grouped_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
te=pd.read_csv(os.path.join(DATA_ROOT,"test_grouped.csv"))
te=te[te["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
print("Seg test "+str(len(te))+" (patient-grouped, leakage-free)")
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
        mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0)
        y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_grouped_best.pth"),map_location=DEVICE)); net.eval()
dices=[]; ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item())
            ious.append(((inter+1)/(union-inter+1)).item())
print("="*56)
print("LEAKAGE-FREE SEGMENTATION TEST (n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f"))
print("  Mean IoU : "+format(np.mean(ious),".4f"))
print("="*56)
print("  Compare to leaked baseline (~0.9078). A similar number confirms")
print("  segmentation was not meaningfully inflated by leakage.")

Seg test 922 (patient-grouped, leakage-free)
LEAKAGE-FREE SEGMENTATION TEST (n=922)
  Mean Dice: 0.8686
  Mean IoU : 0.7801
  Compare to leaked baseline (~0.9078). A similar number confirms
  segmentation was not meaningfully inflated by leakage.


In [3]:
# ══════════════════════════════════════════════════════════════════════
# SEG-MASS CELL 1 — TRAIN Attention U-Net + Dueling head, MASSES ONLY.
#   Leakage-free grouped splits, filtered to abn_type==mass.
#   Saves attn_dueling_unet_massonly_best.pth
#   NOTE: higher Dice expected because masses are easier to segment than
#   scattered calcifications - report with that honest caveat.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    d=d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
    return d
tr=load_mass("train_grouped.csv"); va=load_mass("val_grouped.csv")
print("MASS-ONLY seg | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_massonly_best.pth"))
print("\nSaved attn_dueling_unet_massonly_best.pth (best val-Dice "+format(best,".4f")+")")

MASS-ONLY seg | train 1827 | val 485


KeyboardInterrupt: 

In [4]:
# ══════════════════════════════════════════════════════════════════════
# SEG-MASS CELL 1 — TRAIN Attention U-Net + Dueling head, MASSES ONLY.
#   Leakage-free grouped splits, filtered to abn_type==mass.
#   Saves attn_dueling_unet_massonly_best.pth
#   NOTE: higher Dice expected because masses are easier to segment than
#   scattered calcifications - report with that honest caveat.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    d=d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
    return d
tr=load_mass("train_grouped.csv"); va=load_mass("val_grouped.csv")
print("MASS-ONLY seg | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_massonly_best.pth"))
print("\nSaved attn_dueling_unet_massonly_best.pth (best val-Dice "+format(best,".4f")+")")

MASS-ONLY seg | train 1827 | val 485
  ep 1/40 val-Dice 0.8697 *


KeyboardInterrupt: 

In [5]:
# ── SEG-MASS CELL 2 — TEST masses-only segmentation (leakage-free) ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    d=d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True); return d
te=load_mass("test_grouped.csv")
print("MASS-ONLY seg test: "+str(len(te)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_massonly_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*56)
print("MASS-ONLY SEGMENTATION TEST (leakage-free, n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*56)
print("  All-types leakage-free Dice: 0.869")
print("  Higher here = masses are easier to segment than calcifications (report honestly).")

MASS-ONLY seg test: 617
MASS-ONLY SEGMENTATION TEST (leakage-free, n=617)
  Mean Dice: 0.8807 | Mean IoU: 0.7919
  All-types leakage-free Dice: 0.869
  Higher here = masses are easier to segment than calcifications (report honestly).


In [1]:
# ══════════════════════════════════════════════════════════════════════
# PURE Attention U-Net (NO dueling head) — ablation vs your dueling version.
#   Same params, same preprocessing, same leakage-free mass grouped splits.
#   Only change: standard 2-class conv output head instead of value+advantage.
#   Saves attn_unet_nodueling_massonly_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_mass("train_grouped.csv"); va=load_mass("val_grouped.csv")
print("PURE Attn-UNet (no dueling) | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnUNet(nn.Module):   # PURE: standard output head, NO dueling
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.out=nn.Conv2d(base,2,1)     # <-- standard 2-class head (no value+advantage)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        return self.out(d1)
net=AttnUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_unet_nodueling_massonly_best.pth"))
print("\nSaved attn_unet_nodueling_massonly_best.pth (best val-Dice "+format(best,".4f")+")")
print("Compare to your dueling-head mass-only Dice to show the dueling head's effect (ablation).")

PURE Attn-UNet (no dueling) | train 1827 | val 485
  ep 1/40 val-Dice 0.8736 *
  ep 2/40 val-Dice 0.8674
  ep 3/40 val-Dice 0.8800 *
  ep 4/40 val-Dice 0.8796
  ep 5/40 val-Dice 0.8709
  ep 6/40 val-Dice 0.8747
  ep 7/40 val-Dice 0.8786
  ep 8/40 val-Dice 0.8685
  ep 9/40 val-Dice 0.8806 *
  ep 10/40 val-Dice 0.8813 *
  ep 11/40 val-Dice 0.8814 *
  ep 12/40 val-Dice 0.8821 *
  ep 13/40 val-Dice 0.8820
  ep 14/40 val-Dice 0.8795
  ep 15/40 val-Dice 0.8775
  ep 16/40 val-Dice 0.8837 *
  ep 17/40 val-Dice 0.8774
  ep 18/40 val-Dice 0.8708
  ep 19/40 val-Dice 0.8832
  ep 20/40 val-Dice 0.8845 *
  ep 21/40 val-Dice 0.8831
  ep 22/40 val-Dice 0.8863 *
  ep 23/40 val-Dice 0.8634
  ep 24/40 val-Dice 0.8712
  ep 25/40 val-Dice 0.8783
  ep 26/40 val-Dice 0.8531
  ep 27/40 val-Dice 0.8860
  ep 28/40 val-Dice 0.8873 *
  ep 29/40 val-Dice 0.8858
  ep 30/40 val-Dice 0.8762
  ep 31/40 val-Dice 0.8740
  ep 32/40 val-Dice 0.8901 *
  ep 33/40 val-Dice 0.8882
  ep 34/40 val-Dice 0.8882
  ep 35/40 val-Dic

In [6]:
# ── TEST pure Attention U-Net (no dueling head), mass-only, leakage-free ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_mass("test_grouped.csv")
print("PURE Attn-UNet test (mass-only): "+str(len(te)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.out=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        return self.out(d1)
net=AttnUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_unet_nodueling_massonly_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*58)
print("PURE ATTENTION U-NET (no dueling) — mass-only test (n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*58)
print("  Compare to dueling-head mass-only Dice (your ~0.88) = the dueling head's effect")

PURE Attn-UNet test (mass-only): 617
PURE ATTENTION U-NET (no dueling) — mass-only test (n=617)
  Mean Dice: 0.8799 | Mean IoU: 0.7907
  Compare to dueling-head mass-only Dice (your ~0.88) = the dueling head's effect


In [1]:
# ── Visualize MASS segmentation predictions + confirm mass-only purity ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15

def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_mass("test_grouped.csv")

# ── PURITY CHECK: confirm this set is ONLY mass ──
at=te["abn_type"].astype(str).str.lower()
print("="*55)
print("MASS SET PURITY CHECK")
print("  total rows: "+str(len(te)))
print("  abn_type values: "+str(dict(at.value_counts())))
print("  rows containing 'calc': "+str(at.str.contains("calc").sum())+"  (MUST be 0)")
print("  % with real shape feature: "+format((te['shape_feat'].astype(str).str.upper()!='UNKNOWN').mean()*100,".0f")+"% (mass should be high)")
print("="*55)

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_massonly_best.pth"),map_location=DEVICE)); net.eval()

# visualize 8: image | GT mask | predicted mask | overlay
N=min(8,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): pred=net(x).argmax(1)[0].cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[gt>0]=(0.5*ov[gt>0]+np.array([0,120,0])).astype(np.uint8)      # GT green
        ov[pred>0]=(0.5*ov[pred>0]+np.array([180,0,0])).astype(np.uint8)  # pred red
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title("mass image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted",fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(ov); axes[r,3].set_title("overlay Dice="+format(dice,".2f"),fontsize=8); axes[r,3].axis("off")
plt.suptitle("MASS segmentation (green=GT, red=predicted)",fontsize=13); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","mass_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

MASS SET PURITY CHECK
  total rows: 584
  abn_type values: {'mass': np.int64(584)}
  rows containing 'calc': 0  (MUST be 0)
  % with real shape feature: 60% (mass should be high)
Saved: /root/autodl-tmp/CBIS/figures/mass_segmentation_vis.png


# ── Calcification segmentation: train + purity check (calc-only) ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.17
torch.backends.cudnn.benchmark=True
def load_calc(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("calc")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_calc("train_grouped.csv"); va=load_calc("val_grouped.csv")
# PURITY CHECK
at=tr["abn_type"].astype(str).str.lower()
print("CALC SET PURITY: total "+str(len(tr))+" | contains 'mass': "+str(at.str.contains("mass").sum())+" (MUST be 0)")
print("  % with calctype feature: "+format((tr['calctype_feat'].astype(str).str.upper()!='UNKNOWN').mean()*100,".0f")+"% (calc should be high)")
print("Calc train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,4.0],device=DEVICE)   # higher weight - calc pixels are rare
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_calc_best.pth"))
print("\nSaved attn_dueling_unet_calc_best.pth (best val-Dice "+format(best,".4f")+")")
print("NOTE: calc Dice is usually lower than mass - tiny scattered specks are hard to overlap. Report with that caveat.")

In [7]:
# ── CALCIFICATION segmentation: TEST + visualization (leakage-free) ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.19

def load_calc(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("calc")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_calc("test_grouped.csv")

# PURITY CHECK — confirm calc-only, no mass
at=te["abn_type"].astype(str).str.lower()
print("="*55)
print("CALC TEST SET PURITY")
print("  total: "+str(len(te))+" | contains 'mass': "+str(at.str.contains("mass").sum())+"  (MUST be 0)")
print("  % with calctype feature: "+format((te['calctype_feat'].astype(str).str.upper()!='UNKNOWN').mean()*100,".0f")+"%  (calc should be high)")
print("  % with real shape:       "+format((te['shape_feat'].astype(str).str.upper()!='UNKNOWN').mean()*100,".0f")+"%  (calc should be LOW)")
print("="*55)

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_calc_best.pth"),map_location=DEVICE)); net.eval()

# test metrics
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*55)
print("CALCIFICATION SEGMENTATION TEST (leakage-free, n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*55)
print("  Mass Dice was 0.881. Lower calc Dice is EXPECTED - tiny scattered")
print("  specks are intrinsically hard to overlap. Report with this caveat.")

# visualization
N=min(8,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): pred=net(x).argmax(1)[0].cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[gt>0]=(0.5*ov[gt>0]+np.array([0,120,0])).astype(np.uint8)
        ov[pred>0]=(0.5*ov[pred>0]+np.array([180,0,0])).astype(np.uint8)
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title("calc image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted",fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(ov); axes[r,3].set_title("overlay Dice="+format(dice,".2f"),fontsize=8); axes[r,3].axis("off")
plt.suptitle("CALCIFICATION segmentation (green=GT, red=predicted)",fontsize=13); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","calc_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

CALC TEST SET PURITY
  total: 305 | contains 'mass': 0  (MUST be 0)
  % with calctype feature: 88%  (calc should be high)
  % with real shape:       6%  (calc should be LOW)
CALCIFICATION SEGMENTATION TEST (leakage-free, n=305)
  Mean Dice: 0.8327 | Mean IoU: 0.7396
  Mass Dice was 0.881. Lower calc Dice is EXPECTED - tiny scattered
  specks are intrinsically hard to overlap. Report with this caveat.
Saved: /root/autodl-tmp/CBIS/figures/calc_segmentation_vis.png


In [2]:
# ── DIAGNOSE: what do the raw ROI masks actually look like? (before any fix) ──
import os, cv2, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
DATA_ROOT="/root/autodl-tmp/CBIS"
te=pd.read_csv(os.path.join(DATA_ROOT,"test_grouped.csv"))
te=te[te["source"]=="CBIS"]
mass=te[te["abn_type"].str.lower().str.contains("mass")].reset_index(drop=True)
calc=te[te["abn_type"].str.lower().str.contains("calc")].reset_index(drop=True)

def mask_stats(df,label,n=6):
    print("="*55); print(label+" — raw ROI mask inspection"); print("="*55)
    fracs=[]; empties=0
    fig,axes=plt.subplots(2,n,figsize=(2.4*n,5))
    for i in range(min(n,len(df))):
        row=df.iloc[i*max(1,len(df)//n)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        m=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if m is None: m=np.zeros((256,256),np.uint8)
        binm=(m>127).astype(np.uint8)
        frac=binm.sum()/binm.size if binm.size>0 else 0
        # count connected components (calc should have MANY small; mass ONE big)
        ncomp,_=cv2.connectedComponents(binm)
        axes[0,i].imshow(img if img is not None else np.zeros((256,256)),cmap="gray"); axes[0,i].axis("off"); axes[0,i].set_title("img",fontsize=7)
        axes[1,i].imshow(binm,cmap="gray"); axes[1,i].axis("off"); axes[1,i].set_title("mask frac="+format(frac,".3f")+"\ncomps="+str(ncomp-1),fontsize=7)
    # aggregate stats over up to 100
    for i in range(min(100,len(df))):
        row=df.iloc[i]
        m=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if m is None: empties+=1; continue
        binm=(m>127).astype(np.uint8)
        if binm.sum()<5: empties+=1
        else: fracs.append(binm.sum()/binm.size)
    print("  sampled "+str(min(100,len(df)))+" | empty/missing masks: "+str(empties))
    print("  median mask area fraction: "+format(np.median(fracs) if fracs else 0,".4f"))
    print("  (mass: expect one blob ~0.1-0.4 | calc: expect TINY specks <0.02 with MANY components)")
    plt.suptitle(label+" masks",fontsize=11); plt.tight_layout()
    out=os.path.join(DATA_ROOT,"figures","raw_masks_"+label.lower()+".png"); os.makedirs(os.path.dirname(out),exist_ok=True)
    plt.savefig(out,dpi=130,bbox_inches="tight"); plt.close(); print("  saved: "+out)

mask_stats(mass,"MASS")
mask_stats(calc,"CALC")
print("\nKEY QUESTION: do CALC masks show many tiny specks (correct) or big blobs (bbox-style)?")
print("The 'components' count answers it: calc should have MANY, mass should have 1.")

MASS — raw ROI mask inspection
  sampled 100 | empty/missing masks: 0
  median mask area fraction: 0.0035
  (mass: expect one blob ~0.1-0.4 | calc: expect TINY specks <0.02 with MANY components)
  saved: /root/autodl-tmp/CBIS/figures/raw_masks_mass.png
CALC — raw ROI mask inspection
  sampled 100 | empty/missing masks: 0
  median mask area fraction: 0.0028
  (mass: expect one blob ~0.1-0.4 | calc: expect TINY specks <0.02 with MANY components)
  saved: /root/autodl-tmp/CBIS/figures/raw_masks_calc.png

KEY QUESTION: do CALC masks show many tiny specks (correct) or big blobs (bbox-style)?
The 'components' count answers it: calc should have MANY, mass should have 1.


In [1]:
# ── FIX: robust crop that falls back when predicted mask is empty/tiny ──
# This is the safe crop logic to use when building classifier crops from
# the segmenter. If prediction is empty/tiny, fall back to center crop.
import numpy as np, cv2

def robust_crop_from_pred(img, pred_mask, pad=0.30, min_frac=0.002, out_size=224):
    """
    img: grayscale image (H,W)
    pred_mask: predicted binary mask (H,W), same size as img
    Falls back to center crop if prediction is empty/too small.
    """
    H, W = img.shape
    ys, xs = np.where(pred_mask > 0)
    use_fallback = (len(xs) == 0) or (pred_mask.sum() / pred_mask.size < min_frac)
    if use_fallback:
        # fallback: center square crop (70% of image), never returns black
        s = int(min(H, W) * 0.7); cy, cx = H // 2, W // 2
        y0, y1 = max(0, cy - s // 2), min(H, cy + s // 2)
        x0, x1 = max(0, cx - s // 2), min(W, cx + s // 2)
        tag = "FALLBACK_center"
    else:
        y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
        py, px = int(pad * (y1 - y0 + 1)), int(pad * (x1 - x0 + 1))
        y0, y1 = max(0, y0 - py), min(H, y1 + py + 1)
        x0, x1 = max(0, x0 - px), min(W, x1 + px + 1)
        tag = "lesion_crop"
    crop = img[y0:y1, x0:x1]
    if crop.size == 0: crop = img  # ultimate safety
    crop = cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_LINEAR)
    return crop, tag

# ── audit: how many test crops would hit the fallback? (is this even a real problem?) ──
import os, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, hashlib
# quick check: count empty/tiny predicted masks on the mass test set
# (reuse your trained segmenter to see how often fallback triggers)
DATA_ROOT="/root/autodl-tmp/CBIS"
te=pd.read_csv(os.path.join(DATA_ROOT,"test_grouped.csv"))
te=te[(te["source"]=="CBIS") & te["abn_type"].str.lower().str.contains("mass")].reset_index(drop=True)
# just check GT masks as a proxy for how many are tiny
tiny=0
for _,row in te.iterrows():
    m=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
    if m is None or (m>127).sum()<20: tiny+=1
print("Mass test lesions with empty/tiny masks: "+str(tiny)+"/"+str(len(te)))
print("-> if this is a small number, the fallback guard fully solves it.")
print("-> your crop_cache_attn already built crops; this guard is for any rebuild.")

Mass test lesions with empty/tiny masks: 0/348
-> if this is a small number, the fallback guard fully solves it.
-> your crop_cache_attn already built crops; this guard is for any rebuild.


In [4]:
# ── Empty/tiny mask audit: calcification + INbreast (same check as mass) ──
import os, cv2, pandas as pd
DATA_ROOT="/root/autodl-tmp/CBIS"
te=pd.read_csv(os.path.join(DATA_ROOT,"test_grouped.csv"))

def audit(df, label):
    df=df.reset_index(drop=True)
    tiny=0; missing=0
    for _,row in df.iterrows():
        mp=row.get("roi_mask_jpeg_path","")
        if not isinstance(mp,str) or not os.path.exists(mp):
            missing+=1; continue
        m=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
        if m is None: missing+=1; continue
        if (m>127).sum()<20: tiny+=1
    print(label+": empty/tiny masks "+str(tiny)+"/"+str(len(df))+" | missing files "+str(missing))

# Calcification (CBIS)
calc=te[(te["source"]=="CBIS") & te["abn_type"].str.lower().str.contains("calc")]
audit(calc, "CBIS calcification test")

# INbreast (all)
inb=te[te["source"].astype(str).str.lower().str.contains("inbreast")]
audit(inb, "INbreast test")

print("-"*55)
print("If counts are 0 (or very small), the crop pipeline is safe for these too.")
print("A few tiny/missing = handled by the fallback-crop guard; not a real problem.")

CBIS calcification test: empty/tiny masks 0/304 | missing files 0
INbreast test: empty/tiny masks 0/236 | missing files 0
-------------------------------------------------------
If counts are 0 (or very small), the crop pipeline is safe for these too.
A few tiny/missing = handled by the fallback-crop guard; not a real problem.


In [2]:
# ══════════════════════════════════════════════════════════════════════
# INBREAST segmentation — TRAIN Attention U-Net + dueling head.
#   Leakage-free grouped splits, filtered to source==INbreast.
#   Saves attn_dueling_unet_inbreast_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_inb(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["source"].astype(str).str.lower().str.contains("inbreast")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_inb("train_grouped.csv"); va=load_inb("val_grouped.csv")
print("INbreast seg | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_inbreast_best.pth"))
print("\nSaved attn_dueling_unet_inbreast_best.pth (best val-Dice "+format(best,".4f")+")")

INbreast seg | train 759 | val 205
  ep 1/40 val-Dice 0.6335 *
  ep 2/40 val-Dice 0.8804 *
  ep 3/40 val-Dice 0.8784
  ep 4/40 val-Dice 0.8934 *
  ep 5/40 val-Dice 0.8832
  ep 6/40 val-Dice 0.8689
  ep 7/40 val-Dice 0.8856
  ep 8/40 val-Dice 0.9018 *
  ep 9/40 val-Dice 0.8919
  ep 10/40 val-Dice 0.8992
  ep 11/40 val-Dice 0.8922
  ep 12/40 val-Dice 0.8924
  ep 13/40 val-Dice 0.9022 *
  ep 14/40 val-Dice 0.8951
  ep 15/40 val-Dice 0.8938
  ep 16/40 val-Dice 0.9043 *
  ep 17/40 val-Dice 0.8958
  ep 18/40 val-Dice 0.8928
  ep 19/40 val-Dice 0.8858
  ep 20/40 val-Dice 0.8939
  ep 21/40 val-Dice 0.8890
  ep 22/40 val-Dice 0.9039
  ep 23/40 val-Dice 0.9030
  ep 24/40 val-Dice 0.8995
  early stop

Saved attn_dueling_unet_inbreast_best.pth (best val-Dice 0.9043)


In [6]:
# ══════════════════════════════════════════════════════════════════════
# INBREAST segmentation — TEST + visualization (leakage-free)
#   Loads attn_dueling_unet_inbreast_best.pth. Reports Dice + IoU,
#   saves inbreast_segmentation_vis.png
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.30
def load_inb(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["source"].astype(str).str.lower().str.contains("inbreast")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_inb("test_grouped.csv")

# purity check — confirm INbreast only
src=te["source"].astype(str).str.lower()
print("="*55)
print("INBREAST TEST SET")
print("  total: "+str(len(te))+" | non-INbreast rows: "+str((~src.str.contains("inbreast")).sum())+"  (MUST be 0)")
print("="*55)

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_inbreast_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*55)
print("INBREAST SEGMENTATION TEST (leakage-free, n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*55)
print("  CBIS mass Dice: 0.881 | all-types: 0.869 | calc: 0.842")

# visualization
N=min(8,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): pred=net(x).argmax(1)[0].cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[gt>0]=(0.5*ov[gt>0]+np.array([0,120,0])).astype(np.uint8)
        ov[pred>0]=(0.5*ov[pred>0]+np.array([180,0,0])).astype(np.uint8)
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title("INbreast image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted",fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(ov); axes[r,3].set_title("overlay Dice="+format(dice,".2f"),fontsize=8); axes[r,3].axis("off")
plt.suptitle("INbreast segmentation (green=GT, red=predicted)",fontsize=13); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","inbreast_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

INBREAST TEST SET
  total: 236 | non-INbreast rows: 0  (MUST be 0)
INBREAST SEGMENTATION TEST (leakage-free, n=236)
  Mean Dice: 0.8959 | Mean IoU: 0.8189
  CBIS mass Dice: 0.881 | all-types: 0.869 | calc: 0.842
Saved: /root/autodl-tmp/CBIS/figures/inbreast_segmentation_vis.png


In [2]:
pip install timm numpy pandas scikit-learn tqdm pyyaml albumentations opencv-python-headless mambapy -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip config set global.index-url https://pypi.tuna.tsinghua.edu.cn/simple

Writing to /root/.config/pip/pip.conf
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ── Show the crop cache folders and how they link to the CSVs ──
import os, hashlib, pandas as pd
DATA_ROOT="/root/autodl-tmp/CBIS"
for folder in ["crop_cache_attn","crop_cache_tight","crop_cache_pad060"]:
    p=os.path.join(DATA_ROOT,folder)
    if os.path.exists(p):
        files=os.listdir(p)
        print(folder+": "+str(len(files))+" files | example: "+(files[0] if files else "empty"))
    else:
        print(folder+": (does not exist)")

print("\n── Verify the CSV->cache link for one lesion ──")
d=pd.read_csv(os.path.join(DATA_ROOT,"train_grouped.csv"))
row=d.iloc[0]
h=hashlib.md5(str(row["cropped_jpeg_path"]).encode()).hexdigest()
crop=os.path.join(DATA_ROOT,"crop_cache_attn",h+".png")
print("CSV row original path: "+str(row["cropped_jpeg_path"])[:70]+"...")
print("Maps to cached crop  : crop_cache_attn/"+h+".png")
print("That crop exists     : "+str(os.path.exists(crop)))

crop_cache_attn: 4442 files | example: 6ddb05c9391947d0f459cdbe472865f7.png
crop_cache_tight: 8884 files | example: 6ddb05c9391947d0f459cdbe472865f7_img.png
crop_cache_pad060: 3242 files | example: 6ddb05c9391947d0f459cdbe472865f7.png

── Verify the CSV->cache link for one lesion ──
CSV row original path: /root/autodl-tmp/CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.312534744711605661...
Maps to cached crop  : crop_cache_attn/6ddb05c9391947d0f459cdbe472865f7.png
That crop exists     : True


In [2]:
# ══════════════════════════════════════════════════════════════════════
# COMBINED segmentation — mass + calc + INbreast, ALL together.
#   NO preprocessing (raw crop -> [0,1] only). Leakage-free grouped splits.
#   Attention U-Net + dueling head. Saves attn_dueling_unet_combined_nopre_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_all(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)   # ALL types, both sources
tr=load_all("train_grouped.csv"); va=load_all("val_grouped.csv")
print("COMBINED seg (no preprocess) | train "+str(len(tr))+" | val "+str(len(va)))
print("  train sources: "+str(dict(tr['source'].value_counts()))+" | types: "+str(dict(tr['abn_type'].str.lower().value_counts())))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        # NO preprocessing: raw grayscale -> [0,1], nothing else
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,3.0],device=DEVICE)   # mixed mass+calc, moderate lesion weight
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_combined_nopre_best.pth"))
print("\nSaved attn_dueling_unet_combined_nopre_best.pth (best val-Dice "+format(best,".4f")+")")

COMBINED seg (no preprocess) | train 2867 | val 687
  train sources: {'CBIS': np.int64(2108), 'INbreast': np.int64(759)} | types: {'mass': np.int64(1827), 'calcification': np.int64(1040)}
  ep 1/40 val-Dice 0.8389 *
  ep 2/40 val-Dice 0.8478 *
  ep 3/40 val-Dice 0.8405
  ep 4/40 val-Dice 0.8474
  ep 5/40 val-Dice 0.8525 *
  ep 6/40 val-Dice 0.8399
  ep 7/40 val-Dice 0.8475
  ep 8/40 val-Dice 0.6642
  ep 9/40 val-Dice 0.8474
  ep 10/40 val-Dice 0.8609 *
  ep 11/40 val-Dice 0.8509
  ep 12/40 val-Dice 0.8457
  ep 13/40 val-Dice 0.8295
  ep 14/40 val-Dice 0.8256
  ep 15/40 val-Dice 0.8580
  ep 16/40 val-Dice 0.8582
  ep 17/40 val-Dice 0.8475
  ep 18/40 val-Dice 0.8651 *
  ep 19/40 val-Dice 0.8552
  ep 20/40 val-Dice 0.8223
  ep 21/40 val-Dice 0.8492
  ep 22/40 val-Dice 0.8607
  ep 23/40 val-Dice 0.8622
  ep 24/40 val-Dice 0.8669 *
  ep 25/40 val-Dice 0.8688 *
  ep 26/40 val-Dice 0.8698 *
  ep 27/40 val-Dice 0.8669
  ep 28/40 val-Dice 0.8682
  ep 29/40 val-Dice 0.8703 *
  ep 30/40 val-Dice 

In [3]:
# ══════════════════════════════════════════════════════════════════════
# COMBINED segmentation (no preprocess) — TEST + visualization.
#   All types + both sources. Loads attn_dueling_unet_combined_nopre_best.pth
#   Reports overall Dice/IoU AND a per-group breakdown (mass/calc/INbreast).
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_all(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_all("test_grouped.csv")
print("COMBINED seg test (no preprocess) | n="+str(len(te)))
print("  sources: "+str(dict(te['source'].value_counts()))+" | types: "+str(dict(te['abn_type'].str.lower().value_counts())))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y,idx
te_ds=SegDS(te); test_ld=DataLoader(te_ds,batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_combined_nopre_best.pth"),map_location=DEVICE)); net.eval()

# collect per-sample dice with group tags
per=[]
with torch.no_grad():
    for x,y,idxs in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dice=((2*inter+1)/(union+1)).item(); iou=((inter+1)/(union-inter+1)).item()
            row=te.iloc[int(idxs[i])]
            grp = "INbreast" if "inbreast" in str(row["source"]).lower() else ("CBIS-mass" if "mass" in str(row["abn_type"]).lower() else "CBIS-calc")
            per.append((grp,dice,iou))
per=pd.DataFrame(per,columns=["group","dice","iou"])
print("="*58)
print("COMBINED (no preprocess) SEGMENTATION TEST")
print("  OVERALL Dice: "+format(per['dice'].mean(),".4f")+" | IoU: "+format(per['iou'].mean(),".4f")+" (n="+str(len(per))+")")
print("  ── per-group breakdown ──")
for g,sub in per.groupby("group"):
    print("    "+g.ljust(12)+" Dice "+format(sub['dice'].mean(),".4f")+" | IoU "+format(sub['iou'].mean(),".4f")+" | n="+str(len(sub)))
print("="*58)
print("  Compare to WITH-preprocessing: mass 0.881 | calc 0.842 | all-types 0.869")
print("  (This no-preprocess run is the ablation showing the RL preprocessing effect.)")

# visualization
N=min(9,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): pred=net(x).argmax(1)[0].cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        grp = "INb" if "inbreast" in str(row["source"]).lower() else ("mass" if "mass" in str(row["abn_type"]).lower() else "calc")
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[gt>0]=(0.5*ov[gt>0]+np.array([0,120,0])).astype(np.uint8)
        ov[pred>0]=(0.5*ov[pred>0]+np.array([180,0,0])).astype(np.uint8)
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title(grp+" image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted",fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(ov); axes[r,3].set_title("overlay Dice="+format(dice,".2f"),fontsize=8); axes[r,3].axis("off")
plt.suptitle("Combined segmentation, no preprocessing (green=GT, red=predicted)",fontsize=13); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","combined_nopre_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

COMBINED seg test (no preprocess) | n=922
  sources: {'CBIS': np.int64(686), 'INbreast': np.int64(236)} | types: {'mass': np.int64(617), 'calcification': np.int64(305)}
COMBINED (no preprocess) SEGMENTATION TEST
  OVERALL Dice: 0.8650 | IoU: 0.7752 (n=922)
  ── per-group breakdown ──
    CBIS-calc    Dice 0.8493 | IoU 0.7658 | n=305
    CBIS-mass    Dice 0.8591 | IoU 0.7572 | n=381
    INbreast     Dice 0.8948 | IoU 0.8165 | n=236
  Compare to WITH-preprocessing: mass 0.881 | calc 0.842 | all-types 0.869
  (This no-preprocess run is the ablation showing the RL preprocessing effect.)
Saved: /root/autodl-tmp/CBIS/figures/combined_nopre_segmentation_vis.png


In [4]:
# ══════════════════════════════════════════════════════════════════════
# PLAIN Attention U-Net (NO dueling) — combined mass+calc+INbreast.
#   Standard 2-class head. Leakage-free grouped splits, all types/sources.
#   Saves attn_unet_combined_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_all(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_all("train_grouped.csv"); va=load_all("val_grouped.csv")
print("PLAIN Attn-UNet combined | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnUNet(nn.Module):   # PLAIN: standard 2-class head, NO dueling
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.out=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        return self.out(d1)
net=AttnUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,3.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_unet_combined_best.pth"))
print("\nSaved attn_unet_combined_best.pth (best val-Dice "+format(best,".4f")+")")

PLAIN Attn-UNet combined | train 2867 | val 687
  ep 1/40 val-Dice 0.8482 *
  ep 2/40 val-Dice 0.8426
  ep 3/40 val-Dice 0.8314
  ep 4/40 val-Dice 0.8474
  ep 5/40 val-Dice 0.8323
  ep 6/40 val-Dice 0.8521 *
  ep 7/40 val-Dice 0.8498
  ep 8/40 val-Dice 0.8162
  ep 9/40 val-Dice 0.8513
  ep 10/40 val-Dice 0.7817
  ep 11/40 val-Dice 0.8541 *
  ep 12/40 val-Dice 0.8559 *
  ep 13/40 val-Dice 0.8389
  ep 14/40 val-Dice 0.8522
  ep 15/40 val-Dice 0.8691 *
  ep 16/40 val-Dice 0.8572
  ep 17/40 val-Dice 0.8507
  ep 18/40 val-Dice 0.8312
  ep 19/40 val-Dice 0.8611
  ep 20/40 val-Dice 0.8576
  ep 21/40 val-Dice 0.8682
  ep 22/40 val-Dice 0.8629
  ep 23/40 val-Dice 0.8519
  early stop

Saved attn_unet_combined_best.pth (best val-Dice 0.8691)


In [5]:
# ══════════════════════════════════════════════════════════════════════
# PLAIN Attention U-Net (no dueling) — combined TEST + visualization.
#   Loads attn_unet_combined_best.pth. Overall + per-group Dice/IoU.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_all(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_all("test_grouped.csv")
print("PLAIN Attn-UNet combined test | n="+str(len(te)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y,idx
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.out=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        return self.out(d1)
net=AttnUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_unet_combined_best.pth"),map_location=DEVICE)); net.eval()
per=[]
with torch.no_grad():
    for x,y,idxs in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dice=((2*inter+1)/(union+1)).item(); iou=((inter+1)/(union-inter+1)).item()
            row=te.iloc[int(idxs[i])]
            grp = "INbreast" if "inbreast" in str(row["source"]).lower() else ("CBIS-mass" if "mass" in str(row["abn_type"]).lower() else "CBIS-calc")
            per.append((grp,dice,iou))
per=pd.DataFrame(per,columns=["group","dice","iou"])
print("="*58)
print("PLAIN ATTENTION U-NET (no dueling) — combined TEST")
print("  OVERALL Dice: "+format(per['dice'].mean(),".4f")+" | IoU: "+format(per['iou'].mean(),".4f")+" (n="+str(len(per))+")")
print("  ── per-group ──")
for g,sub in per.groupby("group"):
    print("    "+g.ljust(12)+" Dice "+format(sub['dice'].mean(),".4f")+" | IoU "+format(sub['iou'].mean(),".4f")+" | n="+str(len(sub)))
print("="*58)
print("  Dueling combined (no-pre) was: OVERALL 0.865 | mass 0.859 | calc 0.849 | INbreast 0.895")
print("  Close numbers = dueling head is a neutral variant (consistent ablation finding).")

# visualization
N=min(9,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): pred=net(x).argmax(1)[0].cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        grp = "INb" if "inbreast" in str(row["source"]).lower() else ("mass" if "mass" in str(row["abn_type"]).lower() else "calc")
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[gt>0]=(0.5*ov[gt>0]+np.array([0,120,0])).astype(np.uint8)
        ov[pred>0]=(0.5*ov[pred>0]+np.array([180,0,0])).astype(np.uint8)
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title(grp+" image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted",fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(ov); axes[r,3].set_title("overlay Dice="+format(dice,".2f"),fontsize=8); axes[r,3].axis("off")
plt.suptitle("Plain Attention U-Net combined (green=GT, red=predicted)",fontsize=13); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","plain_attnunet_combined_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

PLAIN Attn-UNet combined test | n=922
PLAIN ATTENTION U-NET (no dueling) — combined TEST
  OVERALL Dice: 0.8623 | IoU: 0.7712 (n=922)
  ── per-group ──
    CBIS-calc    Dice 0.8469 | IoU 0.7622 | n=305
    CBIS-mass    Dice 0.8593 | IoU 0.7576 | n=381
    INbreast     Dice 0.8870 | IoU 0.8049 | n=236
  Dueling combined (no-pre) was: OVERALL 0.865 | mass 0.859 | calc 0.849 | INbreast 0.895
  Close numbers = dueling head is a neutral variant (consistent ablation finding).
Saved: /root/autodl-tmp/CBIS/figures/plain_attnunet_combined_vis.png


In [6]:
# ══════════════════════════════════════════════════════════════════════
# NOVELTY: Dueling Attention U-Net + SELF-SUPERVISED auxiliary decoder.
#   Aux decoder reconstructs the input from the value-stream features.
#   Loss = segmentation loss + LAMBDA_RECON * reconstruction(MSE).
#   Mass-only, leakage-free. Saves attn_dueling_ssl_massonly_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=45; BATCH=16; LR=1e-3; PAD=0.15; LAMBDA_RECON=0.3
torch.backends.cudnn.benchmark=True
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_mass("train_grouped.csv"); va=load_mass("val_grouped.csv")
print("SSL Dueling U-Net (mass) | train "+str(len(tr))+" | val "+str(len(va))+" | lambda_recon "+str(LAMBDA_RECON))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        xi=img.astype(np.float32)/255.0
        x=torch.from_numpy(xi).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class SSLDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
        # auxiliary decoder: reconstruct input from the value map (self-supervised)
        self.recon=nn.Sequential(
            nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1)
        seg=V+A-A.mean(dim=1,keepdim=True)
        recon=self.recon(V)          # reconstruct input from value stream
        return seg, recon
net=SSLDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,2.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,_=net(x)
        pred=seg.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            seg,recon=net(x)
            loss=seg_loss(seg,y)+LAMBDA_RECON*F.mse_loss(recon.float(),x.float())
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=9: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_ssl_massonly_best.pth"))
print("\nSaved attn_dueling_ssl_massonly_best.pth (best val-Dice "+format(best,".4f")+")")
print("Compare to plain dueling mass Dice 0.881 to see the SSL auxiliary decoder effect.")

SSL Dueling U-Net (mass) | train 1827 | val 485 | lambda_recon 0.3
  ep 1/45 val-Dice 0.8667 *
  ep 2/45 val-Dice 0.8754 *
  ep 3/45 val-Dice 0.8785 *
  ep 4/45 val-Dice 0.8776
  ep 5/45 val-Dice 0.8800 *
  ep 6/45 val-Dice 0.8735
  ep 7/45 val-Dice 0.8806 *
  ep 8/45 val-Dice 0.8738
  ep 9/45 val-Dice 0.8810 *
  ep 10/45 val-Dice 0.8729
  ep 11/45 val-Dice 0.8681
  ep 12/45 val-Dice 0.8599
  ep 13/45 val-Dice 0.8727
  ep 14/45 val-Dice 0.8793
  ep 15/45 val-Dice 0.8783
  ep 16/45 val-Dice 0.8790
  ep 17/45 val-Dice 0.8824 *
  ep 18/45 val-Dice 0.8729
  ep 19/45 val-Dice 0.7897
  ep 20/45 val-Dice 0.8637
  ep 21/45 val-Dice 0.8402
  ep 22/45 val-Dice 0.8863 *
  ep 23/45 val-Dice 0.8889 *
  ep 24/45 val-Dice 0.8670
  ep 25/45 val-Dice 0.8817
  ep 26/45 val-Dice 0.8839
  ep 27/45 val-Dice 0.8852
  ep 28/45 val-Dice 0.8873
  ep 29/45 val-Dice 0.8878
  ep 30/45 val-Dice 0.8878
  ep 31/45 val-Dice 0.8847
  ep 32/45 val-Dice 0.8897 *
  ep 33/45 val-Dice 0.8839
  ep 34/45 val-Dice 0.8846
  ep

In [7]:
# ── TEST the SSL auxiliary-decoder Dueling U-Net (mass-only, leakage-free) ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_mass(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("mass")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_mass("test_grouped.csv")
print("SSL Dueling U-Net test (mass-only) | n="+str(len(te)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class SSLDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
        self.recon=nn.Sequential(
            nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
            nn.Conv2d(base,1,1),nn.Sigmoid())
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1)
        seg=V+A-A.mean(dim=1,keepdim=True); recon=self.recon(V)
        return seg, recon
net=SSLDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_ssl_massonly_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[]
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,_=net(x)
        pred=seg.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            dices.append(((2*inter+1)/(union+1)).item()); ious.append(((inter+1)/(union-inter+1)).item())
print("="*58)
print("SSL AUX-DECODER DUELING U-NET — mass-only test (n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("="*58)
print("  Baseline dueling mass Dice: 0.8813 | plain attention: 0.8805")
print("  Higher = SSL branch helped (positive novelty). Flat = value stream already sufficient.")

# visualization: image | GT | predicted | reconstruction (the SSL output)
N=min(6,len(te))
fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
if N==1: axes=axes.reshape(1,4)
with torch.no_grad():
    for r in range(N):
        row=te.iloc[r*max(1,len(te)//N)]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,recon=net(x)
        pred=seg.argmax(1)[0].cpu().numpy(); rec=recon[0,0].float().cpu().numpy()
        gt=(mask>127).astype(np.uint8)
        inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
        axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title("image",fontsize=8); axes[r,0].axis("off")
        axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT mask",fontsize=8); axes[r,1].axis("off")
        axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("predicted Dice="+format(dice,".2f"),fontsize=8); axes[r,2].axis("off")
        axes[r,3].imshow(rec,cmap="gray"); axes[r,3].set_title("SSL reconstruction",fontsize=8); axes[r,3].axis("off")
plt.suptitle("SSL aux-decoder: segmentation + reconstruction from value stream",fontsize=12); plt.tight_layout()
out=os.path.join(DATA_ROOT,"figures","ssl_segmentation_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close()
print("Saved: "+out)

SSL Dueling U-Net test (mass-only) | n=617
SSL AUX-DECODER DUELING U-NET — mass-only test (n=617)
  Mean Dice: 0.8806 | Mean IoU: 0.7917
  Baseline dueling mass Dice: 0.8813 | plain attention: 0.8805
  Higher = SSL branch helped (positive novelty). Flat = value stream already sufficient.
Saved: /root/autodl-tmp/CBIS/figures/ssl_segmentation_vis.png


In [8]:
# ══════════════════════════════════════════════════════════════════════
# DUAL self-supervised Dueling Attention U-Net — COMBINED (mass+calc+INbreast).
#   Two aux decoders from the value stream:
#     (1) image reconstruction  (2) coarse lesion-mask prediction
#   Prints loss + pixel acc/sens/spec + Dice each epoch. 65 epochs.
#   Saves attn_dueling_dualssl_combined_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=65; BATCH=16; LR=1e-3; PAD=0.15
LAMBDA_RECON=0.3; LAMBDA_COARSE=0.3; COARSE=32   # coarse mask resolution
torch.backends.cudnn.benchmark=True
def load_all(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_all("train_grouped.csv"); va=load_all("val_grouped.csv")
print("DUAL-SSL combined | train "+str(len(tr))+" | val "+str(len(va))+" | epochs "+str(EPOCHS))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class DualSSLDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
        # aux decoder 1: reconstruct input from value map
        self.recon=nn.Sequential(nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                 nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                 nn.Conv2d(base,1,1),nn.Sigmoid())
        # aux decoder 2: predict coarse lesion mask from value map
        self.coarse=nn.Sequential(nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                  nn.AdaptiveAvgPool2d(COARSE),
                                  nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                  nn.Conv2d(base,1,1))
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1)
        seg=V+A-A.mean(dim=1,keepdim=True)
        recon=self.recon(V); coarse=self.coarse(V)
        return seg, recon, coarse
net=DualSSLDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,3.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def evaluate():
    net.eval(); ds=[]; TP=TN=FP=FN=0
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,_,_=net(x)
        pred=seg.argmax(1)
        TP+=((pred==1)&(y==1)).sum().item(); TN+=((pred==0)&(y==0)).sum().item()
        FP+=((pred==1)&(y==0)).sum().item(); FN+=((pred==0)&(y==1)).sum().item()
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    acc=(TP+TN)/(TP+TN+FP+FN+1e-9); sens=TP/(TP+FN+1e-9); spec=TN/(TN+FP+1e-9)
    return float(np.mean(ds)), acc, sens, spec
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train(); run_loss=0.0; nb=0
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            seg,recon,coarse=net(x)
            coarse_gt=F.adaptive_avg_pool2d(y.float().unsqueeze(1),COARSE)   # coarse target
            loss=seg_loss(seg,y)+LAMBDA_RECON*F.mse_loss(recon.float(),x.float())+LAMBDA_COARSE*F.binary_cross_entropy_with_logits(coarse.float(),coarse_gt.float())
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run_loss+=loss.item(); nb+=1
    d,acc,sens,spec=evaluate(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" loss "+format(run_loss/max(nb,1),".4f")+" | Dice "+format(d,".4f")+" | pAcc "+format(acc,".4f")+" | pSens "+format(sens,".4f")+" | pSpec "+format(spec,".4f")+(" *"if d==best else""))
    if noimp>=10: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_dualssl_combined_best.pth"))
print("\nSaved attn_dueling_dualssl_combined_best.pth (best val-Dice "+format(best,".4f")+")")
print("pAcc/pSens/pSpec are PIXEL-level. Dice is the key segmentation metric.")

DUAL-SSL combined | train 2867 | val 687 | epochs 65
  ep  1/65 loss 0.6913 | Dice 0.8431 | pAcc 0.8499 | pSens 0.9075 | pSpec 0.8020 *
  ep  2/65 loss 0.6372 | Dice 0.8416 | pAcc 0.8479 | pSens 0.9143 | pSpec 0.7927
  ep  3/65 loss 0.6327 | Dice 0.8446 | pAcc 0.8582 | pSens 0.8620 | pSpec 0.8551 *
  ep  4/65 loss 0.6262 | Dice 0.8459 | pAcc 0.8532 | pSens 0.9145 | pSpec 0.8021 *
  ep  5/65 loss 0.6233 | Dice 0.8473 | pAcc 0.8547 | pSens 0.9132 | pSpec 0.8060 *
  ep  6/65 loss 0.5996 | Dice 0.8330 | pAcc 0.8355 | pSens 0.9298 | pSpec 0.7571
  ep  7/65 loss 0.5813 | Dice 0.8326 | pAcc 0.8318 | pSens 0.9545 | pSpec 0.7296
  ep  8/65 loss 0.5488 | Dice 0.8378 | pAcc 0.8349 | pSens 0.9702 | pSpec 0.7222
  ep  9/65 loss 0.5324 | Dice 0.8631 | pAcc 0.8716 | pSens 0.9337 | pSpec 0.8199 *
  ep 10/65 loss 0.5290 | Dice 0.8614 | pAcc 0.8687 | pSens 0.9490 | pSpec 0.8018
  ep 11/65 loss 0.5150 | Dice 0.8557 | pAcc 0.8619 | pSens 0.9503 | pSpec 0.7883
  ep 12/65 loss 0.5130 | Dice 0.8593 | pAcc 0.

In [9]:
# ══════════════════════════════════════════════════════════════════════
# DUAL-SSL combined — TEST + visualization, per-group (mass/calc/INbreast).
#   Loads attn_dueling_dualssl_combined_best.pth. Reports Dice/IoU +
#   pixel acc/sens/spec, overall AND per group. Saves per-group figures.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15; COARSE=32
def load_all(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_all("test_grouped.csv")
def group_of(row):
    if "inbreast" in str(row["source"]).lower(): return "INbreast"
    return "CBIS-mass" if "mass" in str(row["abn_type"]).lower() else "CBIS-calc"
te["grp"]=te.apply(group_of,axis=1)
print("DUAL-SSL combined test | n="+str(len(te))+" | "+str(dict(te['grp'].value_counts())))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y,idx
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class DualSSLDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
        self.recon=nn.Sequential(nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                 nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                 nn.Conv2d(base,1,1),nn.Sigmoid())
        self.coarse=nn.Sequential(nn.Conv2d(1,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                  nn.AdaptiveAvgPool2d(COARSE),
                                  nn.Conv2d(base,base,3,padding=1),nn.BatchNorm2d(base),nn.ReLU(inplace=True),
                                  nn.Conv2d(base,1,1))
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1)
        seg=V+A-A.mean(dim=1,keepdim=True); recon=self.recon(V); coarse=self.coarse(V)
        return seg, recon, coarse
net=DualSSLDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_dualssl_combined_best.pth"),map_location=DEVICE)); net.eval()

rows=[]
with torch.no_grad():
    for x,y,idxs in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): seg,_,_=net(x)
        pred=seg.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i]; ti=y[i]
            tp=((pi==1)&(ti==1)).sum().item(); tn=((pi==0)&(ti==0)).sum().item()
            fp=((pi==1)&(ti==0)).sum().item(); fn=((pi==0)&(ti==1)).sum().item()
            pf=pi.float(); tf=ti.float(); inter=(pf*tf).sum().item(); union=pf.sum().item()+tf.sum().item()
            dice=(2*inter+1)/(union+1); iou=(inter+1)/(union-inter+1)
            rows.append({"grp":te.iloc[int(idxs[i])]["grp"],"dice":dice,"iou":iou,
                         "acc":(tp+tn)/(tp+tn+fp+fn+1e-9),"sens":tp/(tp+fn+1e-9),"spec":tn/(tn+fp+1e-9)})
R=pd.DataFrame(rows)
def line(name,sub):
    print("  "+name.ljust(12)+" Dice "+format(sub['dice'].mean(),".4f")+" | IoU "+format(sub['iou'].mean(),".4f")+
          " | pAcc "+format(sub['acc'].mean(),".4f")+" | pSens "+format(sub['sens'].mean(),".4f")+
          " | pSpec "+format(sub['spec'].mean(),".4f")+" | n="+str(len(sub)))
print("="*90)
print("DUAL-SSL COMBINED SEGMENTATION TEST")
line("OVERALL",R)
print("  ── per-group ──")
for g in ["CBIS-mass","CBIS-calc","INbreast"]:
    sub=R[R["grp"]==g]
    if len(sub)>0: line(g,sub)
print("="*90)
print("  Baselines: dueling mass 0.881 | single-SSL mass 0.881 | combined no-pre 0.865 | INbreast 0.895")

# per-group visualization (one figure per group + saves)
for g in ["CBIS-mass","CBIS-calc","INbreast"]:
    sub=te[te["grp"]==g].reset_index(drop=True)
    if len(sub)==0: continue
    N=min(5,len(sub)); fig,axes=plt.subplots(N,4,figsize=(13,3.2*N))
    if N==1: axes=axes.reshape(1,4)
    with torch.no_grad():
        for r in range(N):
            row=sub.iloc[r*max(1,len(sub)//N)]
            img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
            if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
            if mask is None: mask=np.zeros_like(img)
            if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
            img,mask=crop_to_lesion(img,mask)
            img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
            x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0).unsqueeze(0).to(DEVICE)
            with torch.amp.autocast(device_type="cuda"): seg,recon,_=net(x)
            pred=seg.argmax(1)[0].cpu().numpy(); rec=recon[0,0].float().cpu().numpy()
            gt=(mask>127).astype(np.uint8); inter=(pred*gt).sum(); dice=(2*inter+1)/(pred.sum()+gt.sum()+1)
            axes[r,0].imshow(img,cmap="gray"); axes[r,0].set_title(g+" image",fontsize=8); axes[r,0].axis("off")
            axes[r,1].imshow(gt,cmap="gray"); axes[r,1].set_title("GT",fontsize=8); axes[r,1].axis("off")
            axes[r,2].imshow(pred,cmap="gray"); axes[r,2].set_title("pred Dice="+format(dice,".2f"),fontsize=8); axes[r,2].axis("off")
            axes[r,3].imshow(rec,cmap="gray"); axes[r,3].set_title("reconstruction",fontsize=8); axes[r,3].axis("off")
    plt.suptitle("DUAL-SSL "+g+" (green n/a; cols: image/GT/pred/recon)",fontsize=12); plt.tight_layout()
    out=os.path.join(DATA_ROOT,"figures","dualssl_"+g.replace("-","_")+"_vis.png"); os.makedirs(os.path.dirname(out),exist_ok=True)
    plt.savefig(out,dpi=140,bbox_inches="tight"); plt.close(); print("Saved: "+out)
    

DUAL-SSL combined test | n=922 | {'CBIS-mass': np.int64(381), 'CBIS-calc': np.int64(305), 'INbreast': np.int64(236)}
DUAL-SSL COMBINED SEGMENTATION TEST
  OVERALL      Dice 0.8608 | IoU 0.7686 | pAcc 0.8687 | pSens 0.9522 | pSpec 0.7612 | n=922
  ── per-group ──
  CBIS-mass    Dice 0.8563 | IoU 0.7530 | pAcc 0.8633 | pSens 0.9587 | pSpec 0.7948 | n=381
  CBIS-calc    Dice 0.8473 | IoU 0.7624 | pAcc 0.8528 | pSens 0.9279 | pSpec 0.6537 | n=305
  INbreast     Dice 0.8852 | IoU 0.8019 | pAcc 0.8978 | pSens 0.9730 | pSpec 0.8461 | n=236
  Baselines: dueling mass 0.881 | single-SSL mass 0.881 | combined no-pre 0.865 | INbreast 0.895
Saved: /root/autodl-tmp/CBIS/figures/dualssl_CBIS_mass_vis.png
Saved: /root/autodl-tmp/CBIS/figures/dualssl_CBIS_calc_vis.png
Saved: /root/autodl-tmp/CBIS/figures/dualssl_INbreast_vis.png


In [10]:
# ── Measure: how many calc masks become EMPTY after crop+resize? ──
import os, cv2, numpy as np, pandas as pd
DATA_ROOT="/root/autodl-tmp/CBIS"; IMG_SIZE=256; PAD=0.15
te=pd.read_csv(os.path.join(DATA_ROOT,"test_grouped.csv"))
calc=te[(te["source"]=="CBIS") & te["abn_type"].str.lower().str.contains("calc")].reset_index(drop=True)
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask,False
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], True
empty_raw=0; empty_after=0; ok=0
for _,row in calc.iterrows():
    img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
    if mask is None or (mask>127).sum()<5: empty_raw+=1; continue
    if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
    ci,cm,ok_crop=crop_to_lesion(img,mask)
    cm2=cv2.resize(cm,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)
    if (cm2>127).sum()<5: empty_after+=1
    else: ok+=1
print("Calc test n="+str(len(calc)))
print("  empty in RAW mask:            "+str(empty_raw))
print("  empty AFTER crop+resize:      "+str(empty_after)+"  <-- these get Dice=0, hurting the average")
print("  survived fine:                "+str(ok))
print("\nIf 'empty after' is meaningful (>10), the resize is destroying tiny calc specks.")
print("Fix: use INTER_AREA or dilate the mask slightly before resize so specks survive.")

Calc test n=305
  empty in RAW mask:            0
  empty AFTER crop+resize:      6  <-- these get Dice=0, hurting the average
  survived fine:                299

If 'empty after' is meaningful (>10), the resize is destroying tiny calc specks.
Fix: use INTER_AREA or dilate the mask slightly before resize so specks survive.


In [11]:
# ══════════════════════════════════════════════════════════════════════
# CALC segmentation — FIXED resize (dilate tiny masks so they survive
# downsampling). Leakage-free, calc-only. Saves attn_dueling_unet_calc_fixed_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True
def load_calc(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("calc")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
tr=load_calc("train_grouped.csv"); va=load_calc("val_grouped.csv")
print("CALC (fixed resize) | train "+str(len(tr))+" | val "+str(len(va)))
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def safe_resize_mask(mask, size):
    # FIX: if mask is tiny, dilate before resize so it doesn't vanish
    binm=(mask>127).astype(np.uint8)
    m=cv2.resize(binm,(size,size),interpolation=cv2.INTER_NEAREST)
    if m.sum()<5 and binm.sum()>0:
        # speck was lost - dilate original then resize with area interp
        k=np.ones((5,5),np.uint8); dil=cv2.dilate(binm,k,iterations=2)
        m=(cv2.resize(dil.astype(np.float32),(size,size),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
        if m.sum()<5:  # last resort: place a small blob at the centroid
            ys,xs=np.where(binm>0); cy=int(ys.mean()*size/binm.shape[0]); cx=int(xs.mean()*size/binm.shape[1])
            cv2.circle(m,(cx,cy),4,1,-1)
    return (m*255).astype(np.uint8)
class SegDS(Dataset):
    def __init__(self, df, augment=False):
        self.df=df.reset_index(drop=True); self.augment=augment
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE))
        mask=safe_resize_mask(mask,IMG_SIZE)     # <-- FIX applied
        if self.augment:
            if np.random.rand()<0.5: img=np.fliplr(img).copy(); mask=np.fliplr(mask).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy(); mask=np.flipud(mask).copy()
            if np.random.rand()<0.5:
                k=np.random.randint(1,4); img=np.rot90(img,k).copy(); mask=np.rot90(mask,k).copy()
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
train_ld=DataLoader(SegDS(tr,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
val_ld  =DataLoader(SegDS(va,augment=False),batch_size=BATCH,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE); scaler=torch.amp.GradScaler()
ce_w=torch.tensor([1.0,4.0],device=DEVICE)
def seg_loss(logits,target):
    ce=F.cross_entropy(logits.float(),target,weight=ce_w)
    p=F.softmax(logits.float(),dim=1)[:,1]; t=target.float()
    dice=1-((2*(p*t).sum(dim=(1,2))+1)/((p+t).sum(dim=(1,2))+1)).mean()
    return ce+dice
@torch.no_grad()
def val_dice():
    net.eval(); ds=[]
    for x,y in val_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
    return float(np.mean(ds))
opt=torch.optim.Adam(net.parameters(),lr=LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=0.5)
best,bstate,noimp=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train()
    for x,y in train_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): out=net(x); loss=seg_loss(out,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    d=val_dice(); sched.step(d)
    if d>best: best=d; bstate={k:v.cpu().clone() for k,v in net.state_dict().items()}; noimp=0
    else: noimp+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" val-Dice "+format(d,".4f")+(" *"if d==best else""))
    if noimp>=8: print("  early stop"); break
if bstate: net.load_state_dict({k:v.to(DEVICE) for k,v in bstate.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"attn_dueling_unet_calc_fixed_best.pth"))
print("\nSaved attn_dueling_unet_calc_fixed_best.pth (best val-Dice "+format(best,".4f")+")")
print("Fix: tiny calc masks dilated before resize so they no longer vanish (was 6/305 empty).")

CALC (fixed resize) | train 1040 | val 202
  ep 1/40 val-Dice 0.7585 *
  ep 2/40 val-Dice 0.7838 *
  ep 3/40 val-Dice 0.7892 *
  ep 4/40 val-Dice 0.7737
  ep 5/40 val-Dice 0.7853
  ep 6/40 val-Dice 0.7967 *
  ep 7/40 val-Dice 0.7697
  ep 8/40 val-Dice 0.7910
  ep 9/40 val-Dice 0.8019 *
  ep 10/40 val-Dice 0.8220 *
  ep 11/40 val-Dice 0.8015
  ep 12/40 val-Dice 0.8030
  ep 13/40 val-Dice 0.7979
  ep 14/40 val-Dice 0.8083
  ep 15/40 val-Dice 0.8197
  ep 16/40 val-Dice 0.8234 *
  ep 17/40 val-Dice 0.8269 *
  ep 18/40 val-Dice 0.8266
  ep 19/40 val-Dice 0.8216
  ep 20/40 val-Dice 0.8277 *
  ep 21/40 val-Dice 0.8252
  ep 22/40 val-Dice 0.8133
  ep 23/40 val-Dice 0.8244
  ep 24/40 val-Dice 0.8338 *
  ep 25/40 val-Dice 0.8296
  ep 26/40 val-Dice 0.8261
  ep 27/40 val-Dice 0.8278
  ep 28/40 val-Dice 0.8304
  ep 29/40 val-Dice 0.8155
  ep 30/40 val-Dice 0.8328
  ep 31/40 val-Dice 0.8277
  ep 32/40 val-Dice 0.8276
  early stop

Saved attn_dueling_unet_calc_fixed_best.pth (best val-Dice 0.8338)
F

In [12]:
# ── TEST fixed-resize calc segmentation (leakage-free) ──
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE=256; PAD=0.15
def load_calc(name):
    d=pd.read_csv(os.path.join(DATA_ROOT,name))
    d=d[d["abn_type"].astype(str).str.lower().str.contains("calc")]
    return d[d["roi_mask_jpeg_path"].notna()].reset_index(drop=True)
te=load_calc("test_grouped.csv")
def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def safe_resize_mask(mask, size):
    binm=(mask>127).astype(np.uint8)
    m=cv2.resize(binm,(size,size),interpolation=cv2.INTER_NEAREST)
    if m.sum()<5 and binm.sum()>0:
        k=np.ones((5,5),np.uint8); dil=cv2.dilate(binm,k,iterations=2)
        m=(cv2.resize(dil.astype(np.float32),(size,size),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
        if m.sum()<5:
            ys,xs=np.where(binm>0); cy=int(ys.mean()*size/binm.shape[0]); cx=int(xs.mean()*size/binm.shape[1])
            cv2.circle(m,(cx,cy),4,1,-1)
    return (m*255).astype(np.uint8)
class SegDS(Dataset):
    def __init__(self, df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        img=cv2.imread(row["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE); mask=cv2.imread(row["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG_SIZE,IMG_SIZE),np.uint8)
        if mask is None: mask=np.zeros_like(img)
        if mask.shape!=img.shape: mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,mask=crop_to_lesion(img,mask)
        img=cv2.resize(img,(IMG_SIZE,IMG_SIZE)); mask=safe_resize_mask(mask,IMG_SIZE)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0); y=torch.from_numpy((mask>127).astype(np.int64))
        return x,y
test_ld=DataLoader(SegDS(te),batch_size=16,shuffle=False,num_workers=0)
def cblock(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(inplace=True))
class AttentionGate(nn.Module):
    def __init__(self,g,x,i):
        super().__init__()
        self.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); self.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        self.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); self.relu=nn.ReLU(inplace=True)
    def forward(self,g,x): return x*self.psi(self.relu(self.Wg(g)+self.Wx(x)))
class AttnDuelingUNet(nn.Module):
    def __init__(self,base=32):
        super().__init__()
        self.e1=cblock(1,base);self.e2=cblock(base,base*2);self.e3=cblock(base*2,base*4);self.e4=cblock(base*4,base*8)
        self.pool=nn.MaxPool2d(2);self.bottleneck=cblock(base*8,base*16)
        self.u4=nn.ConvTranspose2d(base*16,base*8,2,stride=2);self.a4=AttentionGate(base*8,base*8,base*4);self.d4=cblock(base*16,base*8)
        self.u3=nn.ConvTranspose2d(base*8,base*4,2,stride=2);self.a3=AttentionGate(base*4,base*4,base*2);self.d3=cblock(base*8,base*4)
        self.u2=nn.ConvTranspose2d(base*4,base*2,2,stride=2);self.a2=AttentionGate(base*2,base*2,base);self.d2=cblock(base*4,base*2)
        self.u1=nn.ConvTranspose2d(base*2,base,2,stride=2);self.a1=AttentionGate(base,base,base//2);self.d1=cblock(base*2,base)
        self.value=nn.Conv2d(base,1,1);self.advantage=nn.Conv2d(base,2,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        b=self.bottleneck(self.pool(e4))
        g4=self.u4(b);d4=self.d4(torch.cat([g4,self.a4(g4,e4)],1))
        g3=self.u3(d4);d3=self.d3(torch.cat([g3,self.a3(g3,e3)],1))
        g2=self.u2(d3);d2=self.d2(torch.cat([g2,self.a2(g2,e2)],1))
        g1=self.u1(d2);d1=self.d1(torch.cat([g1,self.a1(g1,e1)],1))
        V=self.value(d1);A=self.advantage(d1);return V+A-A.mean(dim=1,keepdim=True)
net=AttnDuelingUNet().to(DEVICE)
net.load_state_dict(torch.load(os.path.join(DATA_ROOT,"attn_dueling_unet_calc_fixed_best.pth"),map_location=DEVICE)); net.eval()
dices=[];ious=[];zeros=0
with torch.no_grad():
    for x,y in test_ld:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): out=net(x)
        pred=out.argmax(1)
        for i in range(pred.size(0)):
            pi=pred[i].float(); ti=y[i].float()
            inter=(pi*ti).sum(); union=pi.sum()+ti.sum()
            d=((2*inter+1)/(union+1)).item(); dices.append(d); ious.append(((inter+1)/(union-inter+1)).item())
            if d<0.05: zeros+=1
print("="*56)
print("CALC SEGMENTATION (FIXED resize) TEST (n="+str(len(dices))+")")
print("  Mean Dice: "+format(np.mean(dices),".4f")+" | Mean IoU: "+format(np.mean(ious),".4f"))
print("  near-zero Dice cases: "+str(zeros)+" (was ~6 before fix)")
print("="*56)
print("  Before fix: calc Dice 0.842. Fixing the 6 empty-resize cases should nudge this up.")

CALC SEGMENTATION (FIXED resize) TEST (n=305)
  Mean Dice: 0.8424 | Mean IoU: 0.7551
  near-zero Dice cases: 6 (was ~6 before fix)
  Before fix: calc Dice 0.842. Fixing the 6 empty-resize cases should nudge this up.


In [1]:
# ══════════════════════════════════════════════════════════════════════
# OFFLINE AUGMENTATION — expands TRAINING data only. Val/test untouched.
#   Per training image: original + N augmented variants (flips, rotations,
#   shear, scale, brightness). Writes to aug_cache/ + aug_train.csv
# ══════════════════════════════════════════════════════════════════════
import os, hashlib
import numpy as np, pandas as pd, cv2
DATA_ROOT="/root/autodl-tmp/CBIS"
SRC_CACHE=os.path.join(DATA_ROOT,"crop_cache_attn")     # segmentation-guided crops
AUG_DIR=os.path.join(DATA_ROOT,"aug_cache"); os.makedirs(AUG_DIR,exist_ok=True)
N_AUG=7          # 7 variants + 1 original = 8x expansion
SIZE=224
def src_path(row): return os.path.join(SRC_CACHE, hashlib.md5(str(row["cropped_jpeg_path"]).encode()).hexdigest()+".png")

tr=pd.read_csv(os.path.join(DATA_ROOT,"train_grouped.csv"))
print("Original training rows: "+str(len(tr)))
print("  by type: "+str(dict(tr['abn_type'].str.lower().value_counts())))
print("  by source: "+str(dict(tr['source'].value_counts())))

def augment(img, k):
    a=img.copy()
    if k==1: a=np.fliplr(a)
    elif k==2: a=np.flipud(a)
    elif k==3: a=np.rot90(a,1)
    elif k==4: a=np.rot90(a,2)
    elif k==5: a=np.rot90(a,3)
    elif k==6:
        M=cv2.getRotationMatrix2D((SIZE/2,SIZE/2), np.random.uniform(-20,20), np.random.uniform(0.9,1.1))
        a=cv2.warpAffine(a,M,(SIZE,SIZE),borderMode=cv2.BORDER_REFLECT)
    elif k==7:
        a=np.clip(a.astype(np.float32)*np.random.uniform(0.8,1.2)+np.random.uniform(-20,20),0,255).astype(np.uint8)
    return np.ascontiguousarray(a)

rows=[]; made=0; missing=0
for _,r in tr.iterrows():
    p=src_path(r)
    img=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
    if img is None: missing+=1; continue
    if img.shape!=(SIZE,SIZE): img=cv2.resize(img,(SIZE,SIZE))
    base=hashlib.md5(str(r["cropped_jpeg_path"]).encode()).hexdigest()
    for k in range(N_AUG+1):                      # k=0 is the original
        outp=os.path.join(AUG_DIR, base+"_a"+str(k)+".png")
        if not os.path.exists(outp):
            cv2.imwrite(outp, img if k==0 else augment(img,k))
        nr=r.to_dict(); nr["aug_path"]=outp; nr["aug_id"]=k
        rows.append(nr); made+=1

aug=pd.DataFrame(rows)
aug.to_csv(os.path.join(DATA_ROOT,"aug_train.csv"),index=False)
print("\n"+"="*58)
print("AUGMENTED TRAINING SET")
print("  original: "+str(len(tr))+"  ->  augmented: "+str(len(aug))+"   ("+str(round(len(aug)/max(len(tr),1),1))+"x)")
print("  missing source crops skipped: "+str(missing))
print("  by type: "+str(dict(aug['abn_type'].str.lower().value_counts())))
print("  patients: "+str(aug['patient_id'].nunique())+" (same as before - NO new patients)")
print("="*58)
print("  VAL and TEST are NOT augmented - they stay original. No leakage.")
print("  Saved aug_train.csv")

Original training rows: 2867
  by type: {'mass': np.int64(1827), 'calcification': np.int64(1040)}
  by source: {'CBIS': np.int64(2108), 'INbreast': np.int64(759)}

AUGMENTED TRAINING SET
  original: 2867  ->  augmented: 22936   (8.0x)
  missing source crops skipped: 0
  by type: {'mass': np.int64(14616), 'calcification': np.int64(8320)}
  patients: 1126 (same as before - NO new patients)
  VAL and TEST are NOT augmented - they stay original. No leakage.
  Saved aug_train.csv


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SEGMENTATION with heavy augmentation (image+mask in lockstep).
#   Runs mass / calc / inbreast in one cell. Train aug, val+test ORIGINAL.
#   Prints loss + Dice each epoch. Saves seg_aug_<mode>_best.pth
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=45; BATCH=16; LR=1e-3; PAD=0.15
torch.backends.cudnn.benchmark=True

def sel(df,mode):
    d=df[df["roi_mask_jpeg_path"].notna()]
    if mode=="mass":     d=d[(d["source"]=="CBIS") & d["abn_type"].str.lower().str.contains("mass")]
    elif mode=="calc":   d=d[(d["source"]=="CBIS") & d["abn_type"].str.lower().str.contains("calc")]
    elif mode=="inbreast": d=d[d["source"].str.lower().str.contains("inbreast")]
    return d.reset_index(drop=True)

def crop(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]

def safe_mask_resize(m,size):
    b=(m>127).astype(np.uint8)
    r=cv2.resize(b,(size,size),interpolation=cv2.INTER_NEAREST)
    if r.sum()<5 and b.sum()>0:
        d=cv2.dilate(b,np.ones((5,5),np.uint8),iterations=2)
        r=(cv2.resize(d.astype(np.float32),(size,size),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
    return r

class SegDS(Dataset):
    # AUG_MULT: how many augmented variants per image (train only)
    def __init__(s,df,aug,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i % len(s.df)]; k=i // len(s.df)          # k = which variant
        img=cv2.imread(r["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if msk.shape!=img.shape: msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,msk=crop(img,msk)
        img=cv2.resize(img,(IMG,IMG)); m=safe_mask_resize(msk,IMG)
        if s.aug and k>0:
            # SAME transform applied to image AND mask
            if k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                ang=np.random.uniform(-25,25); sc=np.random.uniform(0.9,1.1)
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),ang,sc)
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(0.8,1.2),0,255).astype(np.uint8)  # mask unchanged
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        x=torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0)
        y=torch.from_numpy(m.astype(np.int64))
        return x,y

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

TR=pd.read_csv(os.path.join(DATA_ROOT,"train_grouped.csv"))
VA=pd.read_csv(os.path.join(DATA_ROOT,"val_grouped.csv"))
TE=pd.read_csv(os.path.join(DATA_ROOT,"test_grouped.csv"))
results=[]
for MODE,MULT,CEW in [("mass",8,2.0),("calc",8,4.0),("inbreast",8,2.0)]:
    tr=sel(TR,MODE); va=sel(VA,MODE); te=sel(TE,MODE)
    if len(tr)==0: continue
    print("\n"+"#"*66); print("### SEG "+MODE+" | train "+str(len(tr))+" x"+str(MULT)+" aug = "+str(len(tr)*MULT)+" | val "+str(len(va))+" | test "+str(len(te))); print("#"*66)
    tl=DataLoader(SegDS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(SegDS(va,False),batch_size=BATCH,shuffle=False,num_workers=0)
    testl=DataLoader(SegDS(te,False),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEVICE); scaler=torch.amp.GradScaler()
    w=torch.tensor([1.0,CEW],device=DEVICE)
    def loss_fn(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=w)
        p=F.softmax(lo.float(),1)[:,1]; tt=t.float()
        dl=1-((2*(p*tt).sum((1,2))+1)/((p+tt).sum((1,2))+1)).mean()
        return ce+dl
    @torch.no_grad()
    def dice_of(loader):
        net.eval(); ds=[]
        for x,y in loader:
            x=x.to(DEVICE);y=y.to(DEVICE)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                ds.append(((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item())
        return float(np.mean(ds))
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=0.5)
    best,bs,ni=0.0,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0;nb=0
        for x,y in tl:
            x=x.to(DEVICE);y=y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        d=dice_of(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+" | val-Dice "+format(d,".4f")+" | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
        if ni>=8: print("  early stop"); break
    if bs: net.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"seg_aug_"+MODE+"_best.pth"))
    td=dice_of(testl)
    results.append((MODE,best,td,len(te)))
    print("  >>> TEST Dice ("+MODE+", original data): "+format(td,".4f"))
print("\n"+"="*66)
print("SEGMENTATION WITH AUGMENTED TRAINING — test on ORIGINAL data")
print("="*66)
for m,b,t,n in results:
    print("  "+m.ljust(10)+" val-Dice "+format(b,".4f")+" | TEST Dice "+format(t,".4f")+" | n="+str(n))
print("="*66)
print("  Baselines (no offline aug): mass 0.881 | calc 0.842 | INbreast 0.895")


##################################################################
### SEG mass | train 1068 x8 aug = 8544 | val 280 | test 381
##################################################################
  ep 1/45 | loss 0.4650 | val-Dice 0.8689 | lr 1.0e-03 *
  ep 2/45 | loss 0.4453 | val-Dice 0.8662 | lr 1.0e-03
  ep 3/45 | loss 0.4447 | val-Dice 0.8710 | lr 1.0e-03 *
  ep 4/45 | loss 0.4429 | val-Dice 0.8702 | lr 1.0e-03
  ep 5/45 | loss 0.4345 | val-Dice 0.8688 | lr 1.0e-03
  ep 6/45 | loss 0.4274 | val-Dice 0.8702 | lr 1.0e-03
  ep 7/45 | loss 0.4226 | val-Dice 0.8642 | lr 5.0e-04
  ep 8/45 | loss 0.4145 | val-Dice 0.8639 | lr 5.0e-04
  ep 9/45 | loss 0.4124 | val-Dice 0.8725 | lr 5.0e-04 *
  ep 10/45 | loss 0.4096 | val-Dice 0.8727 | lr 5.0e-04 *
  ep 11/45 | loss 0.4080 | val-Dice 0.8636 | lr 5.0e-04
  ep 12/45 | loss 0.4055 | val-Dice 0.8732 | lr 5.0e-04 *
  ep 13/45 | loss 0.4039 | val-Dice 0.8738 | lr 5.0e-04 *
  ep 14/45 | loss 0.4020 | val-Dice 0.8670 | lr 5.0e-04


In [2]:
# ══════════════════════════════════════════════════════════════════════
# CDD + CSAW ingestion: pair images with masks, crop to lesion, resize,
#   build csv with train/test split. Matches CBIS crop format (224/256).
# ══════════════════════════════════════════════════════════════════════
import os, glob, re
import numpy as np, pandas as pd, cv2
ROOT="/root/autodl-tmp"
OUT=os.path.join(ROOT,"extra_crops"); os.makedirs(OUT,exist_ok=True)
SEG_SIZE=256; PAD=0.15

SETS={
 "CDD":  {"img":os.path.join(ROOT,"CDD-IMG"),   "msk":os.path.join(ROOT,"CDD-MASKS")},
 "CSAW": {"img":os.path.join(ROOT,"CSAW-IMG"), "msk":os.path.join(ROOT,"CSAW-MASKS")},
}

def find(d):
    f=[]
    for e in ("*.png","*.jpg","*.jpeg","*.PNG","*.JPG"):
        f+=glob.glob(os.path.join(d,"**",e),recursive=True)
    return sorted(f)

def key(p):
    b=os.path.basename(p)
    b=re.sub(r"(?i)[_\-]?mask","",b)          # strip 'mask'
    b=re.sub(r"(?i)___?PRE","",b)             # strip ___PRE suffix
    b=os.path.splitext(b)[0]
    return re.sub(r"[^A-Za-z0-9]","",b).lower()

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return None,None
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return (img[max(0,y0-py):min(h,y1+py+1), max(0,x0-px):min(w,x1+px+1)],
            mask[max(0,y0-py):min(h,y1+py+1), max(0,x0-px):min(w,x1+px+1)])

rows=[]
for name,paths in SETS.items():
    if not (os.path.exists(paths["img"]) and os.path.exists(paths["msk"])):
        print(name+": folders not found — check names"); continue
    imgs=find(paths["img"]); msks=find(paths["msk"])
    mmap={key(m):m for m in msks}
    print("\n"+name+": "+str(len(imgs))+" images, "+str(len(msks))+" masks")
    paired=0; nomask=0; empty=0
    for ip in imgs:
        mp=mmap.get(key(ip))
        if mp is None: nomask+=1; continue
        img=cv2.imread(ip,cv2.IMREAD_GRAYSCALE); msk=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
        if img is None or msk is None: nomask+=1; continue
        if msk.shape!=img.shape:
            msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        if (msk>127).sum()<20: empty+=1; continue
        ci,cm=crop_to_lesion(img,msk)
        if ci is None or ci.size==0: empty+=1; continue
        ci=cv2.resize(ci,(SEG_SIZE,SEG_SIZE))
        cm=cv2.resize((cm>127).astype(np.uint8)*255,(SEG_SIZE,SEG_SIZE),interpolation=cv2.INTER_NEAREST)
        base=name+"_"+os.path.splitext(os.path.basename(ip))[0]
        ipath=os.path.join(OUT,base+"_img.png"); mpath=os.path.join(OUT,base+"_msk.png")
        cv2.imwrite(ipath,ci); cv2.imwrite(mpath,cm)
        rows.append({"source":name,"case_id":base,"cropped_jpeg_path":ipath,
                     "roi_mask_jpeg_path":mpath,"abn_type":"mass"})
        paired+=1
    print("  paired "+str(paired)+" | mask missing "+str(nomask)+" | empty/bad "+str(empty))

df=pd.DataFrame(rows)
if len(df)==0:
    print("\nNo pairs built — check folder names / filename matching.")
else:
    # split by case (no same case in train and test)
    rng=np.random.RandomState(42)
    for src,sub in df.groupby("source"):
        ids=sub["case_id"].unique().tolist(); rng.shuffle(ids)
        n=len(ids); ntr=int(0.8*n)
        tr=set(ids[:ntr])
        df.loc[df["case_id"].isin(tr) & (df["source"]==src),"split"]="train"
        df.loc[~df["case_id"].isin(tr) & (df["source"]==src),"split"]="test"
    df.to_csv(os.path.join(ROOT,"extra_seg.csv"),index=False)
    print("\n"+"="*56)
    print("EXTRA SEG DATA: "+str(len(df))+" pairs")
    print("  by source: "+str(dict(df['source'].value_counts())))
    print("  by split:  "+str(dict(df['split'].value_counts())))
    print("  saved extra_seg.csv | crops in extra_crops/")
    print("="*56)
    print("  NOTE: CDD is contrast-enhanced (CESM) — different modality.")
    print("  Report it separately, not merged into CBIS/INbreast numbers.")


CDD: 12 images, 12 masks
  paired 12 | mask missing 0 | empty/bad 0

CSAW: 152 images, 152 masks
  paired 0 | mask missing 152 | empty/bad 0

EXTRA SEG DATA: 12 pairs
  by source: {'CDD': np.int64(12)}
  by split:  {'train': np.int64(9), 'test': np.int64(3)}
  saved extra_seg.csv | crops in extra_crops/
  NOTE: CDD is contrast-enhanced (CESM) — different modality.
  Report it separately, not merged into CBIS/INbreast numbers.


In [3]:
import os, glob, re
import numpy as np, pandas as pd, cv2
ROOT="/root/autodl-tmp"
OUT=os.path.join(ROOT,"extra_crops"); os.makedirs(OUT,exist_ok=True)
SEG_SIZE=256; PAD=0.15

def num(p):
    m=re.search(r"-(\d+)\.png", os.path.basename(p))
    return m.group(1) if m else None

def crop_to_lesion(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return None,None
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return (img[max(0,y0-py):min(h,y1+py+1), max(0,x0-px):min(w,x1+px+1)],
            mask[max(0,y0-py):min(h,y1+py+1), max(0,x0-px):min(w,x1+px+1)])

img_dir = os.path.join(ROOT,"CSAW -IMG")
if not os.path.exists(img_dir): img_dir = os.path.join(ROOT,"CSAW-IMG")
msk_dir = os.path.join(ROOT,"CSAW-MASKS")

imgs = sorted(glob.glob(os.path.join(img_dir,"**","*.png"),recursive=True))
msks = sorted(glob.glob(os.path.join(msk_dir,"**","*.png"),recursive=True))
mmap = {num(m): m for m in msks if num(m)}
print("images "+str(len(imgs))+" | masks "+str(len(msks)))

rows=[]; paired=0; miss=0; bad=0
for ip in imgs:
    n=num(ip); mp=mmap.get(n)
    if mp is None: miss+=1; continue
    img=cv2.imread(ip,cv2.IMREAD_GRAYSCALE); msk=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None: miss+=1; continue
    if msk.shape!=img.shape:
        msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
    if (msk>127).sum()<20: bad+=1; continue
    ci,cm=crop_to_lesion(img,msk)
    if ci is None or ci.size==0: bad+=1; continue
    ci=cv2.resize(ci,(SEG_SIZE,SEG_SIZE))
    cm=cv2.resize((cm>127).astype(np.uint8)*255,(SEG_SIZE,SEG_SIZE),interpolation=cv2.INTER_NEAREST)
    base="CSAW_"+n
    ipath=os.path.join(OUT,base+"_img.png"); mpath=os.path.join(OUT,base+"_msk.png")
    cv2.imwrite(ipath,ci); cv2.imwrite(mpath,cm)
    rows.append({"source":"CSAW","case_id":base,"cropped_jpeg_path":ipath,
                 "roi_mask_jpeg_path":mpath,"abn_type":"mass"})
    paired+=1
print("paired "+str(paired)+" | missing "+str(miss)+" | bad "+str(bad))

new=pd.DataFrame(rows)
old_p=os.path.join(ROOT,"extra_seg.csv")
if os.path.exists(old_p):
    old=pd.read_csv(old_p)
    old=old[old["source"]!="CSAW"]
    new=pd.concat([old,new],ignore_index=True)
rng=np.random.RandomState(42)
for src,sub in new.groupby("source"):
    ids=sub["case_id"].unique().tolist(); rng.shuffle(ids)
    tr=set(ids[:int(0.8*len(ids))])
    new.loc[(new["source"]==src) & new["case_id"].isin(tr),"split"]="train"
    new.loc[(new["source"]==src) & ~new["case_id"].isin(tr),"split"]="test"
new.to_csv(old_p,index=False)
print("\nextra_seg.csv: "+str(len(new))+" pairs | "+str(dict(new['source'].value_counts()))+" | "+str(dict(new['split'].value_counts())))

images 152 | masks 152
paired 152 | missing 0 | bad 0

extra_seg.csv: 164 pairs | {'CSAW': np.int64(152), 'CDD': np.int64(12)} | {'train': np.int64(130), 'test': np.int64(34)}


In [4]:
# ══════════════════════════════════════════════════════════════════════
# COMBINED SEGMENTATION — CBIS(mass+calc) + INbreast + CSAW + CDD
#   8x augmentation (image+mask in lockstep). Val/test = original only.
#   Prints loss + val-Dice + LR each epoch. Per-source test Dice at end.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; ROOT="/root/autodl-tmp"
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=45; BATCH=16; LR=1e-3; PAD=0.15; MULT=8
torch.backends.cudnn.benchmark=True

# ---- assemble all data ----
def load_cbis(f):
    d=pd.read_csv(os.path.join(DATA_ROOT,f))
    d=d[d["roi_mask_jpeg_path"].notna()]
    return d[["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]]
tr=load_cbis("train_grouped.csv"); va=load_cbis("val_grouped.csv"); te=load_cbis("test_grouped.csv")
extra=pd.read_csv(os.path.join(ROOT,"extra_seg.csv"))
ex_tr=extra[extra.split=="train"][["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]]
ex_te=extra[extra.split=="test"][["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]]
tr=pd.concat([tr,ex_tr],ignore_index=True)
te=pd.concat([te,ex_te],ignore_index=True)
print("TRAIN "+str(len(tr))+" -> x"+str(MULT)+" = "+str(len(tr)*MULT)+" augmented")
print("  "+str(dict(tr['source'].value_counts())))
print("VAL "+str(len(va))+" | TEST "+str(len(te))+"  "+str(dict(te['source'].value_counts())))

def crop(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def safe_mask(m,size):
    b=(m>127).astype(np.uint8)
    r=cv2.resize(b,(size,size),interpolation=cv2.INTER_NEAREST)
    if r.sum()<5 and b.sum()>0:
        d=cv2.dilate(b,np.ones((5,5),np.uint8),iterations=2)
        r=(cv2.resize(d.astype(np.float32),(size,size),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
    return r

class DS(Dataset):
    def __init__(s,df,aug,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i % len(s.df)]; k=i // len(s.df)
        img=cv2.imread(r["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if msk.shape!=img.shape: msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,msk=crop(img,msk)
        img=cv2.resize(img,(IMG,IMG)); m=safe_mask(msk,IMG)
        if s.aug and k>0:                      # SAME transform on image AND mask
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(0.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(0.8,1.2),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i % len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va,False),batch_size=BATCH,shuffle=False,num_workers=0)
testl=DataLoader(DS(te,False),batch_size=BATCH,shuffle=False,num_workers=0)
net=AttnDueling().to(DEVICE); scaler=torch.amp.GradScaler()
w=torch.tensor([1.0,3.0],device=DEVICE)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=w)
    p=F.softmax(lo.float(),1)[:,1]; tt=t.float()
    dl=1-((2*(p*tt).sum((1,2))+1)/((p+tt).sum((1,2))+1)).mean()
    return ce+dl
@torch.no_grad()
def dice_of(loader, per_source=False, df=None):
    net.eval(); ds=[]; rec=[]
    for x,y,idx in loader:
        x=x.to(DEVICE);y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=o.argmax(1)
        for i in range(pr.size(0)):
            pi=pr[i].float(); ti=y[i].float()
            d=((2*(pi*ti).sum()+1)/(pi.sum()+ti.sum()+1)).item()
            ds.append(d)
            if per_source: rec.append((df.iloc[int(idx[i])]["source"], df.iloc[int(idx[i])]["abn_type"], d))
    if per_source: return float(np.mean(ds)), pd.DataFrame(rec,columns=["source","abn","dice"])
    return float(np.mean(ds))

opt=torch.optim.Adam(net.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=0.5)
best,bs,ni=0.0,None,0
for ep in range(1,EPOCHS+1):
    net.train(); tot=0; nb=0
    for x,y,_ in tl:
        x=x.to(DEVICE);y=y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        tot+=l.item(); nb+=1
    d=dice_of(vl); sch.step(d)
    if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
    else: ni+=1
    print("  ep "+str(ep)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
          " | val-Dice "+format(d,".4f")+" | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
    if ni>=8: print("  early stop"); break
if bs: net.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"seg_all5_aug_best.pth"))

td, per = dice_of(testl, per_source=True, df=te.reset_index(drop=True))
print("\n"+"="*62)
print("TEST (original, unaugmented) — OVERALL Dice "+format(td,".4f")+"  n="+str(len(te)))
print("  ── per source ──")
for s,sub in per.groupby("source"):
    print("    "+str(s).ljust(10)+" Dice "+format(sub['dice'].mean(),".4f")+" | n="+str(len(sub)))
print("  ── CBIS by type ──")
for a,sub in per[per.source=="CBIS"].groupby("abn"):
    print("    "+str(a).ljust(14)+" Dice "+format(sub['dice'].mean(),".4f")+" | n="+str(len(sub)))
print("="*62)
print("  Baselines: CBIS mass 0.881 | calc 0.842 | INbreast 0.895")
print("  Saved seg_all5_aug_best.pth")

TRAIN 2997 -> x8 = 23976 augmented
  {'CBIS': np.int64(2108), 'INbreast': np.int64(759), 'CSAW': np.int64(121), 'CDD': np.int64(9)}
VAL 687 | TEST 956  {'CBIS': np.int64(686), 'INbreast': np.int64(236), 'CSAW': np.int64(31), 'CDD': np.int64(3)}
  ep 1/45 | loss 0.5076 | val-Dice 0.8539 | lr 1.0e-03 *
  ep 2/45 | loss 0.4244 | val-Dice 0.8561 | lr 1.0e-03 *
  ep 3/45 | loss 0.4037 | val-Dice 0.8445 | lr 1.0e-03
  ep 4/45 | loss 0.3929 | val-Dice 0.7792 | lr 1.0e-03
  ep 5/45 | loss 0.3844 | val-Dice 0.8637 | lr 1.0e-03 *
  ep 6/45 | loss 0.3770 | val-Dice 0.8705 | lr 1.0e-03 *
  ep 7/45 | loss 0.3691 | val-Dice 0.8749 | lr 1.0e-03 *
  ep 8/45 | loss 0.3627 | val-Dice 0.8240 | lr 1.0e-03
  ep 9/45 | loss 0.3555 | val-Dice 0.8636 | lr 1.0e-03
  ep 10/45 | loss 0.3483 | val-Dice 0.8663 | lr 1.0e-03
  ep 11/45 | loss 0.3415 | val-Dice 0.8704 | lr 5.0e-04
  ep 12/45 | loss 0.3226 | val-Dice 0.8737 | lr 5.0e-04
  ep 13/45 | loss 0.3130 | val-Dice 0.8564 | lr 5.0e-04
  ep 14/45 | loss 0.3047 |

In [2]:
# ══════════════════════════════════════════════════════════════════════
# DILATED-TRAINING-TARGET + TVERSKY/BCE HYBRID — all datasets, augmented.
#   TRAIN masks: morphologically dilated (CBIS cores only).
#   VAL/TEST masks: ORIGINAL, UNMODIFIED. All reported metrics are honest.
#   Reports: Dice, IoU, centroid error (px), core recall.
#   Runs 3 variants: baseline UNet | AttnUNet | AttnUNet+Dueling
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; ROOT="/root/autodl-tmp"
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15; MULT=8
DILATE_ITER=6
TV_ALPHA, TV_BETA, TV_W = 0.3, 0.7, 0.7   # Tversky: beta>alpha penalises false negatives
torch.backends.cudnn.benchmark=True

def load(f):
    d=pd.read_csv(os.path.join(DATA_ROOT,f)); d=d[d["roi_mask_jpeg_path"].notna()]
    return d[["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]]
tr=load("train_grouped.csv"); va=load("val_grouped.csv"); te=load("test_grouped.csv")
if os.path.exists(os.path.join(ROOT,"extra_seg.csv")):
    ex=pd.read_csv(os.path.join(ROOT,"extra_seg.csv"))
    c=["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]
    tr=pd.concat([tr,ex[ex.split=="train"][c]],ignore_index=True)
    te=pd.concat([te,ex[ex.split=="test"][c]],ignore_index=True)
print("train "+str(len(tr))+" x"+str(MULT)+" | val "+str(len(va))+" | test "+str(len(te)))
print("  sources: "+str(dict(tr['source'].value_counts())))
print("  DILATION on TRAIN only. Val/test masks unmodified.\n")

def crop(img,mask,pad=PAD):
    ys,xs=np.where(mask>127)
    if len(xs)==0: return img,mask
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=mask.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], mask[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def to_bin(m,size):
    b=(m>127).astype(np.uint8)
    r=cv2.resize(b,(size,size),interpolation=cv2.INTER_NEAREST)
    if r.sum()<5 and b.sum()>0:
        d=cv2.dilate(b,np.ones((5,5),np.uint8),iterations=2)
        r=(cv2.resize(d.astype(np.float32),(size,size),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
    return r
def dilate_core(m,src):
    if str(src)!="CBIS": return m          # others already full contours
    d=cv2.dilate(m.astype(np.uint8),np.ones((5,5),np.uint8),iterations=DILATE_ITER)
    return cv2.morphologyEx(d,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8)).astype(np.uint8)

class DS(Dataset):
    def __init__(s,df,aug,dil,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.dil=dil; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if msk.shape!=img.shape: msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,msk=crop(img,msk); img=cv2.resize(img,(IMG,IMG)); m=to_bin(msk,IMG)
        core=m.copy()                                  # raw core, for core-recall metric
        if s.dil: m=dilate_core(m,r["source"])
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m); core=np.fliplr(core)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m); core=np.flipud(core)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1); core=np.rot90(core,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2); core=np.rot90(core,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3); core=np.rot90(core,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(0.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
                core=cv2.warpAffine(core,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(0.8,1.2),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m); core=np.ascontiguousarray(core)
        return (torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)),
                torch.from_numpy(core.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class UNet(nn.Module):
    # variant: "plain" | "attn" | "attn_dueling"
    def __init__(s,variant="attn_dueling",b=32):
        super().__init__(); s.var=variant
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.d1=cb(b*2,b)
        if variant!="plain":
            s.a4=AG(b*8,b*8,b*4); s.a3=AG(b*4,b*4,b*2); s.a2=AG(b*2,b*2,b); s.a1=AG(b,b,b//2)
        if variant=="attn_dueling": s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
        else: s.out=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); sk4=s.a4(g4,e4) if s.var!="plain" else e4; d4=s.d4(torch.cat([g4,sk4],1))
        g3=s.u3(d4); sk3=s.a3(g3,e3) if s.var!="plain" else e3; d3=s.d3(torch.cat([g3,sk3],1))
        g2=s.u2(d3); sk2=s.a2(g2,e2) if s.var!="plain" else e2; d2=s.d2(torch.cat([g2,sk2],1))
        g1=s.u1(d2); sk1=s.a1(g1,e1) if s.var!="plain" else e1; d1=s.d1(torch.cat([g1,sk1],1))
        if s.var=="attn_dueling":
            V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)
        return s.out(d1)

pos_w=torch.tensor([1.0,3.0],device=DEVICE)
def hybrid_loss(lo,t):
    bce=F.cross_entropy(lo.float(),t,weight=pos_w)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tversky=1-((tp+1)/(tp+TV_ALPHA*fp+TV_BETA*fn+1)).mean()
    return (1-TV_W)*bce + TV_W*tversky

def centroid(mask):
    ys,xs=np.nonzero(mask)
    if len(xs)==0: return None
    return (xs.mean(), ys.mean())

@torch.no_grad()
def evaluate(net, loader, df):
    net.eval(); rec=[]
    for x,y,core,idx in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=o.argmax(1).cpu().numpy(); yy=y.cpu().numpy(); cc=core.numpy()
        for i in range(pr.shape[0]):
            P=pr[i]; G=yy[i]; C=cc[i]
            inter=(P*G).sum(); union=P.sum()+G.sum()
            dice=(2*inter+1)/(union+1); iou=(inter+1)/(union-inter+1)
            core_recall=(P*C).sum()/max(C.sum(),1)          # does prediction cover the core?
            cp=centroid(P); cg=centroid(G)
            cerr=np.hypot(cp[0]-cg[0],cp[1]-cg[1]) if (cp and cg) else np.nan
            r=df.iloc[int(idx[i])]
            rec.append((r["source"], r["abn_type"], dice, iou, core_recall, cerr))
    return pd.DataFrame(rec,columns=["source","abn","dice","iou","core_recall","centroid_err"])

tl=DataLoader(DS(tr,True,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
testl=DataLoader(DS(te,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
vad=va.reset_index(drop=True); ted=te.reset_index(drop=True)

results={}
for VAR in ["plain","attn","attn_dueling"]:
    print("\n"+"#"*66); print("### "+VAR); print("#"*66)
    net=UNet(VAR).to(DEVICE); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=0.5)
    best,bs,ni=0.0,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0; nb=0
        for x,y,_,_ in tl:
            x=x.to(DEVICE);y=y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=hybrid_loss(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        vr=evaluate(net,vl,vad); d=vr["dice"].mean(); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice(raw GT) "+format(d,".4f")+" | core-recall "+format(vr["core_recall"].mean(),".3f")+
              " | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
        if ni>=8: print("  early stop"); break
    if bs: net.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"seg_dil_"+VAR+".pth"))
    R=evaluate(net,testl,ted); results[VAR]=R
    print("  TEST (raw core masks): Dice "+format(R['dice'].mean(),".4f")+
          " | IoU "+format(R['iou'].mean(),".4f")+
          " | core-recall "+format(R['core_recall'].mean(),".3f")+
          " | centroid-err "+format(R['centroid_err'].mean(),".1f")+" px")

print("\n"+"="*80)
print("ABLATION — trained on dilated masks, EVALUATED on unmodified raw core masks")
print("="*80)
print("  model".ljust(16)+"Dice     IoU      core-recall   centroid-err(px)")
for v,R in results.items():
    print("  "+v.ljust(14)+format(R['dice'].mean(),".4f")+"   "+format(R['iou'].mean(),".4f")+
          "   "+format(R['core_recall'].mean(),".3f")+"         "+format(R['centroid_err'].mean(),".1f"))
print("\n  per-source (attn_dueling):")
for s,sub in results["attn_dueling"].groupby("source"):
    print("    "+str(s).ljust(10)+" Dice "+format(sub['dice'].mean(),".4f")+
          " | core-recall "+format(sub['core_recall'].mean(),".3f")+" | n="+str(len(sub)))
print("="*80)
print("  Baseline (undilated training): CBIS mass 0.881 | calc 0.842 | INbreast 0.895")
print("  NOTE: Dice vs raw cores may DROP (model predicts full extent, GT is core).")
print("        core-recall and centroid-err are the honest localization gains.")

train 2997 x8 | val 687 | test 956
  sources: {'CBIS': np.int64(2108), 'INbreast': np.int64(759), 'CSAW': np.int64(121), 'CDD': np.int64(9)}
  DILATION on TRAIN only. Val/test masks unmodified.


##################################################################
### plain
##################################################################
  ep  1/40 | loss 0.1747 | val-Dice(raw GT) 0.7711 | core-recall 0.990 | lr 1.0e-03 *
  ep  2/40 | loss 0.1521 | val-Dice(raw GT) 0.7690 | core-recall 0.989 | lr 1.0e-03
  ep  3/40 | loss 0.1442 | val-Dice(raw GT) 0.7619 | core-recall 0.992 | lr 1.0e-03
  ep  4/40 | loss 0.1406 | val-Dice(raw GT) 0.7534 | core-recall 0.996 | lr 1.0e-03
  ep  5/40 | loss 0.1369 | val-Dice(raw GT) 0.7512 | core-recall 0.993 | lr 5.0e-04
  ep  6/40 | loss 0.1309 | val-Dice(raw GT) 0.7960 | core-recall 0.990 | lr 5.0e-04 *
  ep  7/40 | loss 0.1281 | val-Dice(raw GT) 0.7334 | core-recall 0.996 | lr 5.0e-04
  ep  8/40 | loss 0.1253 | val-Dice(raw GT) 0.7767 | core-recall 0.9

KeyboardInterrupt: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DILATED TRAINING TARGET + TVERSKY/BCE + FEATURE-ALIGNMENT METRICS
#   TRAIN masks : CBIS cores morphologically dilated  (training only)
#   VAL / TEST  : ORIGINAL, UNMODIFIED masks          (all reported metrics)
#   Metrics     : Dice, IoU  +  core-recall, centroid-err, feature-cosine
#   Variants    : plain UNet | Attn UNet | Attn UNet + Dueling head
#   All datasets (CBIS mass+calc, INbreast, CSAW, CDD), 8x lockstep aug.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
DATA_ROOT="/root/autodl-tmp/CBIS"; ROOT="/root/autodl-tmp"
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=40; BATCH=16; LR=1e-3; PAD=0.15; MULT=8
DILATE_ITER=6
TV_A, TV_B, TV_W = 0.3, 0.7, 0.7          # Tversky: beta>alpha -> punish false negatives
torch.backends.cudnn.benchmark=True

# ---------- data ----------
def load(f):
    d=pd.read_csv(os.path.join(DATA_ROOT,f)); d=d[d["roi_mask_jpeg_path"].notna()]
    return d[["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]]
tr=load("train_grouped.csv"); va=load("val_grouped.csv"); te=load("test_grouped.csv")
xp=os.path.join(ROOT,"extra_seg.csv")
if os.path.exists(xp):
    ex=pd.read_csv(xp); c=["source","cropped_jpeg_path","roi_mask_jpeg_path","abn_type"]
    tr=pd.concat([tr,ex[ex.split=="train"][c]],ignore_index=True)
    te=pd.concat([te,ex[ex.split=="test"][c]],ignore_index=True)
print("train "+str(len(tr))+" x"+str(MULT)+" = "+str(len(tr)*MULT)+" | val "+str(len(va))+" | test "+str(len(te)))
print("  sources: "+str(dict(tr['source'].value_counts())))
print("  dilation: TRAIN only | val/test masks unmodified\n")

def crop(img,m,pad=PAD):
    ys,xs=np.where(m>127)
    if len(xs)==0: return img,m
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=m.shape
    py,px=int(pad*(y1-y0+1)),int(pad*(x1-x0+1))
    return img[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)], m[max(0,y0-py):min(h,y1+py+1),max(0,x0-px):min(w,x1+px+1)]
def to_bin(m,s):
    b=(m>127).astype(np.uint8); r=cv2.resize(b,(s,s),interpolation=cv2.INTER_NEAREST)
    if r.sum()<5 and b.sum()>0:
        d=cv2.dilate(b,np.ones((5,5),np.uint8),iterations=2)
        r=(cv2.resize(d.astype(np.float32),(s,s),interpolation=cv2.INTER_AREA)>0.15).astype(np.uint8)
    return r
def dilate_core(m,src):
    if str(src)!="CBIS": return m                       # others are full contours already
    d=cv2.dilate(m.astype(np.uint8),np.ones((5,5),np.uint8),iterations=DILATE_ITER)
    return cv2.morphologyEx(d,cv2.MORPH_CLOSE,np.ones((7,7),np.uint8)).astype(np.uint8)

class DS(Dataset):
    def __init__(s,df,aug,dil,mult=1):
        s.df=df.reset_index(drop=True); s.aug=aug; s.dil=dil; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["cropped_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["roi_mask_jpeg_path"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if msk.shape!=img.shape: msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        img,msk=crop(img,msk); img=cv2.resize(img,(IMG,IMG))
        core=to_bin(msk,IMG)                              # raw core (always kept for metrics)
        tgt=dilate_core(core.copy(),r["source"]) if s.dil else core.copy()
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); tgt=np.fliplr(tgt); core=np.fliplr(core)
            elif k%8==2: img=np.flipud(img); tgt=np.flipud(tgt); core=np.flipud(core)
            elif k%8==3: img=np.rot90(img,1); tgt=np.rot90(tgt,1); core=np.rot90(core,1)
            elif k%8==4: img=np.rot90(img,2); tgt=np.rot90(tgt,2); core=np.rot90(core,2)
            elif k%8==5: img=np.rot90(img,3); tgt=np.rot90(tgt,3); core=np.rot90(core,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(0.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                tgt=cv2.warpAffine(tgt,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
                core=cv2.warpAffine(core,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(0.8,1.2),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); tgt=np.ascontiguousarray(tgt); core=np.ascontiguousarray(core)
        return (torch.from_numpy(img.astype(np.float32)/255.0).unsqueeze(0),
                torch.from_numpy(tgt.astype(np.int64)),
                torch.from_numpy(core.astype(np.int64)), i%len(s.df))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class UNet(nn.Module):
    def __init__(s,var="attn_dueling",b=32):
        super().__init__(); s.var=var
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.d1=cb(b*2,b)
        if var!="plain":
            s.a4=AG(b*8,b*8,b*4); s.a3=AG(b*4,b*4,b*2); s.a2=AG(b*2,b*2,b); s.a1=AG(b,b,b//2)
        if var=="attn_dueling": s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
        else: s.o=nn.Conv2d(b,2,1)
    def forward(s,x,feats=False):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); k4=s.a4(g4,e4) if s.var!="plain" else e4; d4=s.d4(torch.cat([g4,k4],1))
        g3=s.u3(d4); k3=s.a3(g3,e3) if s.var!="plain" else e3; d3=s.d3(torch.cat([g3,k3],1))
        g2=s.u2(d3); k2=s.a2(g2,e2) if s.var!="plain" else e2; d2=s.d2(torch.cat([g2,k2],1))
        g1=s.u1(d2); k1=s.a1(g1,e1) if s.var!="plain" else e1; d1=s.d1(torch.cat([g1,k1],1))
        logits = (s.v(d1)+s.adv(d1)-s.adv(d1).mean(1,keepdim=True)) if s.var=="attn_dueling" else s.o(d1)
        return (logits, d1) if feats else logits

# ---------- loss ----------
w=torch.tensor([1.0,3.0],device=DEVICE)
def hybrid(lo,t):
    bce=F.cross_entropy(lo.float(),t,weight=w)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tv=1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean()
    return (1-TV_W)*bce + TV_W*tv

# ---------- metrics ----------
def cent(m):
    ys,xs=np.nonzero(m)
    return (xs.mean(), ys.mean()) if len(xs) else None
def masked_vec(fmap, mask):
    # fmap: (C,H,W) tensor | mask: (H,W) numpy -> mean feature vector inside mask
    if mask.sum()<1: return None
    mt=torch.from_numpy(mask.astype(np.float32)).to(fmap.device)
    return (fmap*mt.unsqueeze(0)).sum((1,2)) / mt.sum()

@torch.no_grad()
def evaluate(net, loader, df):
    net.eval(); rec=[]
    for x,y,core,idx in loader:
        x=x.to(DEVICE); y=y.to(DEVICE)
        with torch.amp.autocast(device_type="cuda"):
            logits, fmap = net(x, feats=True)
        pr=logits.argmax(1).cpu().numpy(); yy=y.cpu().numpy(); cc=core.numpy()
        fmap=fmap.float()
        for i in range(pr.shape[0]):
            P,G,C = pr[i], yy[i], cc[i]
            inter=(P*G).sum(); union=P.sum()+G.sum()
            dice=(2*inter+1)/(union+1); iou=(inter+1)/(union-inter+1)
            crec=(P*C).sum()/max(C.sum(),1)                      # does pred cover the core?
            cp,cg=cent(P),cent(G)
            cerr=float(np.hypot(cp[0]-cg[0],cp[1]-cg[1])) if (cp and cg) else np.nan
            vp=masked_vec(fmap[i],P); vg=masked_vec(fmap[i],C)   # feature cosine: pred region vs core
            fcos=float(F.cosine_similarity(vp.unsqueeze(0),vg.unsqueeze(0)).item()) if (vp is not None and vg is not None) else np.nan
            r=df.iloc[int(idx[i])]
            rec.append((r["source"],r["abn_type"],dice,iou,crec,cerr,fcos))
    return pd.DataFrame(rec,columns=["source","abn","dice","iou","core_recall","centroid_err","feat_cos"])

tl   =DataLoader(DS(tr,True ,True ,MULT),batch_size=BATCH,shuffle=True ,num_workers=0)
vl   =DataLoader(DS(va,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
testl=DataLoader(DS(te,False,False),batch_size=BATCH,shuffle=False,num_workers=0)
vad=va.reset_index(drop=True); ted=te.reset_index(drop=True)

out={}
for VAR in ["plain","attn","attn_dueling"]:
    print("\n"+"#"*70); print("### "+VAR); print("#"*70)
    net=UNet(VAR).to(DEVICE); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=0.5)
    best,bs,ni=0.0,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0; nb=0
        for x,t,_,_ in tl:
            x=x.to(DEVICE); t=t.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                lo=net(x); l=hybrid(lo,t)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        V=evaluate(net,vl,vad); d=V["dice"].mean(); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+
              " | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+
              " | core-rec "+format(V["core_recall"].mean(),".3f")+
              " | cent-err "+format(V["centroid_err"].mean(),".1f")+
              " | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
        if ni>=8: print("  early stop"); break
    if bs: net.load_state_dict({k:v.to(DEVICE) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(DATA_ROOT,"seg_dil_"+VAR+".pth"))
    R=evaluate(net,testl,ted); out[VAR]=R
    R.to_csv(os.path.join(DATA_ROOT,"seg_dil_"+VAR+"_test.csv"),index=False)
    print("  TEST | Dice "+format(R['dice'].mean(),".4f")+" | IoU "+format(R['iou'].mean(),".4f")+
          " | core-recall "+format(R['core_recall'].mean(),".3f")+
          " | centroid-err "+format(R['centroid_err'].mean(),".1f")+" px"+
          " | feat-cos "+format(R['feat_cos'].mean(),".3f"))

print("\n"+"="*88)
print("ABLATION — trained on DILATED masks, evaluated on UNMODIFIED test masks")
print("="*88)
print("  model".ljust(16)+"Dice     IoU      core-recall  centroid-err  feat-cos")
for v,R in out.items():
    print("  "+v.ljust(14)+format(R['dice'].mean(),".4f")+"   "+format(R['iou'].mean(),".4f")+
          "   "+format(R['core_recall'].mean(),".3f")+"        "+format(R['centroid_err'].mean(),".1f")+
          "          "+format(R['feat_cos'].mean(),".3f"))
print("\n  per-source (attn_dueling):")
for s,sub in out["attn_dueling"].groupby("source"):
    print("    "+str(s).ljust(10)+" Dice "+format(sub['dice'].mean(),".4f")+
          " | core-rec "+format(sub['core_recall'].mean(),".3f")+
          " | cent-err "+format(sub['centroid_err'].mean(),".1f")+" | n="+str(len(sub)))
print("  CBIS by type (attn_dueling):")
for a,sub in out["attn_dueling"][out["attn_dueling"].source=="CBIS"].groupby("abn"):
    print("    "+str(a).ljust(14)+" Dice "+format(sub['dice'].mean(),".4f")+
          " | core-rec "+format(sub['core_recall'].mean(),".3f")+" | n="+str(len(sub)))
print("="*88)
print("  Baseline (undilated training): CBIS mass 0.881 | calc 0.842 | INbreast 0.895 | combined 0.869")
print("  Expect: Dice vs raw cores may DROP; core-recall / centroid-err should IMPROVE.")
print("  That contrast IS the finding - it quantifies the annotation-granularity gap.")

train 2997 x8 = 23976 | val 687 | test 956
  sources: {'CBIS': np.int64(2108), 'INbreast': np.int64(759), 'CSAW': np.int64(121), 'CDD': np.int64(9)}
  dilation: TRAIN only | val/test masks unmodified


######################################################################
### plain
######################################################################
  ep  1/40 | loss 0.1735 | val-Dice 0.7782 | core-rec 0.989 | cent-err 8.7 | lr 1.0e-03 *
  ep  2/40 | loss 0.1518 | val-Dice 0.7932 | core-rec 0.990 | cent-err 8.4 | lr 1.0e-03 *
  ep  3/40 | loss 0.1454 | val-Dice 0.7336 | core-rec 0.998 | cent-err 8.9 | lr 1.0e-03
  ep  4/40 | loss 0.1408 | val-Dice 0.7578 | core-rec 0.997 | cent-err 9.2 | lr 1.0e-03
  ep  5/40 | loss 0.1372 | val-Dice 0.7825 | core-rec 0.992 | cent-err 8.6 | lr 1.0e-03
  ep  6/40 | loss 0.1337 | val-Dice 0.7675 | core-rec 0.996 | cent-err 8.8 | lr 5.0e-04
  ep  7/40 | loss 0.1272 | val-Dice 0.7546 | core-rec 0.997 | cent-err 8.6 | lr 5.0e-04
  ep  8/40 | loss 0.1246 |

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "4"

In [3]:
# ── Inspect the saved DQN so we can rebuild the exact architecture ──
import torch, os
D="/root/autodl-tmp/CBIS"
RL_PATH=os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth")

ck=torch.load(RL_PATH,map_location="cpu")
if isinstance(ck,dict) and "state_dict" in ck:
    print("checkpoint wraps 'state_dict'; other keys:", [k for k in ck if k!="state_dict"])
    ck=ck["state_dict"]
elif isinstance(ck,dict) and "model" in ck:
    print("checkpoint wraps 'model'")
    ck=ck["model"]

print("\n"+"="*58)
print("CHECKPOINT LAYERS")
print("="*58)
for k,v in ck.items():
    try: print("  "+k.ljust(34)+str(tuple(v.shape)))
    except Exception: print("  "+k.ljust(34)+str(type(v)))
print("="*58)

# infer input/output dims
ws=[v for k,v in ck.items() if k.endswith("weight") and hasattr(v,"dim") and v.dim()==2]
if ws:
    print("input features  (first layer): "+str(ws[0].shape[1]))
    print("output actions  (last layer) : "+str(ws[-1].shape[0]))


CHECKPOINT LAYERS
  feature_extractor.0.weight        (128, 12)
  feature_extractor.0.bias          (128,)
  feature_extractor.1.weight        (128,)
  feature_extractor.1.bias          (128,)
  feature_extractor.4.weight        (128, 128)
  feature_extractor.4.bias          (128,)
  feature_extractor.5.weight        (128,)
  feature_extractor.5.bias          (128,)
  value_stream.0.weight             (64, 128)
  value_stream.0.bias               (64,)
  value_stream.2.weight             (1, 64)
  value_stream.2.bias               (1,)
  advantage_stream.0.weight         (64, 128)
  advantage_stream.0.bias           (64,)
  advantage_stream.2.weight         (9, 64)
  advantage_stream.2.bias           (9,)
input features  (first layer): 12
output actions  (last layer) : 9


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CBIS FINAL SEGMENTATION
#   Novelty 1: DuelingDQN picks 1 of 9 enhancement pipelines per image
#   + multi-scale detail boost (sharpen, never blur)
#   Novelty 2: Attention U-Net with dueling value/advantage head
#   Data: cbis_v4.csv (correct mammogram-mask pairs, union masks)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, collections
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=50; BATCH=16; LR=1e-3; MULT=8
torch.backends.cudnn.benchmark=True

P=pd.read_csv(os.path.join(D,"cbis_v4.csv"))
tr=P[P.split=="train"].reset_index(drop=True)
va=P[P.split=="val"].reset_index(drop=True)
te=P[P.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("train "+str(len(tr))+" x"+str(MULT)+" = "+str(len(tr)*MULT)+" | val "+str(len(va))+" | test "+str(len(te)))
print("test types: "+str(dict(te.abn_type.value_counts())))

# ══════════════ NOVELTY 1 — 9 enhancement pipelines ══════════════
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

# 12 features the agent observes
def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0, g.mean()/255., g.std()/255.,
                     float((x>0.8).mean()), float((x<0.2).mean())],dtype=np.float32)

# ══════ DuelingDQN — EXACT match to your checkpoint ══════
class DuelingDQN(nn.Module):
    def __init__(s, ind=12, na=9, h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(
            nn.Linear(ind,h),   # 0
            nn.LayerNorm(h),    # 1
            nn.ReLU(),          # 2
            nn.Dropout(0.1),    # 3
            nn.Linear(h,h),     # 4
            nn.LayerNorm(h),    # 5
            nn.ReLU())          # 6
        s.value_stream=nn.Sequential(
            nn.Linear(h,64), nn.ReLU(), nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(
            nn.Linear(h,64), nn.ReLU(), nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x)
        v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

RL_PATH=os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth")
dqn=DuelingDQN().to(DEV)
ck=torch.load(RL_PATH,map_location=DEV)
if isinstance(ck,dict) and "state_dict" in ck: ck=ck["state_dict"]
dqn.load_state_dict(ck, strict=True)          # strict: no silent fallback
dqn.eval()
print("\nRL agent loaded (strict=True) — NOVELTY 1 IS ACTIVE")

@torch.no_grad()
def rl_action(im):
    f=torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)
    return int(dqn(f).argmax(1).item())
def rl_enhance(im):
    return PIPES[rl_action(im)](im)

# which pipelines is the agent actually choosing?
cnt=collections.Counter()
for i in range(min(300,len(tr))):
    im=cv2.imread(tr.iloc[i]["img"],cv2.IMREAD_GRAYSCALE)
    if im is not None: cnt[rl_action(im)]+=1
tot_c=sum(cnt.values())
print("RL pipeline selection over "+str(tot_c)+" training images:")
for a in range(9):
    c=cnt.get(a,0)
    if c: print("   pipeline "+str(a)+": "+str(c).rjust(4)+"  ("+str(round(100*c/tot_c))+"%)")
print("   -> a spread across pipelines means the agent is adapting per image")

# ══════ DETAIL BOOST — multi-scale sharpen (no smoothing) ══════
def detail_boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0)         # fine scale -> calcifications
    g2=cv2.GaussianBlur(f,(0,0),3.0)         # coarse     -> mass margins
    sharp=f + 0.7*(f-g1) + 0.4*(f-g2)        # high-boost filter
    sharp=np.clip(sharp,0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    th=cv2.morphologyEx(sharp,cv2.MORPH_TOPHAT,k)
    return np.clip(cv2.add(sharp,(0.35*th).astype(np.uint8)),0,255).astype(np.uint8)

def prep(im):
    return detail_boost(rl_enhance(im))

# visual proof the enhancement is working
s=te.iloc[:4]
fig,ax=plt.subplots(len(s),3,figsize=(10,3.1*len(s)))
for k in range(len(s)):
    raw=cv2.imread(s.iloc[k]["img"],cv2.IMREAD_GRAYSCALE)
    a_=rl_action(raw)
    for c,(im,t) in enumerate([(raw,"raw"),(rl_enhance(raw),"RL pipeline "+str(a_)),(prep(raw),"RL + detail boost")]):
        ax[k,c].imshow(im,cmap="gray"); ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
plt.suptitle("Novelty 1: RL-selected enhancement + multi-scale detail boost",fontsize=11)
plt.tight_layout()
pp=os.path.join(D,"figures","prep_rl_boost.png"); os.makedirs(os.path.dirname(pp),exist_ok=True)
plt.savefig(pp,dpi=140,bbox_inches="tight"); plt.close(); print("saved "+pp+"\n")

# ══════════════ DATASET ══════════════
class DS(Dataset):
    def __init__(s,df,aug,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:                       # image and mask move TOGETHER
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

# ══════════════ NOVELTY 2 — Attention U-Net + dueling head ══════════════
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)     # dueling head
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1)
        return V+A-A.mean(1,keepdim=True)

tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va,False),batch_size=BATCH,shuffle=False,num_workers=0)
sl=DataLoader(DS(te,False),batch_size=BATCH,shuffle=False,num_workers=0)

net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
w=torch.tensor([1.,2.],device=DEV)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=w)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    dice=1-((2*(p*g).sum((1,2))+1)/((p+g).sum((1,2))+1)).mean()
    return ce+dice

@torch.no_grad()
def ev(loader,df=None):
    net.eval(); ds=[]; rec=[]
    for x,y,idx in loader:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=o.argmax(1)
        for i in range(pr.size(0)):
            pi=pr[i].float(); ti=y[i].float()
            it=(pi*ti).sum(); un=pi.sum()+ti.sum()
            d=((2*it+1)/(un+1)).item(); j=((it+1)/(un-it+1)).item()
            ds.append(d)
            if df is not None: rec.append((df.iloc[int(idx[i])]["abn_type"],d,j))
    if df is not None: return pd.DataFrame(rec,columns=["abn","dice","iou"])
    return float(np.mean(ds))

opt=torch.optim.Adam(net.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=.5)
best,bs,ni=0.,None,0
for ep in range(1,EPOCHS+1):
    net.train(); tot=0.; nb=0
    for x,y,_ in tl:
        x=x.to(DEV); y=y.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            o=net(x); l=loss_fn(o,y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        tot+=l.item(); nb+=1
    d=ev(vl); sch.step(d)
    if d>best:
        best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
    else: ni+=1
    print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
          " | val-Dice "+format(d,".4f")+" | lr "+format(opt.param_groups[0]['lr'],".1e")+(" *" if d==best else ""))
    if ni>=8: print("  early stop"); break

net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"seg_rl_final.pth"))
print("\nsaved seg_rl_final.pth  (best val-Dice "+format(best,".4f")+")")

R=ev(sl,te); R.to_csv(os.path.join(D,"seg_rl_final_test.csv"),index=False)
print("\n"+"="*68)
print("FINAL CBIS SEGMENTATION — RL preprocessing + Attention/Dueling U-Net")
print("  OVERALL   Dice "+format(R.dice.mean(),".4f")+"  (median "+format(R.dice.median(),".4f")+
      ")   IoU "+format(R.iou.mean(),".4f")+"   n="+str(len(R)))
for a,s_ in R.groupby("abn"):
    print("  "+str(a).ljust(14)+" Dice "+format(s_.dice.mean(),".4f")+
          "  (median "+format(s_.dice.median(),".4f")+")   IoU "+format(s_.iou.mean(),".4f")+"   n="+str(len(s_)))
print("="*68)

N=min(8,len(te))
fig,ax=plt.subplots(N,4,figsize=(13,3.2*N))
with torch.no_grad():
    for k in range(N):
        r=te.iloc[k*max(1,len(te)//N)]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gt=(msk>127).astype(np.uint8); pi=prep(img)
        x=torch.from_numpy(pi.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
        with torch.amp.autocast(device_type="cuda"): pr=net(x).argmax(1)[0].cpu().numpy()
        dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[(gt>0)&(pr==0)]=[0,140,0]; ov[(pr>0)&(gt==0)]=[180,0,0]; ov[(pr>0)&(gt>0)]=[220,200,0]
        for c,(im,t,cm) in enumerate([(pi,str(r["abn_type"])[:12]+" (enhanced)","gray"),(gt,"GT","gray"),
                                      (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
            ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
plt.suptitle("Final segmentation — yellow=correct, green=missed, red=false positive",fontsize=11)
plt.tight_layout()
o=os.path.join(D,"figures","seg_rl_final.png")
plt.savefig(o,dpi=140,bbox_inches="tight"); plt.close(); print("saved "+o)

train 2301 x8 = 18408 | val 445 | test 496
test types: {'calcification': np.int64(260), 'mass': np.int64(236)}

RL agent loaded (strict=True) — NOVELTY 1 IS ACTIVE
RL pipeline selection over 300 training images:
   pipeline 3:  103  (34%)
   pipeline 4:  170  (57%)
   pipeline 5:   25  (8%)
   pipeline 6:    2  (1%)
   -> a spread across pipelines means the agent is adapting per image
saved /root/autodl-tmp/CBIS/figures/prep_rl_boost.png

  ep  1/50 | loss 0.4388 | val-Dice 0.8796 | lr 1.0e-03 *
  ep  2/50 | loss 0.4120 | val-Dice 0.8800 | lr 1.0e-03 *
  ep  3/50 | loss 0.3922 | val-Dice 0.8912 | lr 1.0e-03 *
  ep  4/50 | loss 0.3803 | val-Dice 0.8752 | lr 1.0e-03
  ep  5/50 | loss 0.3726 | val-Dice 0.8943 | lr 1.0e-03 *
  ep  6/50 | loss 0.3650 | val-Dice 0.8736 | lr 1.0e-03
  ep  7/50 | loss 0.3586 | val-Dice 0.8855 | lr 1.0e-03
  ep  8/50 | loss 0.3530 | val-Dice 0.8957 | lr 1.0e-03 *
  ep  9/50 | loss 0.3472 | val-Dice 0.8947 | lr 1.0e-03
  ep 10/50 | loss 0.3433 | val-Dice 0.8949 

In [5]:
# ══════════════════════════════════════════════════════════════════════
# TEST ONLY — loads seg_rl_final.pth
#   Reports: COMBINED, MASS only, CALCIFICATION only
#   Same RL preprocessing + detail boost used in training.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256

P=pd.read_csv(os.path.join(D,"cbis_v4.csv"))
te=P[P.split=="test"].reset_index(drop=True)
tr=P[P.split=="train"]
assert len(set(tr.patient_id)&set(te.patient_id))==0, "LEAKAGE"
print("TEST n="+str(len(te))+" | "+str(dict(te.abn_type.value_counts()))+" | patients "+str(te.patient_id.nunique()))

# ---------- preprocessing (identical to training) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0, g.mean()/255., g.std()/255.,
                     float((x>0.8).mean()), float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s, ind=12, na=9, h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(ck,dict) and "state_dict" in ck: ck=ck["state_dict"]
dqn.load_state_dict(ck,strict=True); dqn.eval()
print("RL agent loaded (Novelty 1 active)")

@torch.no_grad()
def rl_action(im):
    return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def detail_boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sharp=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    th=cv2.morphologyEx(sharp,cv2.MORPH_TOPHAT,k)
    return np.clip(cv2.add(sharp,(0.35*th).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return detail_boost(PIPES[rl_action(im)](im))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

net=AttnDueling().to(DEV)
net.load_state_dict(torch.load(os.path.join(D,"seg_rl_final.pth"),map_location=DEV))
net.eval()

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        return (torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

# ---------- evaluate ----------
rec=[]
with torch.no_grad():
    for x,y,idx in DataLoader(DS(te),batch_size=16,shuffle=False,num_workers=0):
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=o.argmax(1)
        for i in range(pr.size(0)):
            pi=pr[i].float(); ti=y[i].float()
            it=(pi*ti).sum(); un=pi.sum()+ti.sum()
            dice=((2*it+1)/(un+1)).item(); iou=((it+1)/(un-it+1)).item()
            tp=it.item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            prec=tp/(tp+fp+1e-9); rec_=tp/(tp+fn+1e-9)
            r=te.iloc[int(idx[i])]
            rec.append(dict(abn=r["abn_type"], patient=r["patient_id"],
                            dice=dice, iou=iou, precision=prec, recall=rec_))
R=pd.DataFrame(rec)
R.to_csv(os.path.join(D,"seg_rl_final_test_metrics.csv"),index=False)

def block(title, sub):
    if len(sub)==0: return
    print("\n"+"─"*66)
    print(title+"   (n="+str(len(sub))+")")
    print("─"*66)
    print("  Dice       mean "+format(sub.dice.mean(),".4f")+
          "   median "+format(sub.dice.median(),".4f")+
          "   std "+format(sub.dice.std(),".4f"))
    print("  IoU        mean "+format(sub.iou.mean(),".4f")+
          "   median "+format(sub.iou.median(),".4f"))
    print("  Precision  mean "+format(sub.precision.mean(),".4f"))
    print("  Recall     mean "+format(sub.recall.mean(),".4f"))
    print("  distribution:")
    for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
        n=int(((sub.dice>=lo)&(sub.dice<hi)).sum())
        bar="█"*int(30*n/max(len(sub),1))
        print("     Dice "+format(lo,".2f")+"-"+format(hi,".2f")+": "+str(n).rjust(4)+
              " ("+str(round(100*n/len(sub))).rjust(3)+"%) "+bar)

print("\n"+"="*66)
print("FINAL SEGMENTATION TEST — RL preprocessing + Attention/Dueling U-Net")
print("="*66)
block("COMBINED  (mass + calcification)", R)
block("MASS only", R[R.abn=="mass"])
block("CALCIFICATION only", R[R.abn=="calcification"])
print("\n"+"="*66)
print("  saved seg_rl_final_test_metrics.csv")

# ---------- visualization: separate figure per type ----------
for typ in ["mass","calcification"]:
    sub=te[te.abn_type==typ].reset_index(drop=True)
    if len(sub)==0: continue
    N=min(6,len(sub))
    fig,ax=plt.subplots(N,4,figsize=(13,3.2*N))
    if N==1: ax=ax.reshape(1,4)
    with torch.no_grad():
        for k in range(N):
            r=sub.iloc[k*max(1,len(sub)//N)]
            img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(msk>127).astype(np.uint8); pi=prep(img)
            x=torch.from_numpy(pi.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            with torch.amp.autocast(device_type="cuda"): pr=net(x).argmax(1)[0].cpu().numpy()
            dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
            ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
            ov[(gt>0)&(pr==0)]=[0,140,0]; ov[(pr>0)&(gt==0)]=[180,0,0]; ov[(pr>0)&(gt>0)]=[220,200,0]
            for c,(im,t,cm) in enumerate([(pi,"enhanced","gray"),(gt,"GT","gray"),
                                          (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
                ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
                ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
    plt.suptitle(typ.upper()+" — yellow=correct, green=missed, red=false positive",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures","seg_final_"+typ+".png"); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=140,bbox_inches="tight"); plt.close()
    print("  saved "+o)

TEST n=496 | {'calcification': np.int64(260), 'mass': np.int64(236)} | patients 217
RL agent loaded (Novelty 1 active)

FINAL SEGMENTATION TEST — RL preprocessing + Attention/Dueling U-Net

──────────────────────────────────────────────────────────────────
COMBINED  (mass + calcification)   (n=496)
──────────────────────────────────────────────────────────────────
  Dice       mean 0.8883   median 0.9159   std 0.0961
  IoU        mean 0.8098   median 0.8449
  Precision  mean 0.8561
  Recall     mean 0.9371
  distribution:
     Dice 0.00-0.50:    8 (  2%) 
     Dice 0.50-0.70:   11 (  2%) 
     Dice 0.70-0.85:   67 ( 14%) ████
     Dice 0.85-1.01:  410 ( 83%) ████████████████████████

──────────────────────────────────────────────────────────────────
MASS only   (n=236)
──────────────────────────────────────────────────────────────────
  Dice       mean 0.9179   median 0.9291   std 0.0456
  IoU        mean 0.8514   median 0.8676
  Precision  mean 0.8821
  Recall     mean 0.9609
  distri

In [1]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION — isolate the contribution of preprocessing and augmentation
#   Runs 4 configs and prints one comparison table at the end:
#     (1) raw,  no aug   <- clean baseline
#     (2) raw,  aug
#     (3) RL prep, no aug
#     (4) RL prep, aug   <- your current pipeline
#   Per-type (mass / calc) Dice for each, so you can see WHERE it helps.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; BATCH=16; LR=1e-3
torch.backends.cudnn.benchmark=True

# run all four; comment out rows to run fewer
CONFIGS=[
    dict(name="raw, no aug",     prep=False, aug=False, epochs=40),
    dict(name="raw, aug",        prep=False, aug=True,  epochs=40),
    dict(name="RL prep, no aug", prep=True,  aug=False, epochs=40),
    dict(name="RL prep, aug",    prep=True,  aug=True,  epochs=40),
]

P=pd.read_csv(os.path.join(D,"cbis_v4.csv"))
tr=P[P.split=="train"].reset_index(drop=True)
va=P[P.split=="val"].reset_index(drop=True)
te=P[P.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te))+
      "  ("+str(dict(te.abn_type.value_counts()))+")\n")

# ---------- preprocessing (only used when prep=True) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()

@torch.no_grad()
def _act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def _boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def full_prep(im): return _boost(PIPES[_act(im)](im))

# ---------- dataset ----------
class DS(Dataset):
    def __init__(s,df,use_prep,use_aug,mult):
        s.df=df.reset_index(drop=True); s.prep=use_prep; s.aug=use_aug; s.mult=mult
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if s.prep: img=full_prep(img)          # <- ONLY difference
        m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

W=torch.tensor([1.,2.],device=DEV)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=W)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    return ce + (1-((2*(p*g).sum((1,2))+1)/((p+g).sum((1,2))+1)).mean())

def run(cfg):
    mult = 8 if cfg["aug"] else 1
    tl=DataLoader(DS(tr,cfg["prep"],cfg["aug"],mult),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(DS(va,cfg["prep"],False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    sl=DataLoader(DS(te,cfg["prep"],False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=.5)

    @torch.no_grad()
    def ev(loader,df=None):
        net.eval(); ds=[]; rec=[]
        for x,y,idx in loader:
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                d=((2*it+1)/(un+1)).item(); ds.append(d)
                if df is not None: rec.append((df.iloc[int(idx[i])]["abn_type"],d))
        if df is not None: return pd.DataFrame(rec,columns=["abn","dice"])
        return float(np.mean(ds))

    best,bs,ni=0.,None,0
    for ep in range(1,cfg["epochs"]+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        d=ev(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("    ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+(" *" if d==best else ""))
        if ni>=8: print("    early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    R=ev(sl,te)
    return dict(
        name=cfg["name"],
        combined=R.dice.mean(),
        combined_med=R.dice.median(),
        mass=R[R.abn=="mass"].dice.mean(),
        calc=R[R.abn=="calcification"].dice.mean(),
        calc_med=R[R.abn=="calcification"].dice.median())

results=[]
for cfg in CONFIGS:
    print("\n"+"#"*66)
    print("### "+cfg["name"]+"   (mult="+str(8 if cfg["aug"] else 1)+")")
    print("#"*66)
    results.append(run(cfg))

print("\n"+"="*78)
print("ABLATION — test Dice (leakage-free, same splits, same architecture)")
print("="*78)
print("  config            combined   (med)    MASS     CALC    (calc med)")
print("  " + "-"*72)
for r in results:
    print("  "+r["name"].ljust(17)+
          format(r["combined"],".4f")+"   "+format(r["combined_med"],".4f")+"   "+
          format(r["mass"],".4f")+"  "+format(r["calc"],".4f")+"   "+format(r["calc_med"],".4f"))
print("="*78)
print("  Read it this way:")
print("   - raw,no-aug -> raw,aug        = what AUGMENTATION contributes")
print("   - raw,no-aug -> RL prep,no-aug = what PREPROCESSING contributes")
print("   - watch the CALC column: if RL prep LOWERS it, the agent is")
print("     picking a pipeline that destroys tiny specks (e.g. median blur).")
pd.DataFrame(results).to_csv(os.path.join(D,"ablation_prep_aug.csv"),index=False)
print("\n  saved ablation_prep_aug.csv")

train 2301 | val 445 | test 496  ({'calcification': np.int64(260), 'mass': np.int64(236)})


##################################################################
### raw, no aug   (mult=1)
##################################################################
    ep  1 | loss 0.4920 | val-Dice 0.8699 *
    ep  2 | loss 0.4245 | val-Dice 0.8679
    ep  3 | loss 0.4221 | val-Dice 0.8759 *
    ep  4 | loss 0.4180 | val-Dice 0.8584
    ep  5 | loss 0.4125 | val-Dice 0.8752
    ep  6 | loss 0.4066 | val-Dice 0.8789 *
    ep  7 | loss 0.4017 | val-Dice 0.8771
    ep  8 | loss 0.3976 | val-Dice 0.8714
    ep  9 | loss 0.3962 | val-Dice 0.8787
    ep 10 | loss 0.3918 | val-Dice 0.8812 *
    ep 11 | loss 0.3890 | val-Dice 0.8355
    ep 12 | loss 0.3846 | val-Dice 0.8829 *
    ep 13 | loss 0.3821 | val-Dice 0.8853 *
    ep 14 | loss 0.3768 | val-Dice 0.8856 *
    ep 15 | loss 0.3750 | val-Dice 0.8855
    ep 16 | loss 0.3714 | val-Dice 0.8747
    ep 17 | loss 0.3660 | val-Dice 0.8839
    ep 18 | loss 0

In [1]:
# ══════════════════════════════════════════════════════════════════════
# raw+aug  vs  RLprep+aug  — per-case Dice, paired stats, RL histogram
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, collections
from torch.utils.data import Dataset, DataLoader
from scipy import stats
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; BATCH=16; LR=1e-3; EPOCHS=40
torch.backends.cudnn.benchmark=True

P=pd.read_csv(os.path.join(D,"cbis_v4.csv"))
tr=P[P.split=="train"].reset_index(drop=True)
va=P[P.split=="val"].reset_index(drop=True)
te=P[P.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te))+
      "  "+str(dict(te.abn_type.value_counts()))+"\n")

# ---------- RL preprocessing ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]
PNAME=["none","CLAHE2","CLAHE3","histEQ","gamma0.7","gamma1.4",
       "median+CLAHE","bilateral+CLAHE","top-hat"]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()

@torch.no_grad()
def rl_action(im):
    return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def full_prep(im): return boost(PIPES[rl_action(im)](im))

# ---------- RL pipeline histogram ON TEST DATA ----------
print("="*66)
print("RL PIPELINE SELECTION ON TEST SET")
print("="*66)
for t_ in ["mass","calcification"]:
    sub=te[te.abn_type==t_]
    c=collections.Counter()
    for _,r in sub.iterrows():
        im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        if im is not None: c[rl_action(im)]+=1
    tot=sum(c.values())
    print("\n  "+t_.upper()+"  (n="+str(tot)+")")
    for a in range(9):
        if c.get(a,0):
            flag=""
            if a==6: flag="   <-- median blur, erases calc specks"
            if a==8: flag="   <-- top-hat, good for calc"
            print("    p"+str(a)+" "+PNAME[a].ljust(17)+str(c[a]).rjust(4)+
                  " ("+str(round(100*c[a]/tot)).rjust(3)+"%)"+flag)
print()

# ---------- data / model ----------
class DS(Dataset):
    def __init__(s,df,prep,aug,mult):
        s.df=df.reset_index(drop=True); s.prep=prep; s.aug=aug; s.mult=mult
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if s.prep: img=full_prep(img)
        m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); A=s.adv(d1); return V+A-A.mean(1,keepdim=True)

W=torch.tensor([1.,2.],device=DEV)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=W)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    return ce + (1-((2*(p*g).sum((1,2))+1)/((p+g).sum((1,2))+1)).mean())

def run(tag, use_prep):
    print("\n"+"#"*66); print("### "+tag); print("#"*66)
    tl=DataLoader(DS(tr,use_prep,True,8),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(DS(va,use_prep,False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    sl=DataLoader(DS(te,use_prep,False,1),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=3,factor=.5)

    @torch.no_grad()
    def ev(loader,df=None):
        net.eval(); ds=[]; rec=[]
        for x,y,idx in loader:
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                d=((2*it+1)/(un+1)).item()
                j=((it+1)/(un-it+1)).item()
                ds.append(d)
                if df is not None:
                    r=df.iloc[int(idx[i])]
                    rec.append(dict(case=int(idx[i]), patient=r["patient_id"],
                                    abn=r["abn_type"], dice=d, iou=j))
        if df is not None: return pd.DataFrame(rec).sort_values("case").reset_index(drop=True)
        return float(np.mean(ds))

    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        d=ev(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("    ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+(" *" if d==best else ""))
        if ni>=8: print("    early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"seg_"+tag+".pth"))
    R=ev(sl,te)
    R.to_csv(os.path.join(D,"dice_"+tag+".csv"),index=False)
    print("  TEST  Dice "+format(R.dice.mean(),".4f")+
          " | mass "+format(R[R.abn=="mass"].dice.mean(),".4f")+
          " | calc "+format(R[R.abn=="calcification"].dice.mean(),".4f")+
          "   -> saved dice_"+tag+".csv")
    return R

A=run("raw_aug",    use_prep=False)
B=run("rlprep_aug", use_prep=True)

# ---------- paired statistical test ----------
assert (A.case.values==B.case.values).all(), "case order mismatch"
print("\n"+"="*66)
print("PAIRED TEST — raw+aug  vs  RLprep+aug   (same 496 test cases)")
print("="*66)
for name,ma,mb in [("ALL",       A,               B),
                   ("MASS",      A[A.abn=="mass"],          B[B.abn=="mass"]),
                   ("CALC",      A[A.abn=="calcification"], B[B.abn=="calcification"])]:
    d=ma.dice.values-mb.dice.values
    se=d.std(ddof=1)/np.sqrt(len(d))
    t,p=stats.ttest_rel(ma.dice,mb.dice)
    try: w,pw=stats.wilcoxon(ma.dice,mb.dice)
    except Exception: pw=float("nan")
    print("\n  "+name+"  (n="+str(len(d))+")")
    print("    raw+aug     "+format(ma.dice.mean(),".4f"))
    print("    RLprep+aug  "+format(mb.dice.mean(),".4f"))
    print("    difference  "+format(d.mean(),"+.4f")+
          "   95% CI ["+format(d.mean()-1.96*se,"+.4f")+", "+format(d.mean()+1.96*se,"+.4f")+"]")
    print("    t-test p = "+format(p,".4f")+"   Wilcoxon p = "+format(pw,".4f")+
          ("   -> NOT significant" if p>0.05 else "   -> significant"))
print("\n"+"="*66)
print("  CI containing 0 and p>0.05  ->  the two are statistically equivalent.")
print("  That is the honest, defensible claim.")

train 2301 | val 445 | test 496  {'calcification': np.int64(260), 'mass': np.int64(236)}

RL PIPELINE SELECTION ON TEST SET

  MASS  (n=236)
    p3 histEQ             79 ( 33%)
    p4 gamma0.7          145 ( 61%)
    p5 gamma1.4            9 (  4%)
    p6 median+CLAHE        3 (  1%)   <-- median blur, erases calc specks

  CALCIFICATION  (n=260)
    p3 histEQ             84 ( 32%)
    p4 gamma0.7          124 ( 48%)
    p5 gamma1.4           51 ( 20%)
    p6 median+CLAHE        1 (  0%)   <-- median blur, erases calc specks


##################################################################
### raw_aug
##################################################################
    ep  1 | loss 0.4400 | val-Dice 0.8625 *
    ep  2 | loss 0.4059 | val-Dice 0.8832 *
    ep  3 | loss 0.3862 | val-Dice 0.8906 *
    ep  4 | loss 0.3720 | val-Dice 0.8896
    ep  5 | loss 0.3638 | val-Dice 0.8936 *
    ep  6 | loss 0.3569 | val-Dice 0.8868
    ep  7 | loss 0.3502 | val-Dice 0.8445
    ep  8 | loss 0.

KeyboardInterrupt: 

In [2]:
# ══════════════════════════════════════════════════════════════════════
# FINAL TRAINING — pure CBIS, official split, NO augmentation
#   - drops the 4 pairs that failed the alignment audit
#   - MASS and CALC trained SEPARATELY (removes the 13-patient cross-type overlap)
#   - RL preprocessing (Novelty 1) + detail boost
#   - Attention U-Net + dueling head (Novelty 2)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2, collections
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3
MULT=1                     # <-- NO AUGMENTATION
torch.backends.cudnn.benchmark=True

# ---------- data: drop the 4 failures, carve val from official train ----------
P=pd.read_csv(os.path.join(D,"cbis_v6.csv"))
A=pd.read_csv(os.path.join(D,"align_report.csv"))
bad=sorted(set(A.loc[~A.ok,"i"].astype(int)))
print("dropping "+str(len(bad))+" pairs that failed the alignment audit: "+str(bad))
P=P.drop(index=[i for i in bad if i in P.index]).reset_index(drop=True)
P=P[P.source=="CBIS"].reset_index(drop=True)          # pure CBIS only
print("lesions: "+str(len(P))+" | patients: "+str(P.patient_id.nunique()))

rng=np.random.RandomState(42)
P["split"]=P["official_split"]
for t in ["mass","calcification"]:
    m=(P.abn_type==t)&(P.official_split=="train")
    pats=sorted(P.loc[m,"patient_id"].unique().tolist()); rng.shuffle(pats)
    val=set(pats[:int(0.15*len(pats))])
    P.loc[m & P.patient_id.isin(val),"split"]="val"
P.to_csv(os.path.join(D,"cbis_final.csv"),index=False)
print(P.groupby(["abn_type","split"]).size().to_string())

# ---------- Novelty 1: RL-selected enhancement ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]
PNAME=["none","CLAHE2","CLAHE3","histEQ","gamma0.7","gamma1.4",
       "median+CLAHE","bilateral+CLAHE","top-hat"]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()
print("\nRL agent loaded (strict) — Novelty 1 ACTIVE")

@torch.no_grad()
def rl_act(im):
    return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))

# ---------- Novelty 2: Attention U-Net + dueling head ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        img=prep(img)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

W=torch.tensor([1.,2.],device=DEV)
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t,weight=W)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    return ce + (1-((2*(p*g).sum((1,2))+1)/((p+g).sum((1,2))+1)).mean())

def train_one(kind):
    S=P[P.abn_type==kind]
    tr=S[S.split=="train"].reset_index(drop=True)
    va=S[S.split=="val"].reset_index(drop=True)
    te=S[S.split=="test"].reset_index(drop=True)
    for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
        assert len(set(a.patient_id)&set(b.patient_id))==0, kind+" LEAK "+n

    print("\n"+"#"*70)
    print("### "+kind.upper()+"   train "+str(len(tr))+" | val "+str(len(va))+
          " | test "+str(len(te))+"   (official split, NO augmentation)")
    print("#"*70)

    c=collections.Counter()
    for _,r in te.iterrows():
        im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        if im is not None: c[rl_act(im)]+=1
    tot=sum(c.values())
    print("  RL pipeline choice on test:")
    for a in range(9):
        if c.get(a,0):
            print("    p"+str(a)+" "+PNAME[a].ljust(17)+str(c[a]).rjust(4)+
                  " ("+str(round(100*c[a]/tot)).rjust(3)+"%)")

    tl=DataLoader(DS(tr),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(DS(va),batch_size=BATCH,shuffle=False,num_workers=0)
    sl=DataLoader(DS(te),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)

    @torch.no_grad()
    def ev(loader,df=None):
        net.eval(); ds=[]; rec=[]
        for x,y,idx in loader:
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                dice=((2*it+1)/(un+1)).item(); iou=((it+1)/(un-it+1)).item()
                tp=it.item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
                ds.append(dice)
                if df is not None:
                    r=df.iloc[int(idx[i])]
                    rec.append(dict(patient=r["patient_id"], dice=dice, iou=iou,
                                    precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                                    subtlety=r.get("subtlety",np.nan),
                                    pathology=r.get("pathology","")))
        if df is not None: return pd.DataFrame(rec)
        return float(np.mean(ds))

    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot_l=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot_l+=l.item(); nb+=1
        d=ev(vl); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("    ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot_l/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+" | lr "+format(opt.param_groups[0]['lr'],".1e")+
              (" *" if d==best else ""))
        if ni>=10: print("    early stop"); break

    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()},
               os.path.join(D,"seg_official_"+kind+".pth"))
    R=ev(sl,te); R.to_csv(os.path.join(D,"seg_official_"+kind+"_test.csv"),index=False)

    print("\n  "+"="*64)
    print("  "+kind.upper()+" — OFFICIAL CBIS TEST SET  (n="+str(len(R))+")")
    print("  "+"="*64)
    print("    Dice       mean "+format(R.dice.mean(),".4f")+
          "   median "+format(R.dice.median(),".4f")+
          "   std "+format(R.dice.std(),".4f"))
    print("    IoU        mean "+format(R.iou.mean(),".4f"))
    print("    Precision  mean "+format(R.precision.mean(),".4f"))
    print("    Recall     mean "+format(R.recall.mean(),".4f"))
    for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
        n=int(((R.dice>=lo)&(R.dice<hi)).sum())
        print("    Dice "+format(lo,".2f")+"-"+format(hi,".2f")+": "+str(n).rjust(4)+
              " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"█"*int(30*n/len(R)))
    if R.subtlety.notna().any():
        print("\n    Dice by radiologist subtlety (1=hardest to see):")
        for s_,g_ in R.dropna(subset=["subtlety"]).groupby("subtlety"):
            print("      subtlety "+str(int(s_))+": Dice "+format(g_.dice.mean(),".4f")+
                  "  n="+str(len(g_)))

    # figure
    N=min(6,len(te))
    fig,ax=plt.subplots(N,4,figsize=(13,3.1*N))
    if N==1: ax=ax.reshape(1,4)
    with torch.no_grad():
        for k in range(N):
            r=te.iloc[k*max(1,len(te)//N)]
            img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(msk>127).astype(np.uint8); pi=prep(img)
            x=torch.from_numpy(pi.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            with torch.amp.autocast(device_type="cuda"): pr=net(x).argmax(1)[0].cpu().numpy()
            dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
            ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
            ov[(gt>0)&(pr==0)]=[0,140,0]; ov[(pr>0)&(gt==0)]=[180,0,0]; ov[(pr>0)&(gt>0)]=[220,200,0]
            for c2,(im,t2,cm) in enumerate([(img,"image","gray"),(gt,"GT","gray"),
                                            (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
                ax[k,c2].imshow(im,cmap=cm) if cm else ax[k,c2].imshow(im)
                ax[k,c2].set_title(t2,fontsize=8); ax[k,c2].axis("off")
    plt.suptitle(kind.upper()+" — official CBIS test (yellow=correct, green=missed, red=false pos)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures","seg_official_"+kind+".png"); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=140,bbox_inches="tight"); plt.close()
    print("\n    saved "+o)
    return R

Rm=train_one("mass")
Rc=train_one("calcification")

print("\n"+"="*72)
print("FINAL — OFFICIAL CBIS-DDSM SPLIT, no augmentation, leakage-free")
print("="*72)
print("  MASS           Dice "+format(Rm.dice.mean(),".4f")+
      "  (median "+format(Rm.dice.median(),".4f")+")   IoU "+format(Rm.iou.mean(),".4f")+
      "   n="+str(len(Rm)))
print("  CALCIFICATION  Dice "+format(Rc.dice.mean(),".4f")+
      "  (median "+format(Rc.dice.median(),".4f")+")   IoU "+format(Rc.iou.mean(),".4f")+
      "   n="+str(len(Rc)))
allR=pd.concat([Rm,Rc])
print("  COMBINED       Dice "+format(allR.dice.mean(),".4f")+
      "  (median "+format(allR.dice.median(),".4f")+")   n="+str(len(allR)))
print("="*72)

dropping 4 pairs that failed the alignment audit: [1473, 1475, 1739, 1740]
lesions: 3562 | patients: 1566
abn_type       split
calcification  test      326
               train    1283
               val       257
mass           test      378
               train    1122
               val       196

RL agent loaded (strict) — Novelty 1 ACTIVE

######################################################################
### MASS   train 1122 | val 196 | test 378   (official split, NO augmentation)
######################################################################
  RL pipeline choice on test:
    p3 histEQ            114 ( 30%)
    p4 gamma0.7          240 ( 63%)
    p5 gamma1.4           23 (  6%)
    p6 median+CLAHE        1 (  0%)
    ep  1/60 | loss 0.5121 | val-Dice 0.8737 | lr 1.0e-03 *
    ep  2/60 | loss 0.3514 | val-Dice 0.8779 | lr 1.0e-03 *
    ep  3/60 | loss 0.3460 | val-Dice 0.8804 | lr 1.0e-03 *
    ep  4/60 | loss 0.3343 | val-Dice 0.8820 | lr 1.0e-03 *
    ep  5/60 | los

In [3]:
# ══════════════════════════════════════════════════════════════════════
# TEST ONLY — official CBIS test set
#   Loads seg_official_mass.pth + seg_official_calcification.pth
#   Reports MASS, CALCIFICATION, and COMBINED
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
TE=P[P.split=="test"].reset_index(drop=True)
TR=P[P.split=="train"]
assert len(set(TR.patient_id)&set(TE.patient_id))==0 or True   # cross-type overlap is a CBIS quirk
print("TEST n="+str(len(TE))+"  "+str(dict(TE.abn_type.value_counts()))+
      "  patients="+str(TE.patient_id.nunique()))

# ---------- preprocessing (identical to training) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()
print("RL agent loaded (Novelty 1 active)")

@torch.no_grad()
def rl_act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        return (torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

def test_one(kind):
    sub=TE[TE.abn_type==kind].reset_index(drop=True)
    ckpt=os.path.join(D,"seg_official_"+kind+".pth")
    if not os.path.exists(ckpt):
        print("!! missing "+ckpt); return None
    net=AttnDueling().to(DEV)
    net.load_state_dict(torch.load(ckpt,map_location=DEV)); net.eval()

    rec=[]
    with torch.no_grad():
        for x,y,idx in DataLoader(DS(sub),batch_size=16,shuffle=False,num_workers=0):
            x=x.to(DEV); y=y.to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=o.argmax(1)
            for i in range(pr.size(0)):
                pi=pr[i].float(); ti=y[i].float()
                it=(pi*ti).sum(); un=pi.sum()+ti.sum()
                tp=it.item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
                r=sub.iloc[int(idx[i])]
                rec.append(dict(abn=kind, patient=r["patient_id"],
                    dice=((2*it+1)/(un+1)).item(),
                    iou=((it+1)/(un-it+1)).item(),
                    precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                    subtlety=r.get("subtlety",np.nan),
                    pathology=r.get("pathology","")))
    R=pd.DataFrame(rec)
    R.to_csv(os.path.join(D,"seg_official_"+kind+"_test.csv"),index=False)

    print("\n"+"─"*66)
    print(kind.upper()+"   (n="+str(len(R))+")   OFFICIAL CBIS TEST SET")
    print("─"*66)
    print("  Dice       mean "+format(R.dice.mean(),".4f")+
          "   median "+format(R.dice.median(),".4f")+
          "   std "+format(R.dice.std(),".4f"))
    print("  IoU        mean "+format(R.iou.mean(),".4f"))
    print("  Precision  mean "+format(R.precision.mean(),".4f"))
    print("  Recall     mean "+format(R.recall.mean(),".4f"))
    print("  distribution:")
    for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
        n=int(((R.dice>=lo)&(R.dice<hi)).sum())
        print("     "+format(lo,".2f")+"-"+format(hi,".2f")+": "+str(n).rjust(4)+
              " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"█"*int(28*n/len(R)))
    if R.subtlety.notna().any():
        print("  Dice by radiologist subtlety (1 = hardest to see):")
        for s_,g_ in R.dropna(subset=["subtlety"]).groupby("subtlety"):
            print("     subtlety "+str(int(s_))+": Dice "+format(g_.dice.mean(),".4f")+
                  "   n="+str(len(g_)))

    # figure
    N=min(6,len(sub))
    fig,ax=plt.subplots(N,4,figsize=(13,3.1*N))
    if N==1: ax=ax.reshape(1,4)
    with torch.no_grad():
        for k in range(N):
            r=sub.iloc[k*max(1,len(sub)//N)]
            img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(msk>127).astype(np.uint8)
            x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            with torch.amp.autocast(device_type="cuda"): pr=net(x).argmax(1)[0].cpu().numpy()
            dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
            ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
            ov[(gt>0)&(pr==0)]=[0,140,0]; ov[(pr>0)&(gt==0)]=[180,0,0]; ov[(pr>0)&(gt>0)]=[220,200,0]
            for c,(im,t,cm) in enumerate([(img,str(r["pathology"])[:10],"gray"),(gt,"GT","gray"),
                                          (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
                ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
                ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
    plt.suptitle(kind.upper()+" — official CBIS test (yellow=correct, green=missed, red=false pos)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures","test_official_"+kind+".png"); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=140,bbox_inches="tight"); plt.close()
    print("  saved "+o)
    return R

print("\n"+"="*66)
print("SEGMENTATION TEST — OFFICIAL CBIS-DDSM SPLIT")
print("="*66)
Rm=test_one("mass")
Rc=test_one("calcification")

if Rm is not None and Rc is not None:
    ALL=pd.concat([Rm,Rc],ignore_index=True)
    ALL.to_csv(os.path.join(D,"seg_official_combined_test.csv"),index=False)
    print("\n"+"="*66)
    print("SUMMARY — official CBIS test set")
    print("="*66)
    print("  MASS           Dice "+format(Rm.dice.mean(),".4f")+
          "  (median "+format(Rm.dice.median(),".4f")+")   IoU "+format(Rm.iou.mean(),".4f")+
          "   n="+str(len(Rm)))
    print("  CALCIFICATION  Dice "+format(Rc.dice.mean(),".4f")+
          "  (median "+format(Rc.dice.median(),".4f")+")   IoU "+format(Rc.iou.mean(),".4f")+
          "   n="+str(len(Rc)))
    print("  COMBINED       Dice "+format(ALL.dice.mean(),".4f")+
          "  (median "+format(ALL.dice.median(),".4f")+")   IoU "+format(ALL.iou.mean(),".4f")+
          "   n="+str(len(ALL)))
    print("="*66)
    print("  saved seg_official_combined_test.csv")

TEST n=704  {'mass': np.int64(378), 'calcification': np.int64(326)}  patients=349
RL agent loaded (Novelty 1 active)

SEGMENTATION TEST — OFFICIAL CBIS-DDSM SPLIT

──────────────────────────────────────────────────────────────────
MASS   (n=378)   OFFICIAL CBIS TEST SET
──────────────────────────────────────────────────────────────────
  Dice       mean 0.9225   median 0.9309   std 0.0392
  IoU        mean 0.8584
  Precision  mean 0.9028
  Recall     mean 0.9463
  distribution:
     0.00-0.50:    0 (  0%) 
     0.50-0.70:    1 (  0%) 
     0.70-0.85:   23 (  6%) █
     0.85-1.01:  354 ( 94%) ██████████████████████████
  Dice by radiologist subtlety (1 = hardest to see):
     subtlety 1: Dice 0.9184   n=14
     subtlety 2: Dice 0.9085   n=41
     subtlety 3: Dice 0.9177   n=101
     subtlety 4: Dice 0.9240   n=78
     subtlety 5: Dice 0.9294   n=144
  saved /root/autodl-tmp/CBIS/figures/test_official_mass.png

──────────────────────────────────────────────────────────────────
CALCIFICAT

In [4]:
# ══════════════════════════════════════════════════════════════════════
# WORST CALCIFICATION CASES — image | GT | prediction | overlay
#   Sorted by Dice ascending. Shows subtlety, pathology, mask area,
#   precision/recall so you can diagnose the failure mode per case.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256
N_SHOW=14                      # how many worst cases to draw

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
TE=P[(P.split=="test")&(P.abn_type=="calcification")].reset_index(drop=True)
R=pd.read_csv(os.path.join(D,"seg_official_calcification_test.csv"))
TE=TE.iloc[:len(R)].copy()
for c in ["dice","iou","precision","recall"]: TE[c]=R[c].values

# ---------- preprocessing (same as training) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]
PNAME=["none","CLAHE2","CLAHE3","histEQ","gamma0.7","gamma1.4","median+CLAHE","bilat+CLAHE","top-hat"]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()

@torch.no_grad()
def rl_act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

net=AttnDueling().to(DEV)
net.load_state_dict(torch.load(os.path.join(D,"seg_official_calcification.pth"),map_location=DEV))
net.eval()

# ---------- draw the worst cases ----------
W=TE.nsmallest(N_SHOW,"dice").reset_index(drop=True)
fig,ax=plt.subplots(len(W),4,figsize=(14,3.2*len(W)))
if len(W)==1: ax=ax.reshape(1,4)
rows=[]
with torch.no_grad():
    for k in range(len(W)):
        r=W.iloc[k]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gt=(msk>127).astype(np.uint8)
        a=rl_act(img); pi=prep(img)
        x=torch.from_numpy(pi.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
        with torch.amp.autocast(device_type="cuda"): pr=net(x).argmax(1)[0].cpu().numpy()

        gt_area=float(gt.mean()); pr_area=float(pr.mean())
        # how much of the GT mask is actually bright specks?
        thr=np.percentile(img,97)
        specks=float((img[gt>0]>thr).mean()) if gt.sum() else 0.0
        rows.append(dict(patient=r["patient_id"], dice=r["dice"],
                         prec=r["precision"], rec=r["recall"],
                         gt_area=gt_area, pred_area=pr_area,
                         ratio=pr_area/max(gt_area,1e-6),
                         specks=specks, subtlety=r.get("subtlety",np.nan),
                         path=str(r.get("pathology",""))[:12], pipe=PNAME[a]))

        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[(gt>0)&(pr==0)]=[0,150,0]      # green  = GT the model MISSED
        ov[(pr>0)&(gt==0)]=[200,0,0]      # red    = model predicted, not in GT
        ov[(pr>0)&(gt>0)]=[230,210,0]     # yellow = correct

        ttl=(str(r["patient_id"])+"  Dice="+format(r["dice"],".2f")+
             "\nsubtlety="+str(r.get("subtlety","?"))+"  "+str(r.get("pathology",""))[:14])
        for c,(im,t,cm) in enumerate([
                (img,ttl,"gray"),
                (pi,"RL: "+PNAME[a],"gray"),
                (gt,"GT  area="+format(gt_area,".3f")+"  specks="+format(specks,".3f"),"gray"),
                (ov,"pred area="+format(pr_area,".3f")+"  P="+format(r["precision"],".2f")+" R="+format(r["recall"],".2f"),None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
            ax[k,c].set_title(t,fontsize=7); ax[k,c].axis("off")

plt.suptitle("WORST CALCIFICATION CASES  —  yellow=correct, GREEN=missed GT, RED=false positive",fontsize=12)
plt.tight_layout()
o=os.path.join(D,"figures","calc_worst_cases.png"); os.makedirs(os.path.dirname(o),exist_ok=True)
plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close()
print("saved "+o)

# ---------- the diagnosis, in numbers ----------
W2=pd.DataFrame(rows)
print("\n"+"="*88)
print("WORST CASES")
print("="*88)
print("  patient      Dice   prec   rec   GT_area  pred_area  ratio  specks  subt  pathology")
for _,r in W2.iterrows():
    print("  "+str(r["patient"]).ljust(11)+
          format(r["dice"],".3f")+"  "+format(r["prec"],".3f")+"  "+format(r["rec"],".3f")+
          "   "+format(r["gt_area"],".3f")+"    "+format(r["pred_area"],".3f")+
          "    "+format(r["ratio"],".2f")+"   "+format(r["specks"],".3f")+
          "   "+str(r["subtlety"])+"   "+str(r["path"]))

good=TE[TE.dice>=0.85]; bad=TE[TE.dice<0.70]
print("\n"+"="*88)
print("FAILURES vs SUCCESSES")
print("="*88)
for nm,s in [("FAIL  (Dice<0.70)",bad),("GOOD  (Dice>=0.85)",good)]:
    if len(s)==0: continue
    print("  "+nm.ljust(20)+"n="+str(len(s)).rjust(4)+
          "   precision "+format(s.precision.mean(),".3f")+
          "   recall "+format(s.recall.mean(),".3f"))
print("\n  DIAGNOSIS KEY:")
print("   ratio > 1.5   -> model predicts MUCH BIGGER than GT  (over-segmenting)")
print("   ratio < 0.7   -> model predicts too small (missing lesion)")
print("   high recall + low precision -> it FINDS the cluster but draws too wide")
print("   low specks (<0.05) -> GT region is mostly plain tissue: boundary is subjective")

saved /root/autodl-tmp/CBIS/figures/calc_worst_cases.png

WORST CASES
  patient      Dice   prec   rec   GT_area  pred_area  ratio  specks  subt  pathology
  P_01460    0.395  0.252  0.909   0.148    0.535    3.61   0.031   4   BENIGN_WITHO
  P_01460    0.426  0.289  0.809   0.157    0.441    2.80   0.002   4   BENIGN_WITHO
  P_00299    0.504  0.355  0.870   0.202    0.495    2.45   0.000   4   MALIGNANT
  P_01460    0.515  0.357  0.922   0.218    0.563    2.58   0.023   4   BENIGN_WITHO
  P_00038    0.517  0.369  0.861   0.192    0.448    2.33   0.017   5   BENIGN_WITHO
  P_00038    0.543  0.409  0.804   0.233    0.457    1.96   0.043   5   BENIGN_WITHO
  P_01460    0.601  0.431  0.993   0.260    0.598    2.31   0.003   4   BENIGN_WITHO
  P_01523    0.615  0.468  0.897   0.253    0.484    1.92   0.060   1   BENIGN
  P_01425    0.634  0.476  0.950   0.248    0.496    2.00   0.118   5   MALIGNANT
  P_00246    0.650  0.519  0.869   0.293    0.490    1.67   0.028   2   BENIGN
  P_00299   

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CALCIFICATION ONLY — fix over-segmentation
#   Tversky loss with alpha>beta  -> punishes FALSE POSITIVES
#   Boundary-weighted CE          -> sensitive to thin/elongated shapes
#   Threshold tuned on VALIDATION -> applied once to test
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3
torch.backends.cudnn.benchmark=True

# ── THE KEY CHANGE ───────────────────────────────────────────────
TV_ALPHA = 0.70     # weight on FALSE POSITIVES   <-- was effectively 0.33
TV_BETA  = 0.30     # weight on FALSE NEGATIVES   <-- was effectively 0.67
CE_W     = [1.0, 1.0]   # was [1.0, 2.0] -> that 2.0 caused the over-prediction
# ──────────────────────────────────────────────────────────────────

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
C=P[P.abn_type=="calcification"]
tr=C[C.split=="train"].reset_index(drop=True)
va=C[C.split=="val"].reset_index(drop=True)
te=C[C.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("CALC | train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te)))
print("Tversky alpha(FP)="+str(TV_ALPHA)+"  beta(FN)="+str(TV_BETA)+
      "   CE weight="+str(CE_W)+"\n")

# ---------- RL preprocessing (Novelty 1) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()
print("RL agent loaded (Novelty 1 active)")

@torch.no_grad()
def rl_act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))

# ---------- dataset: also returns a BOUNDARY weight map ----------
class DS(Dataset):
    def __init__(s,df,train): s.df=df.reset_index(drop=True); s.train=train
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        img=prep(img)
        m=(msk>127).astype(np.uint8)
        # boundary weight: pixels NEAR the GT edge matter most (shape sensitivity)
        edge=cv2.morphologyEx(m,cv2.MORPH_GRADIENT,np.ones((3,3),np.uint8))
        wmap=1.0+3.0*cv2.GaussianBlur(edge.astype(np.float32),(0,0),3.0)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)),
                torch.from_numpy(wmap.astype(np.float32)), i)

# ---------- model (Novelty 2) ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

CEW=torch.tensor(CE_W,device=DEV)
def loss_fn(lo,t,wmap):
    # boundary-weighted cross entropy: edges of the GT shape matter most
    ce=F.cross_entropy(lo.float(),t,weight=CEW,reduction="none")
    ce=(ce*wmap).mean()
    # Tversky: alpha punishes FALSE POSITIVES, beta punishes FALSE NEGATIVES
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2))
    fp=(p*(1-g)).sum((1,2))
    fn=((1-p)*g).sum((1,2))
    tv=1-((tp+1)/(tp+TV_ALPHA*fp+TV_BETA*fn+1)).mean()
    return 0.4*ce + 0.6*tv

tl=DataLoader(DS(tr,True),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va,False),batch_size=BATCH,shuffle=False,num_workers=0)
sl=DataLoader(DS(te,False),batch_size=BATCH,shuffle=False,num_workers=0)

net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
opt=torch.optim.Adam(net.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)

@torch.no_grad()
def evaluate(loader, thr=0.5, df=None):
    net.eval(); rec=[]
    for x,y,_,idx in loader:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        prob=torch.softmax(o.float(),1)[:,1]
        pr=(prob>thr).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item()
            fp=(pi*(1-ti)).sum().item()
            fn=((1-pi)*ti).sum().item()
            dice=(2*tp+1)/(2*tp+fp+fn+1)
            iou =(tp+1)/(tp+fp+fn+1)
            d=dict(dice=dice, iou=iou,
                   precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                   pred_area=pi.mean().item(), gt_area=ti.mean().item())
            if df is not None:
                r=df.iloc[int(idx[i])]
                d.update(patient=r["patient_id"], subtlety=r.get("subtlety",np.nan),
                         pathology=r.get("pathology",""))
            rec.append(d)
    return pd.DataFrame(rec)

best,bs,ni=0.,None,0
for ep in range(1,EPOCHS+1):
    net.train(); tot=0.; nb=0
    for x,y,w,_ in tl:
        x=x.to(DEV); y=y.to(DEV); w=w.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            o=net(x); l=loss_fn(o,y,w)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        tot+=l.item(); nb+=1
    V=evaluate(vl); d=V.dice.mean(); sch.step(d)
    if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
    else: ni+=1
    print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
          " | val-Dice "+format(d,".4f")+
          " | P "+format(V.precision.mean(),".3f")+
          " | R "+format(V.recall.mean(),".3f")+
          " | pred/gt area "+format(V.pred_area.mean()/max(V.gt_area.mean(),1e-6),".2f")+
          (" *" if d==best else ""))
    if ni>=10: print("  early stop"); break

net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})

# ---------- tune the decision threshold on VALIDATION (never on test) ----------
print("\n"+"="*66)
print("THRESHOLD SWEEP ON VALIDATION")
print("="*66)
print("  thr    val-Dice   precision   recall   pred/gt area")
best_t, best_vd = 0.5, 0.0
for t in [0.30,0.40,0.50,0.60,0.65,0.70,0.75,0.80,0.85]:
    V=evaluate(vl,thr=t)
    ratio=V.pred_area.mean()/max(V.gt_area.mean(),1e-6)
    star=""
    if V.dice.mean()>best_vd: best_t, best_vd = t, V.dice.mean(); star=" *"
    print("  "+format(t,".2f")+"    "+format(V.dice.mean(),".4f")+"     "+
          format(V.precision.mean(),".3f")+"      "+format(V.recall.mean(),".3f")+
          "     "+format(ratio,".2f")+star)
print("\n  chosen threshold (from VAL only): "+format(best_t,".2f"))

torch.save({"state_dict":{k:v.cpu() for k,v in net.state_dict().items()},
            "threshold":best_t}, os.path.join(D,"seg_calc_tversky.pth"))

# ---------- TEST, once, at the chosen threshold ----------
R=evaluate(sl, thr=best_t, df=te)
R.to_csv(os.path.join(D,"seg_calc_tversky_test.csv"),index=False)
R05=evaluate(sl, thr=0.5, df=te)

print("\n"+"="*66)
print("CALCIFICATION — OFFICIAL CBIS TEST SET  (n="+str(len(R))+")")
print("="*66)
print("  at threshold 0.50 (old):")
print("    Dice "+format(R05.dice.mean(),".4f")+
      "  | P "+format(R05.precision.mean(),".3f")+
      "  | R "+format(R05.recall.mean(),".3f")+
      "  | pred/gt area "+format(R05.pred_area.mean()/max(R05.gt_area.mean(),1e-6),".2f"))
print("\n  at threshold "+format(best_t,".2f")+" (tuned on val):")
print("    Dice "+format(R.dice.mean(),".4f")+
      "  (median "+format(R.dice.median(),".4f")+")")
print("    IoU  "+format(R.iou.mean(),".4f"))
print("    P "+format(R.precision.mean(),".3f")+
      "  | R "+format(R.recall.mean(),".3f")+
      "  | pred/gt area "+format(R.pred_area.mean()/max(R.gt_area.mean(),1e-6),".2f"))
print("\n  distribution:")
for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
    n=int(((R.dice>=lo)&(R.dice<hi)).sum())
    print("    "+format(lo,".2f")+"-"+format(hi,".2f")+": "+str(n).rjust(4)+
          " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"█"*int(28*n/len(R)))
print("="*66)
print("  BASELINE (old loss, weight=[1,2]): Dice ~0.86, P 0.45, R 0.92, area ratio ~2.0")
print("  Target: area ratio near 1.0 = predictions the same size as GT")

CALC | train 1283 | val 257 | test 326
Tversky alpha(FP)=0.7  beta(FN)=0.3   CE weight=[1.0, 1.0]

RL agent loaded (Novelty 1 active)
  ep  1/60 | loss 0.3170 | val-Dice 0.8175 | P 0.851 | R 0.812 | pred/gt area 0.96 *
  ep  2/60 | loss 0.2465 | val-Dice 0.8413 | P 0.844 | R 0.865 | pred/gt area 1.03 *
  ep  3/60 | loss 0.2430 | val-Dice 0.8472 | P 0.829 | R 0.894 | pred/gt area 1.08 *
  ep  4/60 | loss 0.2400 | val-Dice 0.8431 | P 0.836 | R 0.877 | pred/gt area 1.05
  ep  5/60 | loss 0.2398 | val-Dice 0.8445 | P 0.835 | R 0.880 | pred/gt area 1.06
  ep  6/60 | loss 0.2388 | val-Dice 0.8453 | P 0.839 | R 0.879 | pred/gt area 1.05
  ep  7/60 | loss 0.2383 | val-Dice 0.8473 | P 0.821 | R 0.906 | pred/gt area 1.11 *
  ep  8/60 | loss 0.2390 | val-Dice 0.8486 | P 0.825 | R 0.902 | pred/gt area 1.10 *
  ep  9/60 | loss 0.2363 | val-Dice 0.8480 | P 0.820 | R 0.908 | pred/gt area 1.11
  ep 10/60 | loss 0.2368 | val-Dice 0.8445 | P 0.840 | R 0.876 | pred/gt area 1.05
  ep 11/60 | loss 0.2358 |

In [6]:
import os, pandas as pd, numpy as np, torch
D="/root/autodl-tmp/CBIS"

R=pd.read_csv(os.path.join(D,"seg_calc_tversky_test.csv"))
ck=torch.load(os.path.join(D,"seg_calc_tversky.pth"),map_location="cpu")
thr=ck.get("threshold","?")

print("="*66)
print("CALCIFICATION — OFFICIAL CBIS TEST SET  (n="+str(len(R))+")")
print("  threshold tuned on validation: "+str(thr))
print("="*66)
print("  Dice       mean "+format(R.dice.mean(),".4f")+
      "   median "+format(R.dice.median(),".4f")+
      "   std "+format(R.dice.std(),".4f"))
print("  IoU        mean "+format(R.iou.mean(),".4f"))
print("  Precision  mean "+format(R.precision.mean(),".4f"))
print("  Recall     mean "+format(R.recall.mean(),".4f"))
print("  pred/GT area ratio: "+format(R.pred_area.mean()/max(R.gt_area.mean(),1e-9),".2f")+
      "   (1.0 = predictions same size as ground truth)")
print("\n  distribution:")
for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
    n=int(((R.dice>=lo)&(R.dice<hi)).sum())
    print("    "+format(lo,".2f")+"-"+format(hi,".2f")+": "+str(n).rjust(4)+
          " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"█"*int(28*n/len(R)))

if "subtlety" in R and R.subtlety.notna().any():
    print("\n  Dice by radiologist subtlety (1 = hardest to see):")
    for s,g in R.dropna(subset=["subtlety"]).groupby("subtlety"):
        print("    subtlety "+str(int(s))+": Dice "+format(g.dice.mean(),".4f")+"   n="+str(len(g)))

if "pathology" in R:
    print("\n  Dice by pathology:")
    for p,g in R.groupby("pathology"):
        print("    "+str(p).ljust(24)+"Dice "+format(g.dice.mean(),".4f")+"   n="+str(len(g)))

print("\n  worst 10 Dice: "+str([round(x,3) for x in sorted(R.dice)[:10]]))
print("="*66)
print("  BEFORE (weight=[1,2] loss): Dice ~0.86 | P 0.45 | R 0.92 | area ratio 2.0")
print("  AFTER  (Tversky a=0.7):     see above  — precision nearly doubled")

CALCIFICATION — OFFICIAL CBIS TEST SET  (n=326)
  threshold tuned on validation: 0.5
  Dice       mean 0.8791   median 0.9024   std 0.0835
  IoU        mean 0.7928
  Precision  mean 0.8532
  Recall     mean 0.9183
  pred/GT area ratio: 1.08   (1.0 = predictions same size as ground truth)

  distribution:
    0.00-0.50:    2 (  1%) 
    0.50-0.70:   11 (  3%) 
    0.70-0.85:   55 ( 17%) ████
    0.85-1.01:  258 ( 79%) ██████████████████████

  Dice by radiologist subtlety (1 = hardest to see):
    subtlety 1: Dice 0.8649   n=24
    subtlety 2: Dice 0.8893   n=56
    subtlety 3: Dice 0.8901   n=104
    subtlety 4: Dice 0.8708   n=76
    subtlety 5: Dice 0.8678   n=66

  Dice by pathology:
    BENIGN                  Dice 0.8997   n=130
    BENIGN_WITHOUT_CALLBACK Dice 0.8566   n=67
    MALIGNANT               Dice 0.8701   n=129

  worst 10 Dice: [0.405, 0.452, 0.509, 0.519, 0.538, 0.548, 0.602, 0.637, 0.655, 0.655]
  BEFORE (weight=[1,2] loss): Dice ~0.86 | P 0.45 | R 0.92 | area ratio 

In [7]:
# ══════════════════════════════════════════════════════════════════════
# TEST ONLY — calcification, Tversky model, official CBIS test set
#   Loads seg_calc_tversky.pth (threshold stored inside the checkpoint)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
te=P[(P.split=="test")&(P.abn_type=="calcification")].reset_index(drop=True)
tr=P[(P.split=="train")&(P.abn_type=="calcification")]
assert len(set(tr.patient_id)&set(te.patient_id))==0, "LEAKAGE"
print("CALC TEST n="+str(len(te))+" | patients "+str(te.patient_id.nunique()))

# ---------- preprocessing (identical to training) ----------
def p0(im): return im
def p1(im): return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im): return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im): return cv2.equalizeHist(im)
def p4(im): return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im): return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im): return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]

def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3)
    g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)

class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); v=s.value_stream(z); a=s.advantage_stream(z)
        return v+a-a.mean(1,keepdim=True)

dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()

@torch.no_grad()
def rl_act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32)
    g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))

# ---------- model ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

ck=torch.load(os.path.join(D,"seg_calc_tversky.pth"),map_location=DEV)
sd=ck["state_dict"] if isinstance(ck,dict) and "state_dict" in ck else ck
THR=float(ck.get("threshold",0.5)) if isinstance(ck,dict) else 0.5
net=AttnDueling().to(DEV); net.load_state_dict(sd); net.eval()
print("model loaded | threshold (tuned on val) = "+format(THR,".2f")+"\n")

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        return (torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

@torch.no_grad()
def run(thr):
    net.eval(); rec=[]
    for x,y,idx in DataLoader(DS(te),batch_size=16,shuffle=False,num_workers=0):
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        prob=torch.softmax(o.float(),1)[:,1]
        pr=(prob>thr).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r=te.iloc[int(idx[i])]
            rec.append(dict(patient=r["patient_id"],
                dice=(2*tp+1)/(2*tp+fp+fn+1),
                iou=(tp+1)/(tp+fp+fn+1),
                precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                pred_area=pi.mean().item(), gt_area=ti.mean().item(),
                subtlety=r.get("subtlety",np.nan), pathology=str(r.get("pathology",""))))
    return pd.DataFrame(rec)

R  = run(THR)
R05= run(0.50)
R.to_csv(os.path.join(D,"seg_calc_tversky_test.csv"),index=False)

L=[]
L.append("="*70)
L.append("CALCIFICATION — OFFICIAL CBIS-DDSM TEST SET   (n="+str(len(R))+")")
L.append("  Tversky loss (alpha=0.7 on FP) | threshold "+format(THR,".2f")+" tuned on validation")
L.append("="*70)
L.append("  Dice        mean "+format(R.dice.mean(),".4f")+
         "   median "+format(R.dice.median(),".4f")+
         "   std "+format(R.dice.std(),".4f"))
L.append("  IoU         mean "+format(R.iou.mean(),".4f")+
         "   median "+format(R.iou.median(),".4f"))
L.append("  Precision   mean "+format(R.precision.mean(),".4f"))
L.append("  Recall      mean "+format(R.recall.mean(),".4f"))
L.append("  pred/GT area ratio "+format(R.pred_area.mean()/max(R.gt_area.mean(),1e-9),".2f")+
         "   (1.00 = same size as ground truth)")
L.append("")
L.append("  Dice distribution:")
for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
    n=int(((R.dice>=lo)&(R.dice<hi)).sum())
    L.append("    "+format(lo,".2f")+"-"+format(hi,".2f")+":"+str(n).rjust(5)+
             " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"#"*int(28*n/len(R)))
if R.subtlety.notna().any():
    L.append("")
    L.append("  Dice by radiologist subtlety (1 = hardest to see):")
    for s,g in R.dropna(subset=["subtlety"]).groupby("subtlety"):
        L.append("    subtlety "+str(int(s))+":  Dice "+format(g.dice.mean(),".4f")+"   n="+str(len(g)))
L.append("")
L.append("  Dice by pathology:")
for p,g in R.groupby("pathology"):
    L.append("    "+str(p).ljust(26)+"Dice "+format(g.dice.mean(),".4f")+"   n="+str(len(g)))
L.append("")
L.append("  worst 10 Dice: "+str([round(x,3) for x in sorted(R.dice)[:10]]))
L.append("")
L.append("-"*70)
L.append("  ABLATION — effect of the loss change")
L.append("-"*70)
L.append("  old loss (weighted CE+Dice, w=[1,2]) : Dice ~0.86 | P 0.45 | R 0.92 | area 2.00")
L.append("  Tversky @ thr 0.50                   : Dice "+format(R05.dice.mean(),".4f")+
         " | P "+format(R05.precision.mean(),".3f")+
         " | R "+format(R05.recall.mean(),".3f")+
         " | area "+format(R05.pred_area.mean()/max(R05.gt_area.mean(),1e-9),".2f"))
L.append("  Tversky @ thr "+format(THR,".2f")+" (tuned)          : Dice "+format(R.dice.mean(),".4f")+
         " | P "+format(R.precision.mean(),".3f")+
         " | R "+format(R.recall.mean(),".3f")+
         " | area "+format(R.pred_area.mean()/max(R.gt_area.mean(),1e-9),".2f"))
L.append("="*70)

txt="\n".join(L)
print(txt)
with open(os.path.join(D,"calc_test_summary.txt"),"w") as f: f.write(txt)
print("\nsaved seg_calc_tversky_test.csv  +  calc_test_summary.txt")

# ---------- figure ----------
N=min(8,len(te))
fig,ax=plt.subplots(N,4,figsize=(13,3.1*N))
if N==1: ax=ax.reshape(1,4)
with torch.no_grad():
    for k in range(N):
        r=te.iloc[k*max(1,len(te)//N)]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gt=(msk>127).astype(np.uint8)
        x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[0,1]>THR).cpu().numpy().astype(np.uint8)
        dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[(gt>0)&(pr==0)]=[0,150,0]; ov[(pr>0)&(gt==0)]=[200,0,0]; ov[(pr>0)&(gt>0)]=[230,210,0]
        for c,(im,t,cm) in enumerate([(img,str(r["pathology"])[:12],"gray"),(gt,"GT","gray"),
                                      (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
            ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
plt.suptitle("Calcification (Tversky) — yellow=correct, green=missed, red=false positive",fontsize=11)
plt.tight_layout()
o=os.path.join(D,"figures","calc_tversky_test.png"); os.makedirs(os.path.dirname(o),exist_ok=True)
plt.savefig(o,dpi=140,bbox_inches="tight"); plt.close()
print("saved "+o)

CALC TEST n=326 | patients 151
model loaded | threshold (tuned on val) = 0.50

CALCIFICATION — OFFICIAL CBIS-DDSM TEST SET   (n=326)
  Tversky loss (alpha=0.7 on FP) | threshold 0.50 tuned on validation
  Dice        mean 0.8791   median 0.9024   std 0.0835
  IoU         mean 0.7928   median 0.8222
  Precision   mean 0.8532
  Recall      mean 0.9183
  pred/GT area ratio 1.08   (1.00 = same size as ground truth)

  Dice distribution:
    0.00-0.50:    2 (  1%) 
    0.50-0.70:   11 (  3%) 
    0.70-0.85:   55 ( 17%) ####
    0.85-1.01:  258 ( 79%) ######################

  Dice by radiologist subtlety (1 = hardest to see):
    subtlety 1:  Dice 0.8649   n=24
    subtlety 2:  Dice 0.8893   n=56
    subtlety 3:  Dice 0.8901   n=104
    subtlety 4:  Dice 0.8708   n=76
    subtlety 5:  Dice 0.8678   n=66

  Dice by pathology:
    BENIGN                    Dice 0.8997   n=130
    BENIGN_WITHOUT_CALLBACK   Dice 0.8566   n=67
    MALIGNANT                 Dice 0.8701   n=129

  worst 10 Dice: [

In [2]:
# ══════════════════════════════════════════════════════════════════════
# VISUALIZE calcification test results (from saved CSV + model)
#   Row groups: BEST cases | WORST cases | mid-range
#   Plus: Dice histogram, Dice-vs-subtlety bar, precision/recall scatter
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu"); IMG=256

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
te=P[(P.split=="test")&(P.abn_type=="calcification")].reset_index(drop=True)
R=pd.read_csv(os.path.join(D,"seg_calc_tversky_test.csv"))
te=te.iloc[:len(R)].copy()
for c in ["dice","precision","recall"]: te[c]=R[c].values

# preprocessing + model (same as before)
def p0(im):return im
def p1(im):return cv2.createCLAHE(2.0,(8,8)).apply(im)
def p2(im):return cv2.createCLAHE(3.0,(8,8)).apply(im)
def p3(im):return cv2.equalizeHist(im)
def p4(im):return np.clip(((im/255.)**0.7)*255,0,255).astype(np.uint8)
def p5(im):return np.clip(((im/255.)**1.4)*255,0,255).astype(np.uint8)
def p6(im):return cv2.createCLAHE(2.0,(8,8)).apply(cv2.medianBlur(im,3))
def p7(im):return cv2.createCLAHE(2.0,(8,8)).apply(cv2.bilateralFilter(im,5,50,5))
def p8(im):
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15))
    return np.clip(cv2.add(im,cv2.morphologyEx(im,cv2.MORPH_TOPHAT,k)),0,255).astype(np.uint8)
PIPES=[p0,p1,p2,p3,p4,p5,p6,p7,p8]
def feats(im):
    x=im.astype(np.float32)/255.
    h=cv2.calcHist([im],[0],None,[32],[0,256]).ravel(); h=h/(h.sum()+1e-9)
    ent=float(-(h[h>0]*np.log2(h[h>0])).sum())
    gx=cv2.Sobel(im,cv2.CV_32F,1,0,3); gy=cv2.Sobel(im,cv2.CV_32F,0,1,3); g=np.sqrt(gx**2+gy**2)
    return np.array([x.mean(),x.std(),x.min(),x.max(),np.median(x),
                     float(np.percentile(x,25)),float(np.percentile(x,75)),
                     ent/5.0,g.mean()/255.,g.std()/255.,
                     float((x>0.8).mean()),float((x<0.2).mean())],dtype=np.float32)
class DuelingDQN(nn.Module):
    def __init__(s,ind=12,na=9,h=128):
        super().__init__()
        s.feature_extractor=nn.Sequential(nn.Linear(ind,h),nn.LayerNorm(h),nn.ReLU(),
                                          nn.Dropout(0.1),nn.Linear(h,h),nn.LayerNorm(h),nn.ReLU())
        s.value_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,1))
        s.advantage_stream=nn.Sequential(nn.Linear(h,64),nn.ReLU(),nn.Linear(64,na))
    def forward(s,x):
        z=s.feature_extractor(x); return s.value_stream(z)+s.advantage_stream(z)-s.advantage_stream(z).mean(1,keepdim=True)
dqn=DuelingDQN().to(DEV)
_ck=torch.load(os.path.join(D,"rl_200_test","dueling_dqn_200_test.pth"),map_location=DEV)
if isinstance(_ck,dict) and "state_dict" in _ck: _ck=_ck["state_dict"]
dqn.load_state_dict(_ck,strict=True); dqn.eval()
@torch.no_grad()
def rl_act(im): return int(dqn(torch.from_numpy(feats(im)).unsqueeze(0).to(DEV)).argmax(1).item())
def boost(im):
    f=im.astype(np.float32); g1=cv2.GaussianBlur(f,(0,0),1.0); g2=cv2.GaussianBlur(f,(0,0),3.0)
    sh=np.clip(f+0.7*(f-g1)+0.4*(f-g2),0,255).astype(np.uint8)
    k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9))
    return np.clip(cv2.add(sh,(0.35*cv2.morphologyEx(sh,cv2.MORPH_TOPHAT,k)).astype(np.uint8)),0,255).astype(np.uint8)
def prep(im): return boost(PIPES[rl_act(im)](im))
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.v(d1)+s.adv(d1)-s.adv(d1).mean(1,keepdim=True)
ck=torch.load(os.path.join(D,"seg_calc_tversky.pth"),map_location=DEV)
sd=ck["state_dict"] if isinstance(ck,dict) and "state_dict" in ck else ck
THR=float(ck.get("threshold",0.5)) if isinstance(ck,dict) else 0.5
net=AttnDueling().to(DEV); net.load_state_dict(sd); net.eval()

def draw(cases,title,fname):
    N=len(cases)
    fig,ax=plt.subplots(N,4,figsize=(13,3.0*N))
    if N==1: ax=ax.reshape(1,4)
    with torch.no_grad():
        for k in range(N):
            r=cases.iloc[k]
            img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(msk>127).astype(np.uint8)
            x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=(torch.softmax(o.float(),1)[0,1]>THR).cpu().numpy().astype(np.uint8)
            ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
            ov[(gt>0)&(pr==0)]=[0,150,0]; ov[(pr>0)&(gt==0)]=[210,0,0]; ov[(pr>0)&(gt>0)]=[235,215,0]
            ttl=str(r["patient_id"])+" | sub="+str(r.get("subtlety","?"))+" | "+str(r.get("pathology",""))[:10]
            for c,(im,t,cm) in enumerate([(img,ttl,"gray"),(gt,"GT","gray"),
                    (pr,"pred","gray"),(ov,"Dice="+format(r["dice"],".2f")+" P="+format(r["precision"],".2f")+" R="+format(r["recall"],".2f"),None)]):
                ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
                ax[k,c].set_title(t,fontsize=7); ax[k,c].axis("off")
    plt.suptitle(title+"   (yellow=correct, green=missed GT, red=false positive)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures",fname); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)

draw(te.nlargest(6,"dice").reset_index(drop=True), "BEST calcification cases", "calc_best.png")
draw(te.nsmallest(6,"dice").reset_index(drop=True), "WORST calcification cases", "calc_worst.png")
mid=te[(te.dice>0.82)&(te.dice<0.90)].head(6).reset_index(drop=True)
if len(mid): draw(mid, "TYPICAL calcification cases", "calc_typical.png")

# summary panels
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
ax[0].hist(te.dice,bins=20,color="#3b7",edgecolor="k")
ax[0].axvline(te.dice.mean(),color="r",ls="--",label="mean "+format(te.dice.mean(),".3f"))
ax[0].axvline(te.dice.median(),color="b",ls="--",label="median "+format(te.dice.median(),".3f"))
ax[0].set_title("Dice distribution (n="+str(len(te))+")"); ax[0].set_xlabel("Dice"); ax[0].legend()
sub=te.dropna(subset=["subtlety"]).groupby("subtlety").dice.mean()
ax[1].bar(sub.index.astype(int).astype(str),sub.values,color="#59f",edgecolor="k")
ax[1].set_ylim(0.7,1.0); ax[1].set_title("Dice by subtlety (1=hardest)"); ax[1].set_xlabel("subtlety")
ax[2].scatter(te.recall,te.precision,c=te.dice,cmap="viridis",s=18)
ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision"); ax[2].set_title("precision vs recall (color=Dice)")
ax[2].plot([0,1],[0,1],"k:",alpha=.3)
plt.tight_layout()
o=os.path.join(D,"figures","calc_summary_panels.png")
plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)
print("\nDone. Open calc_best.png, calc_worst.png, calc_typical.png, calc_summary_panels.png")

saved /root/autodl-tmp/CBIS/figures/calc_best.png
saved /root/autodl-tmp/CBIS/figures/calc_worst.png
saved /root/autodl-tmp/CBIS/figures/calc_typical.png
saved /root/autodl-tmp/CBIS/figures/calc_summary_panels.png

Done. Open calc_best.png, calc_worst.png, calc_typical.png, calc_summary_panels.png


In [3]:
# ══════════════════════════════════════════════════════════════════════
# CALCIFICATION — balanced precision/recall + fixed CLAHE
#   Loss: symmetric Tversky (a=b=0.5) + explicit |FP-FN| balance penalty
#   Preprocessing: CLAHE on ALL train + test images (no RL selection here)
#   Threshold tuned on VAL, applied once to TEST.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3
BAL_LAMBDA=0.5     # strength of the |FP-FN| balance penalty
torch.backends.cudnn.benchmark=True

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
C=P[P.abn_type=="calcification"]
tr=C[C.split=="train"].reset_index(drop=True)
va=C[C.split=="val"].reset_index(drop=True)
te=C[C.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("CALC | train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te)))
print("loss: symmetric Tversky + "+str(BAL_LAMBDA)+"*|FP-FN|  |  preprocessing: CLAHE (all)\n")

# ---------- CLAHE preprocessing on ALL images ----------
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        img=prep(img)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

# ---------- model (Attention U-Net + dueling head) ----------
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i))
        s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

def loss_fn(lo,t):
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tversky=1-((tp+1)/(tp+0.5*fp+0.5*fn+1)).mean()          # symmetric = balanced
    balance=(torch.abs(fp-fn)/(tp+fp+fn+1)).mean()          # punish P/R imbalance
    ce=F.cross_entropy(lo.float(),t)
    return 0.3*ce + 0.7*tversky + BAL_LAMBDA*balance

tl=DataLoader(DS(tr),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va),batch_size=BATCH,shuffle=False,num_workers=0)
sl=DataLoader(DS(te),batch_size=BATCH,shuffle=False,num_workers=0)
net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
opt=torch.optim.Adam(net.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)

@torch.no_grad()
def evaluate(loader,thr=0.5,df=None):
    net.eval(); rec=[]
    for x,y,idx in loader:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        prob=torch.softmax(o.float(),1)[:,1]; pr=(prob>thr).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            d=dict(dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                   precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                   pred_area=pi.mean().item(), gt_area=ti.mean().item())
            if df is not None:
                r=df.iloc[int(idx[i])]
                d.update(patient_id=r["patient_id"], subtlety=r.get("subtlety",np.nan),
                         pathology=r.get("pathology",""))
            rec.append(d)
    return pd.DataFrame(rec)

best,bs,ni=0.,None,0
for ep in range(1,EPOCHS+1):
    net.train(); tot=0.; nb=0
    for x,y,_ in tl:
        x=x.to(DEV); y=y.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        tot+=l.item(); nb+=1
    V=evaluate(vl); d=V.dice.mean(); sch.step(d)
    if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
    else: ni+=1
    print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
          " | val-Dice "+format(d,".4f")+
          " | P "+format(V.precision.mean(),".3f")+
          " | R "+format(V.recall.mean(),".3f")+
          " | |P-R| "+format(abs(V.precision.mean()-V.recall.mean()),".3f")+
          (" *" if d==best else ""))
    if ni>=10: print("  early stop"); break
net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})

# threshold that MINIMISES |precision-recall| on validation
print("\nthreshold sweep on VAL (looking for balanced P/R):")
best_t, best_gap = 0.5, 9.9
for t in [0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]:
    V=evaluate(vl,thr=t)
    gap=abs(V.precision.mean()-V.recall.mean())
    print("  thr "+format(t,".2f")+" | Dice "+format(V.dice.mean(),".4f")+
          " | P "+format(V.precision.mean(),".3f")+" | R "+format(V.recall.mean(),".3f")+
          " | |P-R| "+format(gap,".3f"))
    if gap<best_gap: best_gap, best_t = gap, t
print("chosen threshold (most balanced P/R): "+format(best_t,".2f"))

torch.save({"state_dict":{k:v.cpu() for k,v in net.state_dict().items()},"threshold":best_t},
           os.path.join(D,"seg_calc_balanced.pth"))

R=evaluate(sl,thr=best_t,df=te)
R.to_csv(os.path.join(D,"seg_calc_balanced_test.csv"),index=False)

out=[]
out.append("="*66)
out.append("CALCIFICATION — balanced loss + CLAHE  |  OFFICIAL TEST (n="+str(len(R))+")")
out.append("  threshold "+format(best_t,".2f")+" (chosen for balanced P/R on val)")
out.append("="*66)
out.append("  Dice       mean "+format(R.dice.mean(),".4f")+"  median "+format(R.dice.median(),".4f")+
           "  std "+format(R.dice.std(),".4f"))
out.append("  IoU        mean "+format(R.iou.mean(),".4f"))
out.append("  Precision  mean "+format(R.precision.mean(),".4f"))
out.append("  Recall     mean "+format(R.recall.mean(),".4f"))
out.append("  |P - R|         "+format(abs(R.precision.mean()-R.recall.mean()),".4f")+"   (smaller = more balanced)")
out.append("  pred/GT area    "+format(R.pred_area.mean()/max(R.gt_area.mean(),1e-9),".2f"))
out.append("")
out.append("  distribution:")
for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
    n=int(((R.dice>=lo)&(R.dice<hi)).sum())
    out.append("    "+format(lo,".2f")+"-"+format(hi,".2f")+":"+str(n).rjust(5)+
               " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"#"*int(28*n/len(R)))
out.append("")
out.append("  COMPARISON:")
out.append("    old weighted CE+Dice : Dice ~0.86 | P 0.45 | R 0.92 | |P-R| 0.47")
out.append("    Tversky a=0.7        : Dice 0.879 | P 0.85 | R 0.92 | |P-R| 0.07")
out.append("    this (balanced+CLAHE): Dice "+format(R.dice.mean(),".4f")+
           " | P "+format(R.precision.mean(),".3f")+" | R "+format(R.recall.mean(),".3f")+
           " | |P-R| "+format(abs(R.precision.mean()-R.recall.mean()),".3f"))
out.append("="*66)
txt="\n".join(out); print("\n"+txt)
with open(os.path.join(D,"calc_balanced_summary.txt"),"w") as f: f.write(txt)
print("\nsaved seg_calc_balanced.pth + seg_calc_balanced_test.csv + calc_balanced_summary.txt")

CALC | train 1283 | val 257 | test 326
loss: symmetric Tversky + 0.5*|FP-FN|  |  preprocessing: CLAHE (all)

  ep  1/60 | loss 0.3330 | val-Dice 0.8442 | P 0.809 | R 0.913 | |P-R| 0.104 *
  ep  2/60 | loss 0.2608 | val-Dice 0.8471 | P 0.822 | R 0.904 | |P-R| 0.082 *
  ep  3/60 | loss 0.2565 | val-Dice 0.8466 | P 0.827 | R 0.895 | |P-R| 0.068
  ep  4/60 | loss 0.2542 | val-Dice 0.8480 | P 0.820 | R 0.907 | |P-R| 0.087 *
  ep  5/60 | loss 0.2560 | val-Dice 0.8449 | P 0.820 | R 0.901 | |P-R| 0.081
  ep  6/60 | loss 0.2527 | val-Dice 0.8478 | P 0.825 | R 0.901 | |P-R| 0.075
  ep  7/60 | loss 0.2520 | val-Dice 0.8478 | P 0.824 | R 0.903 | |P-R| 0.079
  ep  8/60 | loss 0.2507 | val-Dice 0.8483 | P 0.822 | R 0.906 | |P-R| 0.083 *
  ep  9/60 | loss 0.2526 | val-Dice 0.8482 | P 0.821 | R 0.907 | |P-R| 0.086
  ep 10/60 | loss 0.2515 | val-Dice 0.8482 | P 0.822 | R 0.906 | |P-R| 0.084
  ep 11/60 | loss 0.2509 | val-Dice 0.8477 | P 0.819 | R 0.905 | |P-R| 0.086
  ep 12/60 | loss 0.2534 | val-Dice 

In [4]:
# ══════════════════════════════════════════════════════════════════════
# VISUALIZE — calcification balanced+CLAHE results
#   best / worst / typical case sheets + summary panels
#   uses seg_calc_balanced.pth + seg_calc_balanced_test.csv (CLAHE preproc)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu"); IMG=256

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
te=P[(P.split=="test")&(P.abn_type=="calcification")].reset_index(drop=True)
R=pd.read_csv(os.path.join(D,"seg_calc_balanced_test.csv"))
te=te.iloc[:len(R)].copy()
for c in ["dice","precision","recall"]:
    if c in R: te[c]=R[c].values

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

ck=torch.load(os.path.join(D,"seg_calc_balanced.pth"),map_location=DEV)
sd=ck["state_dict"] if isinstance(ck,dict) and "state_dict" in ck else ck
THR=float(ck.get("threshold",0.5)) if isinstance(ck,dict) else 0.5
net=AttnDueling().to(DEV); net.load_state_dict(sd); net.eval()
print("model loaded | threshold "+format(THR,".2f"))

def draw(cases,title,fname):
    N=len(cases)
    if N==0: return
    fig,ax=plt.subplots(N,4,figsize=(13,3.0*N))
    if N==1: ax=ax.reshape(1,4)
    with torch.no_grad():
        for k in range(N):
            r=cases.iloc[k]
            img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            gt=(msk>127).astype(np.uint8)
            x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
            with torch.amp.autocast(device_type="cuda"): o=net(x)
            pr=(torch.softmax(o.float(),1)[0,1]>THR).cpu().numpy().astype(np.uint8)
            ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
            ov[(gt>0)&(pr==0)]=[0,150,0]; ov[(pr>0)&(gt==0)]=[210,0,0]; ov[(pr>0)&(gt>0)]=[235,215,0]
            ttl=str(r.get("patient_id","?"))+" | sub="+str(r.get("subtlety","?"))+" | "+str(r.get("pathology",""))[:10]
            met="Dice="+format(r["dice"],".2f")+" P="+format(r["precision"],".2f")+" R="+format(r["recall"],".2f")
            for c,(im,t,cm) in enumerate([(img,ttl,"gray"),(gt,"GT","gray"),(pr,"pred","gray"),(ov,met,None)]):
                ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
                ax[k,c].set_title(t,fontsize=7); ax[k,c].axis("off")
    plt.suptitle(title+"   (yellow=correct, green=missed GT, red=false positive)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures",fname); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)

draw(te.nlargest(6,"dice").reset_index(drop=True),  "BEST — balanced+CLAHE",    "calcbal_best.png")
draw(te.nsmallest(6,"dice").reset_index(drop=True), "WORST — balanced+CLAHE",   "calcbal_worst.png")
mid=te[(te.dice>0.82)&(te.dice<0.90)].head(6).reset_index(drop=True)
draw(mid, "TYPICAL — balanced+CLAHE", "calcbal_typical.png")

# summary panels
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
ax[0].hist(te.dice,bins=20,color="#3b7",edgecolor="k")
ax[0].axvline(te.dice.mean(),color="r",ls="--",label="mean "+format(te.dice.mean(),".3f"))
ax[0].axvline(te.dice.median(),color="b",ls="--",label="median "+format(te.dice.median(),".3f"))
ax[0].set_title("Dice distribution (n="+str(len(te))+")"); ax[0].set_xlabel("Dice"); ax[0].legend()
if "subtlety" in te and te.subtlety.notna().any():
    sub=te.dropna(subset=["subtlety"]).groupby("subtlety").dice.mean()
    ax[1].bar(sub.index.astype(int).astype(str),sub.values,color="#59f",edgecolor="k")
    ax[1].set_ylim(0.7,1.0); ax[1].set_title("Dice by subtlety (1=hardest)"); ax[1].set_xlabel("subtlety")
ax[2].scatter(te.recall,te.precision,c=te.dice,cmap="viridis",s=18)
ax[2].plot([0,1],[0,1],"k:",alpha=.3); ax[2].set_xlim(0,1); ax[2].set_ylim(0,1)
ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision")
ax[2].set_title("precision vs recall  (balanced: points hug diagonal)")
plt.tight_layout()
o=os.path.join(D,"figures","calcbal_summary_panels.png")
plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)
print("\nDone: calcbal_best / calcbal_worst / calcbal_typical / calcbal_summary_panels")

model loaded | threshold 0.70
saved /root/autodl-tmp/CBIS/figures/calcbal_best.png
saved /root/autodl-tmp/CBIS/figures/calcbal_worst.png
saved /root/autodl-tmp/CBIS/figures/calcbal_typical.png
saved /root/autodl-tmp/CBIS/figures/calcbal_summary_panels.png

Done: calcbal_best / calcbal_worst / calcbal_typical / calcbal_summary_panels


In [5]:
# ══════════════════════════════════════════════════════════════════════
# CALCIFICATION — per-image adaptive threshold + optional shape/mining
#   Switches at top. Start with everything OFF except ADAPTIVE_THR.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3
torch.backends.cudnn.benchmark=True

# ── SWITCHES ───────────────────────────────────────────────────────
ADAPTIVE_THR = True     # per-image Otsu threshold at test  (recommended)
USE_CLDICE   = False    # add skeleton (thin-structure) loss  (optional)
USE_MINING   = False    # hard-example mining on low-Dice     (optional)
TV_ALPHA, TV_BETA = 0.7, 0.3   # 0.7 on FP = punish over-segmentation
CLDICE_W = 0.3
# ───────────────────────────────────────────────────────────────────

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
C=P[P.abn_type=="calcification"]
tr=C[C.split=="train"].reset_index(drop=True)
va=C[C.split=="val"].reset_index(drop=True)
te=C[C.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("CALC | train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te)))
print("switches: adaptive_thr="+str(ADAPTIVE_THR)+" clDice="+str(USE_CLDICE)+" mining="+str(USE_MINING)+
      " | Tversky a="+str(TV_ALPHA)+" b="+str(TV_BETA)+"\n")

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        img=prep(img)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

def soft_skeleton(p, iters=6):
    # differentiable skeleton (for clDice) via iterative min/max pooling
    for _ in range(iters):
        er=-F.max_pool2d(-p,3,1,1)
        op=F.max_pool2d(er,3,1,1)
        p=p-F.relu(p-op)
    return p
def cldice(p,g):
    sp=soft_skeleton(p); sg=soft_skeleton(g)
    tprec=((sp*g).sum((2,3))+1)/(sp.sum((2,3))+1)
    tsens=((sg*p).sum((2,3))+1)/(sg.sum((2,3))+1)
    return 1-(2*tprec*tsens/(tprec+tsens)).mean()

def base_loss(lo,t,reduce=True):
    ce=F.cross_entropy(lo.float(),t,reduction="none").mean((1,2))
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tv=1-((tp+1)/(tp+TV_ALPHA*fp+TV_BETA*fn+1))
    per=0.3*ce+0.7*tv
    if USE_CLDICE:
        per=per+CLDICE_W*cldice(p.unsqueeze(1),g.unsqueeze(1))
    return per if not reduce else per.mean()

tl=DataLoader(DS(tr),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va),batch_size=BATCH,shuffle=False,num_workers=0)
sl=DataLoader(DS(te),batch_size=BATCH,shuffle=False,num_workers=0)
net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
opt=torch.optim.Adam(net.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)

def otsu_thr(prob):
    # Otsu on a single image's probability map (numpy, 0..1)
    q=(prob*255).astype(np.uint8)
    t,_=cv2.threshold(q,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    return max(0.30,min(0.80,t/255.0))    # clamp to a sane band

@torch.no_grad()
def evaluate(loader,fixed_thr=0.5,df=None):
    net.eval(); rec=[]
    for x,y,idx in loader:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        prob=torch.softmax(o.float(),1)[:,1]
        for i in range(prob.size(0)):
            pm=prob[i].cpu().numpy()
            thr=otsu_thr(pm) if ADAPTIVE_THR else fixed_thr
            pi=(prob[i]>thr).float(); ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            d=dict(dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                   precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                   pred_area=pi.mean().item(), gt_area=ti.mean().item(), thr=thr)
            if df is not None:
                r=df.iloc[int(idx[i])]
                d.update(patient_id=r["patient_id"], subtlety=r.get("subtlety",np.nan),
                         pathology=r.get("pathology",""))
            rec.append(d)
    return pd.DataFrame(rec)

best,bs,ni=0.,None,0
for ep in range(1,EPOCHS+1):
    net.train(); tot=0.; nb=0
    for x,y,_ in tl:
        x=x.to(DEV); y=y.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"):
            o=net(x)
            if USE_MINING and ep>10:
                per=base_loss(o,y,reduce=False)
                k=max(1,int(0.7*per.numel()))          # hardest 70%
                l=torch.topk(per,k).values.mean()
            else:
                l=base_loss(o,y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        tot+=l.item(); nb+=1
    V=evaluate(vl); d=V.dice.mean(); sch.step(d)
    if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
    else: ni+=1
    print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
          " | val-Dice "+format(d,".4f")+" | P "+format(V.precision.mean(),".3f")+
          " | R "+format(V.recall.mean(),".3f")+(" *" if d==best else ""))
    if ni>=10: print("  early stop"); break
net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
torch.save({"state_dict":{k:v.cpu() for k,v in net.state_dict().items()},
            "adaptive":ADAPTIVE_THR}, os.path.join(D,"seg_calc_adaptive.pth"))

R=evaluate(sl,df=te); R.to_csv(os.path.join(D,"seg_calc_adaptive_test.csv"),index=False)
out=[]
out.append("="*66)
out.append("CALCIFICATION — adaptive threshold"+(" +clDice" if USE_CLDICE else "")+
           (" +mining" if USE_MINING else "")+"  |  TEST (n="+str(len(R))+")")
out.append("="*66)
out.append("  Dice       mean "+format(R.dice.mean(),".4f")+"  median "+format(R.dice.median(),".4f")+
           "  std "+format(R.dice.std(),".4f"))
out.append("  IoU        mean "+format(R.iou.mean(),".4f"))
out.append("  Precision  mean "+format(R.precision.mean(),".4f"))
out.append("  Recall     mean "+format(R.recall.mean(),".4f"))
out.append("  |P - R|         "+format(abs(R.precision.mean()-R.recall.mean()),".4f"))
out.append("  pred/GT area    "+format(R.pred_area.mean()/max(R.gt_area.mean(),1e-9),".2f"))
out.append("  per-image threshold: mean "+format(R.thr.mean(),".2f")+
           " range ["+format(R.thr.min(),".2f")+", "+format(R.thr.max(),".2f")+"]")
out.append("")
for lo,hi in [(0,.5),(.5,.7),(.7,.85),(.85,1.01)]:
    n=int(((R.dice>=lo)&(R.dice<hi)).sum())
    out.append("    "+format(lo,".2f")+"-"+format(hi,".2f")+":"+str(n).rjust(5)+
               " ("+str(round(100*n/len(R))).rjust(3)+"%) "+"#"*int(28*n/len(R)))
out.append("")
out.append("  vs Tversky a=0.7 (fixed thr): Dice 0.879 | P 0.85 | R 0.92")
out.append("  vs balanced+CLAHE           : Dice 0.876 | P 0.87 | R 0.89 | |P-R| 0.014")
out.append("="*66)
txt="\n".join(out); print("\n"+txt)
with open(os.path.join(D,"calc_adaptive_summary.txt"),"w") as f: f.write(txt)
print("\nsaved seg_calc_adaptive.pth + seg_calc_adaptive_test.csv + calc_adaptive_summary.txt")

CALC | train 1283 | val 257 | test 326
switches: adaptive_thr=True clDice=False mining=False | Tversky a=0.7 b=0.3

  ep  1/60 | loss 0.2735 | val-Dice 0.8456 | P 0.820 | R 0.902 *
  ep  2/60 | loss 0.2130 | val-Dice 0.8461 | P 0.832 | R 0.889 *
  ep  3/60 | loss 0.2050 | val-Dice 0.8409 | P 0.848 | R 0.860
  ep  4/60 | loss 0.2057 | val-Dice 0.8457 | P 0.838 | R 0.880
  ep  5/60 | loss 0.2028 | val-Dice 0.8470 | P 0.827 | R 0.896 *
  ep  6/60 | loss 0.2005 | val-Dice 0.8414 | P 0.845 | R 0.866
  ep  7/60 | loss 0.2009 | val-Dice 0.8482 | P 0.819 | R 0.910 *
  ep  8/60 | loss 0.1982 | val-Dice 0.8454 | P 0.828 | R 0.891
  ep  9/60 | loss 0.1999 | val-Dice 0.8479 | P 0.838 | R 0.887
  ep 10/60 | loss 0.1997 | val-Dice 0.8502 | P 0.824 | R 0.907 *
  ep 11/60 | loss 0.1981 | val-Dice 0.8266 | P 0.820 | R 0.865
  ep 12/60 | loss 0.1983 | val-Dice 0.8490 | P 0.834 | R 0.894
  ep 13/60 | loss 0.1959 | val-Dice 0.8486 | P 0.837 | R 0.889
  ep 14/60 | loss 0.1946 | val-Dice 0.8511 | P 0.830 | 

In [6]:
# ══════════════════════════════════════════════════════════════════════
# VISUALIZE — calcification adaptive-threshold results
#   best / worst / typical sheets + summary panels
#   uses seg_calc_adaptive.pth + seg_calc_adaptive_test.csv
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu"); IMG=256

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
te=P[(P.split=="test")&(P.abn_type=="calcification")].reset_index(drop=True)
R=pd.read_csv(os.path.join(D,"seg_calc_adaptive_test.csv"))
te=te.iloc[:len(R)].copy()
for c in ["dice","precision","recall","thr"]:
    if c in R: te[c]=R[c].values

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)
def otsu_thr(pm):
    q=(pm*255).astype(np.uint8)
    t,_=cv2.threshold(q,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    return max(0.30,min(0.80,t/255.0))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

ck=torch.load(os.path.join(D,"seg_calc_adaptive.pth"),map_location=DEV)
sd=ck["state_dict"] if isinstance(ck,dict) and "state_dict" in ck else ck
net=AttnDueling().to(DEV); net.load_state_dict(sd); net.eval()
print("model loaded (per-image adaptive threshold)")

def predict(img):
    x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
        pm=torch.softmax(net(x).float(),1)[0,1].cpu().numpy()
    thr=otsu_thr(pm)
    return (pm>thr).astype(np.uint8), thr

def draw(cases,title,fname):
    N=len(cases)
    if N==0: return
    fig,ax=plt.subplots(N,4,figsize=(13,3.0*N))
    if N==1: ax=ax.reshape(1,4)
    for k in range(N):
        r=cases.iloc[k]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gt=(msk>127).astype(np.uint8)
        pr,thr=predict(img)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[(gt>0)&(pr==0)]=[0,150,0]; ov[(pr>0)&(gt==0)]=[210,0,0]; ov[(pr>0)&(gt>0)]=[235,215,0]
        ttl=str(r.get("patient_id","?"))+" | sub="+str(r.get("subtlety","?"))+" | "+str(r.get("pathology",""))[:10]
        met="Dice="+format(r["dice"],".2f")+" P="+format(r["precision"],".2f")+" R="+format(r["recall"],".2f")+" thr="+format(thr,".2f")
        for c,(im,t,cm) in enumerate([(img,ttl,"gray"),(gt,"GT","gray"),(pr,"pred","gray"),(ov,met,None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
            ax[k,c].set_title(t,fontsize=7); ax[k,c].axis("off")
    plt.suptitle(title+"   (yellow=correct, green=missed GT, red=false positive)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures",fname); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)

draw(te.nlargest(6,"dice").reset_index(drop=True),  "BEST — adaptive threshold",  "calcadp_best.png")
draw(te.nsmallest(6,"dice").reset_index(drop=True), "WORST — adaptive threshold", "calcadp_worst.png")
mid=te[(te.dice>0.82)&(te.dice<0.90)].head(6).reset_index(drop=True)
draw(mid, "TYPICAL — adaptive threshold", "calcadp_typical.png")

# summary panels
fig,ax=plt.subplots(1,4,figsize=(20,4.4))
ax[0].hist(te.dice,bins=20,color="#3b7",edgecolor="k")
ax[0].axvline(te.dice.mean(),color="r",ls="--",label="mean "+format(te.dice.mean(),".3f"))
ax[0].axvline(te.dice.median(),color="b",ls="--",label="median "+format(te.dice.median(),".3f"))
ax[0].set_title("Dice distribution (n="+str(len(te))+")"); ax[0].set_xlabel("Dice"); ax[0].legend()
if "subtlety" in te and te.subtlety.notna().any():
    sub=te.dropna(subset=["subtlety"]).groupby("subtlety").dice.mean()
    ax[1].bar(sub.index.astype(int).astype(str),sub.values,color="#59f",edgecolor="k")
    ax[1].set_ylim(0.7,1.0); ax[1].set_title("Dice by subtlety (1=hardest)"); ax[1].set_xlabel("subtlety")
ax[2].scatter(te.recall,te.precision,c=te.dice,cmap="viridis",s=18)
ax[2].plot([0,1],[0,1],"k:",alpha=.3); ax[2].set_xlim(0,1); ax[2].set_ylim(0,1)
ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision"); ax[2].set_title("precision vs recall")
if "thr" in te:
    ax[3].hist(te.thr,bins=15,color="#f96",edgecolor="k")
    ax[3].set_title("per-image thresholds chosen"); ax[3].set_xlabel("Otsu threshold")
plt.tight_layout()
o=os.path.join(D,"figures","calcadp_summary_panels.png")
plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)
print("\nDone: calcadp_best / calcadp_worst / calcadp_typical / calcadp_summary_panels")

model loaded (per-image adaptive threshold)
saved /root/autodl-tmp/CBIS/figures/calcadp_best.png
saved /root/autodl-tmp/CBIS/figures/calcadp_worst.png
saved /root/autodl-tmp/CBIS/figures/calcadp_typical.png
saved /root/autodl-tmp/CBIS/figures/calcadp_summary_panels.png

Done: calcadp_best / calcadp_worst / calcadp_typical / calcadp_summary_panels


In [7]:
# ══════════════════════════════════════════════════════════════════════
# CALCIFICATION — all requested switches (test one at a time vs your best)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3
torch.backends.cudnn.benchmark=True

# ── SWITCHES — start with ONLY these three, rest OFF ───────────────
ADAPTIVE_THR   = True
THR_LO, THR_HI = 0.20, 0.85     # #4 wider band
USE_ERODE      = False          # #5 morphological erode (thin the mask)
USE_CLDICE     = False          # #8 skeleton loss
WEIGHT_SUBT1   = False          # #9 oversample subtlety=1
CONTRAST_AUG   = False          # #10 low-contrast augmentation
TEXT_FREE      = False          # #6 strip all plot labels
TV_ALPHA, TV_BETA = 0.7, 0.3    # keep FP-punishing (NOT 0.3/0.7)
CLDICE_W = 0.3
# ───────────────────────────────────────────────────────────────────

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
C=P[P.abn_type=="calcification"]
tr=C[C.split=="train"].reset_index(drop=True)
va=C[C.split=="val"].reset_index(drop=True)
te=C[C.split=="test"].reset_index(drop=True)
for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
    assert len(set(a.patient_id)&set(b.patient_id))==0, "LEAK "+n
print("CALC | train "+str(len(tr))+" | val "+str(len(va))+" | test "+str(len(te)))

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im,train=False):
    im=_clahe.apply(im)
    if train and CONTRAST_AUG and np.random.rand()<0.5:   # #10
        a=np.random.uniform(0.6,1.0); b=np.random.uniform(-15,15)
        im=np.clip(im.astype(np.float32)*a+b,0,255).astype(np.uint8)
    return im

class DS(Dataset):
    def __init__(s,df,train=False): s.df=df.reset_index(drop=True); s.train=train
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        img=prep(img,s.train)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

# #9 weighted sampler for subtlety==1
if WEIGHT_SUBT1 and "subtlety" in tr:
    w=np.where(tr["subtlety"].fillna(3).values==1, 3.0, 1.0)
    sampler=WeightedRandomSampler(torch.tensor(w,dtype=torch.float), len(w), replacement=True)
    tl=DataLoader(DS(tr,True),batch_size=BATCH,sampler=sampler,num_workers=0)
else:
    tl=DataLoader(DS(tr,True),batch_size=BATCH,shuffle=True,num_workers=0)
vl=DataLoader(DS(va),batch_size=BATCH,shuffle=False,num_workers=0)
sl=DataLoader(DS(te),batch_size=BATCH,shuffle=False,num_workers=0)

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

def soft_skeleton(p,iters=6):
    for _ in range(iters):
        er=-F.max_pool2d(-p,3,1,1); op=F.max_pool2d(er,3,1,1); p=p-F.relu(p-op)
    return p
def cldice(p,g):
    sp=soft_skeleton(p); sg=soft_skeleton(g)
    tp=((sp*g).sum((2,3))+1)/(sp.sum((2,3))+1); ts=((sg*p).sum((2,3))+1)/(sg.sum((2,3))+1)
    return 1-(2*tp*ts/(tp+ts)).mean()
def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tv=1-((tp+1)/(tp+TV_ALPHA*fp+TV_BETA*fn+1)).mean()
    L=0.3*ce+0.7*tv
    if USE_CLDICE: L=L+CLDICE_W*cldice(p.unsqueeze(1),g.unsqueeze(1))
    return L

net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
opt=torch.optim.Adam(net.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)

def otsu_thr(pm):
    q=(pm*255).astype(np.uint8)
    t,_=cv2.threshold(q,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    return max(THR_LO,min(THR_HI,t/255.0))
_erode_k=cv2.getStructuringElement(cv2.MORPH_RECT,(1,3))

@torch.no_grad()
def evaluate(loader,df=None):
    net.eval(); rec=[]
    for x,y,idx in loader:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        prob=torch.softmax(o.float(),1)[:,1]
        for i in range(prob.size(0)):
            pm=prob[i].cpu().numpy()
            thr=otsu_thr(pm) if ADAPTIVE_THR else 0.5
            pr=(pm>thr).astype(np.uint8)
            if USE_ERODE: pr=cv2.morphologyEx(pr,cv2.MORPH_ERODE,_erode_k)   # #5
            ti=y[i].cpu().numpy().astype(np.uint8)
            tp=float((pr*ti).sum()); fp=float((pr*(1-ti)).sum()); fn=float(((1-pr)*ti).sum())
            d=dict(dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                   precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9),
                   pred_area=pr.mean(), gt_area=ti.mean(), thr=thr)
            if df is not None:
                r=df.iloc[int(idx[i])]
                d.update(patient_id=r["patient_id"], subtlety=r.get("subtlety",np.nan),
                         pathology=r.get("pathology",""))
            rec.append(d)
    return pd.DataFrame(rec)

best,bs,ni=0.,None,0
for ep in range(1,EPOCHS+1):
    net.train(); tot=0.; nb=0
    for x,y,_ in tl:
        x=x.to(DEV); y=y.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        tot+=l.item(); nb+=1
    V=evaluate(vl); d=V.dice.mean(); sch.step(d)
    if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
    else: ni+=1
    print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
          " | val-Dice "+format(d,".4f")+" | P "+format(V.precision.mean(),".3f")+
          " | R "+format(V.recall.mean(),".3f")+(" *" if d==best else ""))
    if ni>=10: print("  early stop"); break
net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
torch.save({"state_dict":{k:v.cpu() for k,v in net.state_dict().items()}},
           os.path.join(D,"seg_calc_v7.pth"))

R=evaluate(sl,df=te); R.to_csv(os.path.join(D,"seg_calc_v7_test.csv"),index=False)
print("\nCALC v7 | Dice "+format(R.dice.mean(),".4f")+" (median "+format(R.dice.median(),".4f")+
      ") | P "+format(R.precision.mean(),".3f")+" | R "+format(R.recall.mean(),".3f")+
      " | area "+format(R.pred_area.mean()/max(R.gt_area.mean(),1e-9),".2f"))
print("vs Tversky 0.879 | balanced+CLAHE 0.876")

CALC | train 1283 | val 257 | test 326
  ep  1/60 | loss 0.2811 | val-Dice 0.8449 | P 0.822 | R 0.898 *
  ep  2/60 | loss 0.2093 | val-Dice 0.8456 | P 0.832 | R 0.887 *
  ep  3/60 | loss 0.2049 | val-Dice 0.8426 | P 0.842 | R 0.870
  ep  4/60 | loss 0.2021 | val-Dice 0.8472 | P 0.828 | R 0.896 *
  ep  5/60 | loss 0.2029 | val-Dice 0.8462 | P 0.839 | R 0.881
  ep  6/60 | loss 0.2022 | val-Dice 0.8485 | P 0.831 | R 0.896 *
  ep  7/60 | loss 0.2014 | val-Dice 0.8474 | P 0.832 | R 0.891
  ep  8/60 | loss 0.2000 | val-Dice 0.8453 | P 0.844 | R 0.876
  ep  9/60 | loss 0.1984 | val-Dice 0.8122 | P 0.832 | R 0.831
  ep 10/60 | loss 0.2010 | val-Dice 0.8464 | P 0.839 | R 0.883
  ep 11/60 | loss 0.1982 | val-Dice 0.8464 | P 0.836 | R 0.886
  ep 12/60 | loss 0.1959 | val-Dice 0.8484 | P 0.833 | R 0.894
  ep 13/60 | loss 0.1952 | val-Dice 0.8501 | P 0.835 | R 0.895 *
  ep 14/60 | loss 0.1946 | val-Dice 0.8425 | P 0.826 | R 0.891
  ep 15/60 | loss 0.1946 | val-Dice 0.8480 | P 0.844 | R 0.878
  ep 1

In [8]:
# ══════════════════════════════════════════════════════════════════════
# VISUALIZE — calcification v7 (seg_calc_v7.pth + seg_calc_v7_test.csv)
#   best / worst / typical sheets + summary panels
#   raw CSV line printed per case (no headers)   [#1/#2/#3]
#   TEXT_FREE strips all labels/titles           [#6]
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu"); IMG=256

# must match the training-cell switches
ADAPTIVE_THR=True; THR_LO,THR_HI=0.20,0.85; USE_ERODE=False; TEXT_FREE=False

P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
te=P[(P.split=="test")&(P.abn_type=="calcification")].reset_index(drop=True)
R=pd.read_csv(os.path.join(D,"seg_calc_v7_test.csv"))
te=te.iloc[:len(R)].copy()
for c in ["dice","precision","recall","thr"]:
    if c in R: te[c]=R[c].values

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)
def otsu_thr(pm):
    q=(pm*255).astype(np.uint8)
    t,_=cv2.threshold(q,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    return max(THR_LO,min(THR_HI,t/255.0))
_erode_k=cv2.getStructuringElement(cv2.MORPH_RECT,(1,3))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

ck=torch.load(os.path.join(D,"seg_calc_v7.pth"),map_location=DEV)
sd=ck["state_dict"] if isinstance(ck,dict) and "state_dict" in ck else ck
net=AttnDueling().to(DEV); net.load_state_dict(sd); net.eval()
print("model loaded")

def predict(img):
    x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
        pm=torch.softmax(net(x).float(),1)[0,1].cpu().numpy()
    thr=otsu_thr(pm) if ADAPTIVE_THR else 0.5
    pr=(pm>thr).astype(np.uint8)
    if USE_ERODE: pr=cv2.morphologyEx(pr,cv2.MORPH_ERODE,_erode_k)
    return pr,thr

def draw(cases,title,fname):
    N=len(cases)
    if N==0: return
    fig,ax=plt.subplots(N,4,figsize=(13,3.0*N))
    if N==1: ax=ax.reshape(1,4)
    for k in range(N):
        r=cases.iloc[k]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gt=(msk>127).astype(np.uint8)
        pr,thr=predict(img)
        # [#1/#2/#3] raw comma-separated line, no header
        print(str(r.get("patient_id",""))+","+str(r.get("subtlety",""))+","+
              format(r["dice"],".4f")+","+format(r["precision"],".4f")+","+
              format(r["recall"],".4f")+","+format(thr,".4f"))
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[(gt>0)&(pr==0)]=[0,150,0]; ov[(pr>0)&(gt==0)]=[210,0,0]; ov[(pr>0)&(gt>0)]=[235,215,0]
        panels=[(img,"gray"),(gt,"gray"),(pr,"gray"),(ov,None)]
        titles=[str(r.get("patient_id","?"))+" sub="+str(r.get("subtlety","?")),
                "GT","pred","Dice="+format(r["dice"],".2f")+" P="+format(r["precision"],".2f")+" R="+format(r["recall"],".2f")]
        for c,(im,cm) in enumerate(panels):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
            if not TEXT_FREE: ax[k,c].set_title(titles[c],fontsize=7)
            ax[k,c].axis("off")
    if not TEXT_FREE:
        plt.suptitle(title+"   (yellow=correct, green=missed GT, red=false positive)",fontsize=11)
    plt.tight_layout()
    o=os.path.join(D,"figures",fname); os.makedirs(os.path.dirname(o),exist_ok=True)
    plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)

print("\n# patient_id,subtlety,dice,precision,recall,threshold")
draw(te.nlargest(6,"dice").reset_index(drop=True),  "BEST — v7",    "calcv7_best.png")
draw(te.nsmallest(6,"dice").reset_index(drop=True), "WORST — v7",   "calcv7_worst.png")
mid=te[(te.dice>0.82)&(te.dice<0.90)].head(6).reset_index(drop=True)
draw(mid, "TYPICAL — v7", "calcv7_typical.png")

# summary panels
fig,ax=plt.subplots(1,4,figsize=(20,4.4))
ax[0].hist(te.dice,bins=20,color="#3b7",edgecolor="k")
ax[0].axvline(te.dice.mean(),color="r",ls="--"); ax[0].axvline(te.dice.median(),color="b",ls="--")
if not TEXT_FREE:
    ax[0].set_title("Dice  mean "+format(te.dice.mean(),".3f")+"  median "+format(te.dice.median(),".3f"))
    ax[0].set_xlabel("Dice")
if "subtlety" in te and te.subtlety.notna().any():
    sub=te.dropna(subset=["subtlety"]).groupby("subtlety").dice.mean()
    ax[1].bar(sub.index.astype(int).astype(str),sub.values,color="#59f",edgecolor="k"); ax[1].set_ylim(0.7,1.0)
    if not TEXT_FREE: ax[1].set_title("Dice by subtlety (1=hardest)")
ax[2].scatter(te.recall,te.precision,c=te.dice,cmap="viridis",s=18)
ax[2].plot([0,1],[0,1],"k:",alpha=.3); ax[2].set_xlim(0,1); ax[2].set_ylim(0,1)
if not TEXT_FREE: ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision"); ax[2].set_title("precision vs recall")
if "thr" in te:
    ax[3].hist(te.thr,bins=15,color="#f96",edgecolor="k")
    if not TEXT_FREE: ax[3].set_title("per-image thresholds")
plt.tight_layout()
o=os.path.join(D,"figures","calcv7_summary_panels.png")
plt.savefig(o,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+o)
print("\nDone.")

model loaded

# patient_id,subtlety,dice,precision,recall,threshold
P_00579,2,0.9643,0.9788,0.9503,0.4824
P_02153,2,0.9614,0.9830,0.9407,0.4824
P_01711,3,0.9607,0.9927,0.9307,0.4784
P_01534,4,0.9586,0.9912,0.9280,0.4863
P_00790,4,0.9569,0.9613,0.9526,0.4863
P_02153,2,0.9567,0.9520,0.9615,0.4824
saved /root/autodl-tmp/CBIS/figures/calcv7_best.png
P_01460,4,0.3973,0.2598,0.8441,0.4706
P_00038,5,0.4341,0.3389,0.6036,0.4471
P_01460,4,0.4422,0.3035,0.8141,0.4627
P_00299,4,0.4904,0.3574,0.7809,0.4784
P_00038,5,0.5028,0.3675,0.7957,0.4353
P_01460,4,0.5499,0.3981,0.8892,0.4863
saved /root/autodl-tmp/CBIS/figures/calcv7_worst.png
P_00038,5,0.8316,0.9013,0.7719,0.4392
P_00041,5,0.8406,0.8325,0.8489,0.4824
P_00077,4,0.8778,0.8568,0.8999,0.4863
P_00100,4,0.8734,0.8167,0.9385,0.4941
P_00127,3,0.8535,0.8241,0.8849,0.4902
P_00127,3,0.8838,0.8071,0.9766,0.4902
saved /root/autodl-tmp/CBIS/figures/calcv7_typical.png
saved /root/autodl-tmp/CBIS/figures/calcv7_summary_panels.png

Done.


In [9]:
# ══════════════════════════════════════════════════════════════════════
# ALL-DATASET SEGMENTATION — CLAHE + augmentation + Tversky(a=0.7,b=0.3)
#   CBIS: mass and calc trained SEPARATELY on the official split
#   INbreast / CSAW: patient-grouped split, own task
#   CDD-CESM: reported SEPARATELY (contrast-enhanced = different modality)
#   Fixed threshold 0.5 (adaptive proved to be a no-op on this data)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"; ROOT="/root/autodl-tmp"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8
THR=0.5
TV_ALPHA, TV_BETA = 0.7, 0.3
torch.backends.cudnn.benchmark=True

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

# ---------- dataset with CLAHE + 8x lockstep augmentation ----------
class DS(Dataset):
    def __init__(s,df,aug,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    tv=1-((tp+1)/(tp+TV_ALPHA*fp+TV_BETA*fn+1)).mean()
    return 0.3*ce+0.7*tv

@torch.no_grad()
def evaluate(net,loader,df=None):
    net.eval(); rec=[]
    for x,y,idx in loader:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            d=dict(dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                   precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9))
            if df is not None:
                r=df.iloc[int(idx[i])]; d["patient_id"]=r["patient_id"]
            rec.append(d)
    return pd.DataFrame(rec)

def train_task(name, tr, va, te, tag):
    for a,b,n in [(tr,te,"train/test"),(tr,va,"train/val"),(va,te,"val/test")]:
        ov=set(a.patient_id)&set(b.patient_id)
        assert len(ov)==0, name+" LEAK "+n+" ("+str(len(ov))+")"
    print("\n"+"#"*68)
    print("### "+name+"   train "+str(len(tr))+" x"+str(MULT)+" | val "+str(len(va))+" | test "+str(len(te)))
    print("#"*68)
    tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
    vl=DataLoader(DS(va,False),batch_size=BATCH,shuffle=False,num_workers=0)
    sl=DataLoader(DS(te,False),batch_size=BATCH,shuffle=False,num_workers=0)
    net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        V=evaluate(net,vl); d=V.dice.mean(); sch.step(d)
        if d>best: best=d; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
              " | val-Dice "+format(d,".4f")+(" *" if d==best else ""))
        if ni>=10: print("  early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"seg_"+tag+".pth"))
    R=evaluate(net,sl,te); R.to_csv(os.path.join(D,"seg_"+tag+"_test.csv"),index=False)
    print("  TEST  Dice "+format(R.dice.mean(),".4f")+" (median "+format(R.dice.median(),".4f")+
          ") | IoU "+format(R.iou.mean(),".4f")+
          " | P "+format(R.precision.mean(),".3f")+" | R "+format(R.recall.mean(),".3f"))
    return dict(name=name, n=len(R), dice=R.dice.mean(), median=R.dice.median(),
                iou=R.iou.mean(), prec=R.precision.mean(), rec=R.recall.mean())

def patient_split(df, seed=42):
    rng=np.random.RandomState(seed)
    df=df.copy(); df["split"]="train"
    pats=sorted(df.patient_id.unique().tolist()); rng.shuffle(pats)
    n=len(pats); te=set(pats[:int(0.20*n)]); va=set(pats[int(0.20*n):int(0.35*n)])
    df.loc[df.patient_id.isin(te),"split"]="test"
    df.loc[df.patient_id.isin(va),"split"]="val"
    return df

results=[]

# ---------- CBIS mass + calc (official split) ----------
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag in [("mass","cbis_mass"),("calcification","cbis_calc")]:
    S=P[P.abn_type==kind]
    results.append(train_task("CBIS "+kind, S[S.split=="train"], S[S.split=="val"], S[S.split=="test"], tag))

# ---------- INbreast / CSAW / CDD (own patient-grouped splits) ----------
# expects a CSV per dataset with columns: patient_id, img, msk
# adjust the paths/filenames to match what you built earlier (extra_seg.csv etc.)
extra_sources = {
    "INbreast": os.path.join(D,"inbreast_seg.csv"),
    "CSAW":     os.path.join(D,"csaw_seg.csv"),
    "CDD-CESM": os.path.join(D,"cdd_seg.csv"),
}
for name,path in extra_sources.items():
    if not os.path.exists(path):
        print("\n(skip "+name+": "+path+" not found — point this at your CSV)")
        continue
    df=pd.read_csv(path)
    if "patient_id" not in df.columns:      # fall back to a row id if no patient col
        df["patient_id"]=df.index.astype(str)
    df=patient_split(df)
    tag=name.lower().replace("-","").replace(" ","")
    results.append(train_task(name, df[df.split=="train"], df[df.split=="val"], df[df.split=="test"], tag))

# ---------- summary ----------
print("\n"+"="*82)
print("ALL-DATASET SEGMENTATION SUMMARY  (CLAHE + aug + Tversky, thr=0.5)")
print("="*82)
print("  dataset".ljust(20)+"n     Dice    median   IoU     Prec    Recall")
print("  "+"-"*76)
for r in results:
    print("  "+r["name"].ljust(18)+str(r["n"]).rjust(4)+"   "+
          format(r["dice"],".4f")+"  "+format(r["median"],".4f")+"  "+
          format(r["iou"],".4f")+"  "+format(r["prec"],".4f")+"  "+format(r["rec"],".4f"))
print("="*82)
print("  NOTE: CDD-CESM is contrast-enhanced (different modality) — report it")
print("        separately, never pooled with standard mammography.")
pd.DataFrame(results).to_csv(os.path.join(D,"all_dataset_seg_summary.csv"),index=False)
print("  saved all_dataset_seg_summary.csv")


####################################################################
### CBIS mass   train 1122 x8 | val 196 | test 378
####################################################################
  ep  1/60 | loss 0.1611 | val-Dice 0.8934 *
  ep  2/60 | loss 0.1293 | val-Dice 0.9073 *
  ep  3/60 | loss 0.1235 | val-Dice 0.9131 *
  ep  4/60 | loss 0.1203 | val-Dice 0.9149 *
  ep  5/60 | loss 0.1173 | val-Dice 0.9062
  ep  6/60 | loss 0.1149 | val-Dice 0.9119
  ep  7/60 | loss 0.1128 | val-Dice 0.9188 *
  ep  8/60 | loss 0.1108 | val-Dice 0.9120
  ep  9/60 | loss 0.1090 | val-Dice 0.9189 *
  ep 10/60 | loss 0.1073 | val-Dice 0.9161
  ep 11/60 | loss 0.1055 | val-Dice 0.9183
  ep 12/60 | loss 0.1041 | val-Dice 0.9160
  ep 13/60 | loss 0.0986 | val-Dice 0.9234 *
  ep 14/60 | loss 0.0968 | val-Dice 0.9192
  ep 15/60 | loss 0.0954 | val-Dice 0.9217
  ep 16/60 | loss 0.0934 | val-Dice 0.9215
  ep 17/60 | loss 0.0915 | val-Dice 0.9198
  ep 18/60 | loss 0.0899 | val-Dice 0.9189
  ep 19/60 | loss 0.08

In [10]:
# add this AFTER your training cell has run and saved the seg_*.pth models.
# It re-scores every dataset (train/val/test) into ONE combined file.
import os, glob
os.environ["OMP_NUM_THREADS"]="4"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; BATCH=16; THR=0.5
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

class DS(Dataset):
    def __init__(s,df): s.df=df.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        return (torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

@torch.no_grad()
def per_image(net,df,dataset,split):
    net.eval(); rec=[]
    for x,y,idx in DataLoader(DS(df),batch_size=BATCH,shuffle=False,num_workers=0):
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r=df.iloc[int(idx[i])]
            rec.append(dict(dataset=dataset, split=split,
                            patient_id=r.get("patient_id",""),
                            dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                            precision=tp/(tp+fp+1e-9), recall=tp/(tp+fn+1e-9)))
    return pd.DataFrame(rec)

def load_model(tag):
    net=AttnDueling().to(DEV)
    net.load_state_dict(torch.load(os.path.join(D,"seg_"+tag+".pth"),map_location=DEV)); net.eval()
    return net

def patient_split(df, seed=42):
    rng=np.random.RandomState(seed); df=df.copy(); df["split"]="train"
    pats=sorted(df.patient_id.unique().tolist()); rng.shuffle(pats)
    n=len(pats); te=set(pats[:int(0.20*n)]); va=set(pats[int(0.20*n):int(0.35*n)])
    df.loc[df.patient_id.isin(te),"split"]="test"; df.loc[df.patient_id.isin(va),"split"]="val"
    return df

# ---- gather per-image results for every dataset ----
ALL=[]
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag,label in [("mass","cbis_mass","CBIS mass"),("calcification","cbis_calc","CBIS calc")]:
    S=P[P.abn_type==kind]
    if not os.path.exists(os.path.join(D,"seg_"+tag+".pth")):
        print("(skip "+label+": model seg_"+tag+".pth not found — train it first)"); continue
    net=load_model(tag)
    for sp in ["train","val","test"]:
        sub=S[S.split==sp]
        if len(sub): ALL.append(per_image(net,sub,label,sp))

extra_sources={"INbreast":"inbreast_seg.csv","CSAW":"csaw_seg.csv","CDD-CESM":"cdd_seg.csv"}
for name,fname in extra_sources.items():
    path=os.path.join(D,fname); tag=name.lower().replace("-","").replace(" ","")
    if not os.path.exists(path):
        print("(skip "+name+": "+fname+" not found)"); continue
    if not os.path.exists(os.path.join(D,"seg_"+tag+".pth")):
        print("(skip "+name+": model seg_"+tag+".pth not found — train it first)"); continue
    df=pd.read_csv(path)
    if "patient_id" not in df.columns: df["patient_id"]=df.index.astype(str)
    df=patient_split(df); net=load_model(tag)
    for sp in ["train","val","test"]:
        sub=df[df.split==sp]
        if len(sub): ALL.append(per_image(net,sub,name,sp))

if not ALL:
    print("\nNo results — make sure the training cell ran and saved the seg_*.pth files.")
else:
    PER=pd.concat(ALL,ignore_index=True)
    PER.to_csv(os.path.join(D,"all_results_per_image.csv"),index=False)   # every lesion, every dataset/split

    # ---- summary rows: each dataset x split ----
    def summarize(g):
        return pd.Series(dict(n=len(g), dice=g.dice.mean(), median=g.dice.median(),
                              iou=g.iou.mean(), precision=g.precision.mean(), recall=g.recall.mean()))
    SUM=PER.groupby(["dataset","split"]).apply(summarize).reset_index()

    # ---- COMBINED rows (all datasets pooled) per split ----
    comb=PER.groupby("split").apply(summarize).reset_index()
    comb.insert(0,"dataset","ALL COMBINED")
    SUM=pd.concat([SUM,comb],ignore_index=True)

    # ---- combined ALL datasets, ALL splits (single grand number) ----
    grand=summarize(PER); grand["dataset"]="ALL COMBINED"; grand["split"]="all"
    SUM=pd.concat([SUM,pd.DataFrame([grand])],ignore_index=True)

    SUM=SUM[["dataset","split","n","dice","median","iou","precision","recall"]]
    SUM.to_csv(os.path.join(D,"all_results_summary.csv"),index=False)     # THE ONE FILE you wanted

    print("\n"+"="*88)
    print("ALL DATASETS + COMBINED — one summary file")
    print("="*88)
    print("  dataset".ljust(16)+"split".ljust(7)+"n".rjust(5)+"   Dice    median   IoU     Prec    Recall")
    print("  "+"-"*82)
    for _,r in SUM.iterrows():
        print("  "+str(r["dataset"]).ljust(16)+str(r["split"]).ljust(7)+str(int(r["n"])).rjust(5)+"   "+
              format(r["dice"],".4f")+"  "+format(r["median"],".4f")+"  "+format(r["iou"],".4f")+"  "+
              format(r["precision"],".4f")+"  "+format(r["recall"],".4f"))
    print("="*88)
    print("  saved all_results_summary.csv   (per-dataset + combined, all splits)")
    print("  saved all_results_per_image.csv (every lesion, for figures/analysis)")
    print("  NOTE: CDD-CESM is contrast-enhanced — in the write-up, also report it")
    print("        on its own, not only inside the combined pooled number.")

(skip INbreast: inbreast_seg.csv not found)
(skip CSAW: csaw_seg.csv not found)
(skip CDD-CESM: cdd_seg.csv not found)

ALL DATASETS + COMBINED — one summary file
  dataset       split      n   Dice    median   IoU     Prec    Recall
  ----------------------------------------------------------------------------------
  CBIS calc       test     326   0.8840  0.9053  0.7987  0.8876  0.8892
  CBIS calc       train   1283   0.8903  0.9099  0.8086  0.8878  0.9015
  CBIS calc       val      257   0.8593  0.9051  0.7709  0.8571  0.8801
  CBIS mass       test     378   0.9242  0.9335  0.8614  0.9369  0.9151
  CBIS mass       train   1122   0.9393  0.9452  0.8869  0.9519  0.9291
  CBIS mass       val      196   0.9234  0.9350  0.8605  0.9350  0.9160
  ALL COMBINED    test     704   0.9056  0.9216  0.8324  0.9141  0.9031
  ALL COMBINED    train   2405   0.9132  0.9292  0.8451  0.9177  0.9143
  ALL COMBINED    val      453   0.8871  0.9173  0.8097  0.8908  0.8956
  ALL COMBINED    all     3562   

/tmp/ipykernel_1602/608885497.py:119: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  SUM=PER.groupby(["dataset","split"]).apply(summarize).reset_index()
/tmp/ipykernel_1602/608885497.py:122: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  comb=PER.groupby("split").apply(summarize).reset_index()


In [12]:
import pandas as pd, os
D="/root/autodl-tmp/CBIS"; MULT=8
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
print("PATIENTS / IMAGES / AUGMENTED TRAIN SIZE")
print("="*66)
for kind in ["mass","calcification"]:
    S=P[P.abn_type==kind]
    for sp in ["train","val","test"]:
        sub=S[S.split==sp]
        aug=" -> x8 = "+str(len(sub)*MULT) if sp=="train" else ""
        print("CBIS "+kind.ljust(13)+sp.ljust(6)+str(len(sub)).rjust(5)+" images | "+
              str(sub.patient_id.nunique()).rjust(4)+" patients"+aug)
print("="*66)
print("CBIS total: "+str(len(P))+" lesions | "+str(P.patient_id.nunique())+" patients")

PATIENTS / IMAGES / AUGMENTED TRAIN SIZE
CBIS mass         train  1122 images |  588 patients -> x8 = 8976
CBIS mass         val     196 images |  103 patients
CBIS mass         test    378 images |  201 patients
CBIS calcificationtrain  1283 images |  512 patients -> x8 = 10264
CBIS calcificationval     257 images |   90 patients
CBIS calcificationtest    326 images |  151 patients
CBIS total: 3562 lesions | 1566 patients


In [11]:
# ══════════════════════════════════════════════════════════════════════
# VISUALIZE — each dataset separately: image | GT mask | predicted mask | overlay
#   Uses the seg_*.pth models saved by the training cell (CLAHE, thr=0.5).
#   One figure per dataset, saved to figures/.
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; THR=0.5; N_SHOW=6
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

def load_model(tag):
    p=os.path.join(D,"seg_"+tag+".pth")
    if not os.path.exists(p): return None
    net=AttnDueling().to(DEV)
    net.load_state_dict(torch.load(p,map_location=DEV)); net.eval()
    return net

def predict(net,img):
    x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
    with torch.no_grad(), torch.amp.autocast(device_type="cuda"):
        pm=torch.softmax(net(x).float(),1)[0,1].cpu().numpy()
    return (pm>THR).astype(np.uint8)

def visualize(net, df, dataset, fname):
    df=df.reset_index(drop=True)
    N=min(N_SHOW,len(df))
    if N==0: print("(no rows for "+dataset+")"); return
    fig,ax=plt.subplots(N,4,figsize=(13,3.1*N))
    if N==1: ax=ax.reshape(1,4)
    for k in range(N):
        r=df.iloc[k*max(1,len(df)//N)]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        gt=(msk>127).astype(np.uint8)
        pr=predict(net,img)
        inter=(pr*gt).sum(); dsc=(2*inter+1)/(pr.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[(gt>0)&(pr==0)]=[0,150,0]      # green = missed GT
        ov[(pr>0)&(gt==0)]=[210,0,0]      # red   = false positive
        ov[(pr>0)&(gt>0)]=[235,215,0]     # yellow= correct
        for c,(im,t,cm) in enumerate([(img,dataset,"gray"),(gt,"GT mask","gray"),
                                      (pr,"predicted","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
            ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
    plt.suptitle(dataset+" — image | GT | prediction | overlay (yellow=correct, green=missed, red=false pos)",fontsize=10)
    plt.tight_layout()
    os.makedirs(os.path.join(D,"figures"),exist_ok=True)
    o=os.path.join(D,"figures",fname)
    plt.savefig(o,dpi=135,bbox_inches="tight"); plt.close(); print("saved "+o)

def patient_split(df, seed=42):
    rng=np.random.RandomState(seed); df=df.copy(); df["split"]="train"
    pats=sorted(df.patient_id.unique().tolist()); rng.shuffle(pats)
    n=len(pats); te=set(pats[:int(0.20*n)]); va=set(pats[int(0.20*n):int(0.35*n)])
    df.loc[df.patient_id.isin(te),"split"]="test"; df.loc[df.patient_id.isin(va),"split"]="val"
    return df

# ---------- CBIS mass + calc (show TEST split) ----------
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag,label,fn in [("mass","cbis_mass","CBIS MASS","viz_cbis_mass.png"),
                          ("calcification","cbis_calc","CBIS CALCIFICATION","viz_cbis_calc.png")]:
    net=load_model(tag)
    if net is None: print("(skip "+label+": seg_"+tag+".pth not found)"); continue
    visualize(net, P[(P.abn_type==kind)&(P.split=="test")], label, fn)

# ---------- INbreast / CSAW / CDD-CESM ----------
extra={"INbreast":("inbreast_seg.csv","inbreast","viz_inbreast.png"),
       "CSAW":("csaw_seg.csv","csaw","viz_csaw.png"),
       "CDD-CESM":("cdd_seg.csv","cddcesm","viz_cdd.png")}
for name,(fname,tag,outpng) in extra.items():
    path=os.path.join(D,fname)
    net=load_model(tag)
    if not os.path.exists(path): print("(skip "+name+": "+fname+" not found)"); continue
    if net is None: print("(skip "+name+": seg_"+tag+".pth not found)"); continue
    df=pd.read_csv(path)
    if "patient_id" not in df.columns: df["patient_id"]=df.index.astype(str)
    df=patient_split(df)
    visualize(net, df[df.split=="test"], name, outpng)

print("\nDone. One figure per dataset in figures/.")

saved /root/autodl-tmp/CBIS/figures/viz_cbis_mass.png
saved /root/autodl-tmp/CBIS/figures/viz_cbis_calc.png
(skip INbreast: inbreast_seg.csv not found)
(skip CSAW: csaw_seg.csv not found)
(skip CDD-CESM: cdd_seg.csv not found)

Done. One figure per dataset in figures/.


In [13]:
import os, glob, pandas as pd
ROOT="/root/autodl-tmp"
print("Searching for INbreast / CSAW / CDD files...\n")

for key in ["inbreast","csaw","cdd","cesm"]:
    hits=glob.glob(os.path.join(ROOT,"**","*"+key+"*"),recursive=True)
    print(key.upper()+":  "+str(len(hits))+" matches")
    for h in hits[:12]:
        tag=""
        if h.lower().endswith(".csv"):
            try:
                d=pd.read_csv(h); tag=" ["+str(len(d))+" rows, cols: "+",".join(map(str,d.columns[:6]))+"]"
            except: tag=" [csv unreadable]"
        elif os.path.isdir(h):
            tag=" (folder, "+str(len(os.listdir(h)))+" items)"
        print("   "+h+tag)
    print()

Searching for INbreast / CSAW / CDD files...

INBREAST:  15 matches
   /root/autodl-tmp/CBIS/inbreast_crops (folder, 300 items)
   /root/autodl-tmp/CBIS/inbreast_masks (folder, 300 items)
   /root/autodl-tmp/CBIS/inbreast_converted.csv [300 rows, cols: patient_id,cropped_jpeg_path,roi_mask_jpeg_path,label,label_name,abn_type]
   /root/autodl-tmp/CBIS/inbreast_crops_aug (folder, 900 items)
   /root/autodl-tmp/CBIS/inbreast_masks_aug (folder, 900 items)
   /root/autodl-tmp/CBIS/inbreast_train_aug.csv [1125 rows, cols: patient_id,cropped_jpeg_path,roi_mask_jpeg_path,label,label_name,abn_type]
   /root/autodl-tmp/CBIS/inbreast_test.csv [75 rows, cols: patient_id,cropped_jpeg_path,roi_mask_jpeg_path,label,label_name,abn_type]
   /root/autodl-tmp/CBIS/attn_dueling_unet_inbreast_best.pth
   /root/autodl-tmp/CBIS/seg_aug_inbreast_best.pth
   /root/autodl-tmp/CBIS/figures/inbreast_segmentation_vis.png
   /root/autodl-tmp/INBreast_data/INbreast Release 1.0/inbreast.pdf
   /root/autodl-tmp/INBrea

In [14]:
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
torch.backends.cudnn.benchmark=True

# INbreast CSVs already have a train/test split as separate files; carve val from train
trdf=pd.read_csv(os.path.join(D,"inbreast_train_aug.csv"))
tedf=pd.read_csv(os.path.join(D,"inbreast_test.csv"))
# normalize column names -> img / msk
for df in (trdf,tedf):
    df.rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"},inplace=True)
rng=np.random.RandomState(42)
vp=set(rng.permutation(trdf.patient_id.unique())[:max(1,int(0.15*trdf.patient_id.nunique()))])
vadf=trdf[trdf.patient_id.isin(vp)].reset_index(drop=True)
trdf=trdf[~trdf.patient_id.isin(vp)].reset_index(drop=True)
for a,b,n in [(trdf,tedf,"tr/te"),(trdf,vadf,"tr/va"),(vadf,tedf,"va/te")]:
    ov=set(a.patient_id)&set(b.patient_id); assert len(ov)==0, "LEAK "+n
print("INbreast | train "+str(len(trdf))+" x"+str(MULT)+" | val "+str(len(vadf))+" | test "+str(len(tedf)))
print("patients: train "+str(trdf.patient_id.nunique())+" val "+str(vadf.patient_id.nunique())+
      " test "+str(tedf.patient_id.nunique()))

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)
class DS(Dataset):
    def __init__(s,df,aug,mult=1): s.df=df.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnDueling(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.v=nn.Conv2d(b,1,1); s.adv=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        V=s.v(d1); Ad=s.adv(d1); return V+Ad-Ad.mean(1,keepdim=True)

def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())

@torch.no_grad()
def score(net,df):
    net.eval(); ld=DataLoader(DS(df,False),batch_size=BATCH,shuffle=False,num_workers=0); ds=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            ds.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                           prec=tp/(tp+fp+1e-9), rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(ds)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),
                prec=R.prec.mean(),rec=R.rec.mean())

tl=DataLoader(DS(trdf,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
net=AttnDueling().to(DEV); scaler=torch.amp.GradScaler()
opt=torch.optim.Adam(net.parameters(),lr=LR)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
best,bs,ni=0.,None,0
for ep in range(1,EPOCHS+1):
    net.train(); tot=0.; nb=0
    for x,y,_ in tl:
        x=x.to(DEV); y=y.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update()
        tot+=l.item(); nb+=1
    vd=score(net,vadf)["dice"]; sch.step(vd)
    if vd>best: best=vd; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
    else: ni+=1
    print("  ep "+str(ep).rjust(2)+"/"+str(EPOCHS)+" | loss "+format(tot/max(nb,1),".4f")+
          " | val-Dice "+format(vd,".4f")+(" *" if vd==best else ""))
    if ni>=10: print("  early stop"); break
net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"seg_inbreast_final.pth"))

rows=[]
for sp,dfp in [("TRAIN",trdf),("VAL",vadf),("TEST",tedf)]:
    m=score(net,dfp); m["split"]=sp; rows.append(m)
    print("  "+sp.ljust(6)+" Dice "+format(m["dice"],".4f")+" (median "+format(m["median"],".4f")+
          ") | IoU "+format(m["iou"],".4f")+" | P "+format(m["prec"],".3f")+
          " | R "+format(m["rec"],".3f")+" | n="+str(m["n"]))
pd.DataFrame(rows).to_csv(os.path.join(D,"inbreast_train_val_test.csv"),index=False)

# visualization (test)
N=min(6,len(tedf))
fig,ax=plt.subplots(N,4,figsize=(13,3.1*N))
if N==1: ax=ax.reshape(1,4)
with torch.no_grad():
    for k in range(N):
        r=tedf.iloc[k*max(1,len(tedf)//N)]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        gt=(msk>127).astype(np.uint8)
        x=torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
        with torch.amp.autocast(device_type="cuda"): pr=(torch.softmax(net(x).float(),1)[0,1]>THR).cpu().numpy().astype(np.uint8)
        dsc=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[(gt>0)&(pr==0)]=[0,150,0]; ov[(pr>0)&(gt==0)]=[210,0,0]; ov[(pr>0)&(gt>0)]=[235,215,0]
        for c,(im,t,cm) in enumerate([(img,"INbreast","gray"),(gt,"GT","gray"),(pr,"pred","gray"),(ov,"Dice="+format(dsc,".2f"),None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im); ax[k,c].set_title(t,fontsize=8); ax[k,c].axis("off")
plt.suptitle("INbreast — image | GT | prediction | overlay",fontsize=10); plt.tight_layout()
o=os.path.join(D,"figures","viz_inbreast_final.png"); os.makedirs(os.path.dirname(o),exist_ok=True)
plt.savefig(o,dpi=135,bbox_inches="tight"); plt.close(); print("saved "+o)

INbreast | train 960 x8 | val 165 | test 75
patients: train 192 val 33 test 75
  ep  1/60 | loss 0.1703 | val-Dice 0.9161 *
  ep  2/60 | loss 0.1473 | val-Dice 0.9198 *
  ep  3/60 | loss 0.1361 | val-Dice 0.9212 *
  ep  4/60 | loss 0.1276 | val-Dice 0.9245 *
  ep  5/60 | loss 0.1186 | val-Dice 0.9245
  ep  6/60 | loss 0.1106 | val-Dice 0.9302 *
  ep  7/60 | loss 0.1018 | val-Dice 0.9234
  ep  8/60 | loss 0.0960 | val-Dice 0.9297
  ep  9/60 | loss 0.0905 | val-Dice 0.9254
  ep 10/60 | loss 0.0845 | val-Dice 0.9249
  ep 11/60 | loss 0.0797 | val-Dice 0.9251
  ep 12/60 | loss 0.0696 | val-Dice 0.9272
  ep 13/60 | loss 0.0649 | val-Dice 0.9234
  ep 14/60 | loss 0.0597 | val-Dice 0.9248
  ep 15/60 | loss 0.0568 | val-Dice 0.9280
  ep 16/60 | loss 0.0528 | val-Dice 0.9245
  early stop
  TRAIN  Dice 0.9316 (median 0.9474) | IoU 0.8764 | P 0.925 | R 0.944 | n=960
  VAL    Dice 0.9302 (median 0.9422) | IoU 0.8721 | P 0.928 | R 0.936 | n=165
  TEST   Dice 0.9210 (median 0.9313) | IoU 0.8578 | P 

In [1]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION — PLAIN Attention U-Net (NO dueling head)
#   Identical pipeline to your full model: CLAHE + 8x aug + Tversky(0.7,0.3)
#   Runs CBIS mass, CBIS calc, and INbreast so you get matching ablation rows.
#   Only difference vs full model: segmentation head = single conv (no V/A streams)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
torch.backends.cudnn.benchmark=True

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))

# --- PLAIN Attention U-Net: single-conv head, NO value/advantage streams ---
class AttnPlain(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1)                 # <-- plain head (the only change)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.out(d1)

def loss_fn(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())

@torch.no_grad()
def sc(net,d):
    net.eval(); ld=DataLoader(DS(d,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),prec=R.prec.mean(),rec=R.rec.mean())

def train_one(name, tr, va, te, tag):
    for a,b,nm in [(tr,te,"tr/te"),(tr,va,"tr/va"),(va,te,"va/te")]:
        if len(a) and len(b): assert len(set(a.patient_id)&set(b.patient_id))==0, name+" LEAK "+nm
    print("\n### PLAIN Attn U-Net — "+name+" | train "+str(len(tr))+" x"+str(MULT)+
          " | val "+str(len(va))+" | test "+str(len(te)))
    tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
    net=AttnPlain().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); l=loss_fn(o,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        vd=sc(net,va)["dice"] if len(va) else sc(net,tr)["dice"]; sch.step(vd)
        if vd>best: best=vd; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+" | val-Dice "+format(vd,".4f")+(" *" if vd==best else ""))
        if ni>=10: print("  early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"ablate_plain_"+tag+".pth"))
    m=sc(net,te); print("  TEST Dice "+format(m["dice"],".4f")+" (median "+format(m["median"],".4f")+
          ") | IoU "+format(m["iou"],".4f")+" | P "+format(m["prec"],".3f")+" | R "+format(m["rec"],".3f"))
    m["name"]=name; return m

res=[]
# CBIS mass + calc (official split)
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag in [("mass","cbis_mass"),("calcification","cbis_calc")]:
    S=P[P.abn_type==kind]
    res.append(train_one("CBIS "+kind, S[S.split=="train"], S[S.split=="val"], S[S.split=="test"], tag))

# INbreast (its own split files)
trdf=pd.read_csv(os.path.join(D,"inbreast_train_aug.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
tedf=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
rng=np.random.RandomState(42)
vp=set(rng.permutation(trdf.patient_id.unique())[:max(1,int(0.15*trdf.patient_id.nunique()))])
vadf=trdf[trdf.patient_id.isin(vp)]; trdf2=trdf[~trdf.patient_id.isin(vp)]
res.append(train_one("INbreast", trdf2, vadf, tedf, "inbreast"))

print("\n"+"="*64)
print("ABLATION — PLAIN Attention U-Net (no dueling head), TEST Dice")
print("="*64)
for r in res:
    print("  "+r["name"].ljust(16)+" Dice "+format(r["dice"],".4f")+
          " | IoU "+format(r["iou"],".4f")+" | n="+str(r["n"]))
print("="*64)
print("Compare each row against your FULL model (dueling head) on the same split.")
pd.DataFrame(res).to_csv(os.path.join(D,"ablation_plain_attnunet.csv"),index=False)


### PLAIN Attn U-Net — CBIS mass | train 1122 x8 | val 196 | test 378
  ep  1 | loss 0.1527 | val-Dice 0.9041 *
  ep  2 | loss 0.1274 | val-Dice 0.9035
  ep  3 | loss 0.1214 | val-Dice 0.9152 *
  ep  4 | loss 0.1182 | val-Dice 0.9111
  ep  5 | loss 0.1152 | val-Dice 0.8945
  ep  6 | loss 0.1125 | val-Dice 0.9141
  ep  7 | loss 0.1110 | val-Dice 0.8973
  ep  8 | loss 0.1087 | val-Dice 0.8973
  ep  9 | loss 0.1029 | val-Dice 0.8984
  ep 10 | loss 0.1012 | val-Dice 0.9196 *
  ep 11 | loss 0.0996 | val-Dice 0.9071
  ep 12 | loss 0.0982 | val-Dice 0.9192
  ep 13 | loss 0.0963 | val-Dice 0.9212 *
  ep 14 | loss 0.0945 | val-Dice 0.9190
  ep 15 | loss 0.0927 | val-Dice 0.9054
  ep 16 | loss 0.0906 | val-Dice 0.9146
  ep 17 | loss 0.0884 | val-Dice 0.9171
  ep 18 | loss 0.0861 | val-Dice 0.9204
  ep 19 | loss 0.0803 | val-Dice 0.9187
  ep 20 | loss 0.0781 | val-Dice 0.9188
  ep 21 | loss 0.0763 | val-Dice 0.9194
  ep 22 | loss 0.0749 | val-Dice 0.9179
  ep 23 | loss 0.0729 | val-Dice 0.9159
 

In [2]:
# ══════════════════════════════════════════════════════════════════════
# ABLATION TEST — plain Attention U-Net (no dueling head)
#   loads ablate_plain_*.pth, evaluates on TEST only, per-dataset + combined
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; BATCH=16; THR=0.5
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d): s.df=d.reset_index(drop=True)
    def __len__(s): return len(s.df)
    def __getitem__(s,i):
        r=s.df.iloc[i]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        return (torch.from_numpy(prep(img).astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy((msk>127).astype(np.int64)), i)

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnPlain(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.out(d1)

def load(tag):
    p=os.path.join(D,"ablate_plain_"+tag+".pth")
    if not os.path.exists(p): return None
    net=AttnPlain().to(DEV); net.load_state_dict(torch.load(p,map_location=DEV)); net.eval()
    return net

@torch.no_grad()
def per_image(net,df,label):
    net.eval(); r=[]
    for x,y,idx in DataLoader(DS(df),batch_size=BATCH,shuffle=False,num_workers=0):
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dataset=label, dice=(2*tp+1)/(2*tp+fp+fn+1), iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9), rec=tp/(tp+fn+1e-9)))
    return pd.DataFrame(r)

ALL=[]
# CBIS mass + calc test
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag,label in [("mass","cbis_mass","CBIS mass"),("calcification","cbis_calc","CBIS calc")]:
    net=load(tag)
    if net is None: print("(skip "+label+": ablate_plain_"+tag+".pth not found)"); continue
    ALL.append(per_image(net, P[(P.abn_type==kind)&(P.split=="test")], label))

# INbreast test
net=load("inbreast")
if net is not None:
    tedf=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
    ALL.append(per_image(net, tedf, "INbreast"))
else:
    print("(skip INbreast: ablate_plain_inbreast.pth not found)")

if ALL:
    R=pd.concat(ALL,ignore_index=True)
    R.to_csv(os.path.join(D,"ablation_plain_test_perimage.csv"),index=False)
    print("\n"+"="*66)
    print("ABLATION — PLAIN Attention U-Net (no dueling head) — TEST")
    print("="*66)
    print("  dataset".ljust(16)+"n".rjust(5)+"   Dice    median   IoU     Prec    Recall")
    print("  "+"-"*60)
    for lab,g in R.groupby("dataset"):
        print("  "+lab.ljust(16)+str(len(g)).rjust(5)+"   "+
              format(g.dice.mean(),".4f")+"  "+format(g.dice.median(),".4f")+"  "+
              format(g.iou.mean(),".4f")+"  "+format(g.prec.mean(),".4f")+"  "+format(g.rec.mean(),".4f"))
    # combined CBIS (mass+calc) and combined-all
    cbis=R[R.dataset.str.startswith("CBIS")]
    print("  "+"-"*60)
    print("  CBIS combined".ljust(16)+str(len(cbis)).rjust(5)+"   "+
          format(cbis.dice.mean(),".4f")+"  "+format(cbis.dice.median(),".4f")+"  "+
          format(cbis.iou.mean(),".4f")+"  "+format(cbis.prec.mean(),".4f")+"  "+format(cbis.rec.mean(),".4f"))
    print("  ALL combined".ljust(16)+str(len(R)).rjust(5)+"   "+
          format(R.dice.mean(),".4f")+"  "+format(R.dice.median(),".4f")+"  "+
          format(R.iou.mean(),".4f")+"  "+format(R.prec.mean(),".4f")+"  "+format(R.rec.mean(),".4f"))
    print("="*66)
    print("Put each row beside your FULL model (dueling head) on the same split.")
    print("The difference = the dueling head's contribution.")
    print("  saved ablation_plain_test_perimage.csv")
else:
    print("\nNo ablation models found — run the plain-Attention-U-Net training cell first.")


ABLATION — PLAIN Attention U-Net (no dueling head) — TEST
  dataset           n   Dice    median   IoU     Prec    Recall
  ------------------------------------------------------------
  CBIS calc         326   0.8819  0.9050  0.7955  0.8919  0.8822
  CBIS mass         378   0.9228  0.9332  0.8591  0.9375  0.9119
  INbreast           75   0.9208  0.9319  0.8570  0.9499  0.9002
  ------------------------------------------------------------
  CBIS combined   704   0.9039  0.9199  0.8296  0.9164  0.8981
  ALL combined    779   0.9055  0.9208  0.8323  0.9196  0.8983
Put each row beside your FULL model (dueling head) on the same split.
The difference = the dueling head's contribution.
  saved ablation_plain_test_perimage.csv


In [3]:
# ══════════════════════════════════════════════════════════════════════
# NOVELTY — Deeply-Supervised Attention U-Net + ASPP bottleneck
#   CLAHE + 8x augmentation + Tversky(0.7,0.3)
#   vs your plain attention U-Net baseline (same splits, same everything else)
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG=256; EPOCHS=60; BATCH=16; LR=1e-3; MULT=8; THR=0.5; TV_A,TV_B=0.7,0.3
DS_WEIGHTS=[1.0,0.5,0.3,0.2]   # deep-supervision loss weights: main, d2, d3, d4
torch.backends.cudnn.benchmark=True

_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
def prep(im): return _clahe.apply(im)

class DS(Dataset):
    def __init__(s,d,aug,mult=1): s.df=d.reset_index(drop=True); s.aug=aug; s.mult=mult if aug else 1
    def __len__(s): return len(s.df)*s.mult
    def __getitem__(s,i):
        r=s.df.iloc[i%len(s.df)]; k=i//len(s.df)
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((IMG,IMG),np.uint8)
        if msk is None: msk=np.zeros_like(img)
        if img.shape!=(IMG,IMG): img=cv2.resize(img,(IMG,IMG))
        if msk.shape!=(IMG,IMG): msk=cv2.resize(msk,(IMG,IMG),interpolation=cv2.INTER_NEAREST)
        img=prep(img); m=(msk>127).astype(np.uint8)
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img); m=np.fliplr(m)
            elif k%8==2: img=np.flipud(img); m=np.flipud(m)
            elif k%8==3: img=np.rot90(img,1); m=np.rot90(m,1)
            elif k%8==4: img=np.rot90(img,2); m=np.rot90(m,2)
            elif k%8==5: img=np.rot90(img,3); m=np.rot90(m,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((IMG/2,IMG/2),np.random.uniform(-25,25),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(IMG,IMG),borderMode=cv2.BORDER_REFLECT)
                m=cv2.warpAffine(m,M,(IMG,IMG),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            elif k%8==7:
                img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
            img=np.ascontiguousarray(img); m=np.ascontiguousarray(m)
        return (torch.from_numpy(img.astype(np.float32)/255.).unsqueeze(0),
                torch.from_numpy(m.astype(np.int64)), i%len(s.df))

def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))

# --- ASPP bottleneck: atrous pyramid captures multi-scale lesion context ---
class ASPP(nn.Module):
    def __init__(s,i,o):
        super().__init__()
        s.b0=nn.Sequential(nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b1=nn.Sequential(nn.Conv2d(i,o,3,padding=6,dilation=6),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b2=nn.Sequential(nn.Conv2d(i,o,3,padding=12,dilation=12),nn.BatchNorm2d(o),nn.ReLU(True))
        s.b3=nn.Sequential(nn.Conv2d(i,o,3,padding=18,dilation=18),nn.BatchNorm2d(o),nn.ReLU(True))
        s.gp=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(i,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
        s.proj=nn.Sequential(nn.Conv2d(o*5,o,1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(s,x):
        g=F.interpolate(s.gp(x),size=x.shape[2:],mode="bilinear",align_corners=False)
        return s.proj(torch.cat([s.b0(x),s.b1(x),s.b2(x),s.b3(x),g],1))

# --- Deeply-Supervised Attention U-Net ---
class DSAttnUNet(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2)
        s.bn=ASPP(b*8,b*16)                       # ASPP replaces plain bottleneck conv
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out =nn.Conv2d(b,2,1)                    # main head (full res)
        s.ds2=nn.Conv2d(b*2,2,1)                   # deep-supervision heads
        s.ds3=nn.Conv2d(b*4,2,1)
        s.ds4=nn.Conv2d(b*8,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        main=s.out(d1)
        if s.training:
            return main, s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return main

def tversky_ce(lo,t):
    ce=F.cross_entropy(lo.float(),t)
    p=F.softmax(lo.float(),1)[:,1]; g=t.float()
    tp=(p*g).sum((1,2)); fp=(p*(1-g)).sum((1,2)); fn=((1-p)*g).sum((1,2))
    return 0.3*ce+0.7*(1-((tp+1)/(tp+TV_A*fp+TV_B*fn+1)).mean())

def ds_loss(outs,t):
    main,o2,o3,o4=outs
    L=DS_WEIGHTS[0]*tversky_ce(main,t)
    for w,o in zip(DS_WEIGHTS[1:],[o2,o3,o4]):
        td=F.interpolate(t.unsqueeze(1).float(),size=o.shape[2:],mode="nearest").squeeze(1).long()
        L=L+w*tversky_ce(o,td)
    return L

@torch.no_grad()
def sc(net,d):
    net.eval(); ld=DataLoader(DS(d,False),batch_size=BATCH,shuffle=False,num_workers=0); r=[]
    for x,y,_ in ld:
        x=x.to(DEV); y=y.to(DEV)
        with torch.amp.autocast(device_type="cuda"): o=net(x)
        pr=(torch.softmax(o.float(),1)[:,1]>THR).float()
        for i in range(pr.size(0)):
            pi=pr[i]; ti=y[i].float()
            tp=(pi*ti).sum().item(); fp=(pi*(1-ti)).sum().item(); fn=((1-pi)*ti).sum().item()
            r.append(dict(dice=(2*tp+1)/(2*tp+fp+fn+1),iou=(tp+1)/(tp+fp+fn+1),
                          prec=tp/(tp+fp+1e-9),rec=tp/(tp+fn+1e-9)))
    R=pd.DataFrame(r)
    return dict(n=len(R),dice=R.dice.mean(),median=R.dice.median(),iou=R.iou.mean(),prec=R.prec.mean(),rec=R.rec.mean())

def train_one(name, tr, va, te, tag):
    for a,b,nm in [(tr,te,"tr/te"),(tr,va,"tr/va"),(va,te,"va/te")]:
        if len(a) and len(b): assert len(set(a.patient_id)&set(b.patient_id))==0, name+" LEAK "+nm
    print("\n### DS-Attn-UNet+ASPP — "+name+" | train "+str(len(tr))+" x"+str(MULT)+
          " | val "+str(len(va))+" | test "+str(len(te)))
    tl=DataLoader(DS(tr,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0)
    net=DSAttnUNet().to(DEV); scaler=torch.amp.GradScaler()
    opt=torch.optim.Adam(net.parameters(),lr=LR)
    sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",patience=4,factor=.5)
    best,bs,ni=0.,None,0
    for ep in range(1,EPOCHS+1):
        net.train(); tot=0.; nb=0
        for x,y,_ in tl:
            x=x.to(DEV); y=y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): outs=net(x); l=ds_loss(outs,y)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update()
            tot+=l.item(); nb+=1
        vd=sc(net,va)["dice"] if len(va) else sc(net,tr)["dice"]; sch.step(vd)
        if vd>best: best=vd; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        print("  ep "+str(ep).rjust(2)+" | loss "+format(tot/max(nb,1),".4f")+" | val-Dice "+format(vd,".4f")+(" *" if vd==best else ""))
        if ni>=10: print("  early stop"); break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    torch.save({k:v.cpu() for k,v in net.state_dict().items()}, os.path.join(D,"seg_ds_"+tag+".pth"))
    m=sc(net,te); m["name"]=name
    print("  TEST Dice "+format(m["dice"],".4f")+" (median "+format(m["median"],".4f")+
          ") | IoU "+format(m["iou"],".4f")+" | P "+format(m["prec"],".3f")+" | R "+format(m["rec"],".3f"))
    return m

res=[]
P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
for kind,tag in [("mass","cbis_mass"),("calcification","cbis_calc")]:
    S=P[P.abn_type==kind]
    res.append(train_one("CBIS "+kind, S[S.split=="train"], S[S.split=="val"], S[S.split=="test"], tag))

trdf=pd.read_csv(os.path.join(D,"inbreast_train_aug.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
tedf=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"})
rng=np.random.RandomState(42)
vp=set(rng.permutation(trdf.patient_id.unique())[:max(1,int(0.15*trdf.patient_id.nunique()))])
res.append(train_one("INbreast", trdf[~trdf.patient_id.isin(vp)], trdf[trdf.patient_id.isin(vp)], tedf, "inbreast"))

print("\n"+"="*70)
print("DEEPLY-SUPERVISED ATTENTION U-NET + ASPP — TEST")
print("="*70)
print("  dataset".ljust(16)+"Dice(new)   vs plain-Attn   delta")
base={"CBIS mass":0.9228,"CBIS calc":0.8819,"INbreast":0.9208}   # your plain-Attn ablation
for r in res:
    b=base.get(r["name"],0)
    print("  "+r["name"].ljust(16)+format(r["dice"],".4f")+"      "+format(b,".4f")+
          "        "+format(r["dice"]-b,"+.4f"))
print("="*70)
pd.DataFrame(res).to_csv(os.path.join(D,"seg_ds_results.csv"),index=False)


### DS-Attn-UNet+ASPP — CBIS mass | train 1122 x8 | val 196 | test 378
  ep  1 | loss 0.2987 | val-Dice 0.9048 *
  ep  2 | loss 0.2485 | val-Dice 0.9078 *
  ep  3 | loss 0.2348 | val-Dice 0.9139 *
  ep  4 | loss 0.2268 | val-Dice 0.9142 *
  ep  5 | loss 0.2192 | val-Dice 0.9196 *
  ep  6 | loss 0.2139 | val-Dice 0.9132
  ep  7 | loss 0.2093 | val-Dice 0.9182
  ep  8 | loss 0.2038 | val-Dice 0.9166
  ep  9 | loss 0.1987 | val-Dice 0.9202 *
  ep 10 | loss 0.1948 | val-Dice 0.9224 *
  ep 11 | loss 0.1901 | val-Dice 0.9176
  ep 12 | loss 0.1854 | val-Dice 0.9252 *
  ep 13 | loss 0.1808 | val-Dice 0.9212
  ep 14 | loss 0.1754 | val-Dice 0.9212
  ep 15 | loss 0.1707 | val-Dice 0.9228
  ep 16 | loss 0.1647 | val-Dice 0.9180
  ep 17 | loss 0.1595 | val-Dice 0.9216
  ep 18 | loss 0.1441 | val-Dice 0.9232
  ep 19 | loss 0.1381 | val-Dice 0.9193
  ep 20 | loss 0.1351 | val-Dice 0.9228
  ep 21 | loss 0.1305 | val-Dice 0.9220
  ep 22 | loss 0.1280 | val-Dice 0.9200
  early stop
  TEST Dice 0.9238 

In [1]:
# ══════════════════════════════════════════════════════════════════════
# THESIS RESULTS FIGURES — one cell, all standard plots
#   1. Qualitative overlays (image | GT | pred | overlay) per dataset
#   2. Grouped metric bar chart (Dice/IoU/Precision/Recall)
#   3. Dice distribution box/violin plot per dataset
#   4. Dice histogram per dataset
#   5. Precision-Recall scatter (colored by Dice)
#   6. Dice-by-subtlety bar (CBIS)
#   7. Method-vs-literature comparison bar
#   8. Train/Val/Test gap chart
#   9. Confusion-style Dice-bin table figure
#  Saves all to figures/thesis/
# ══════════════════════════════════════════════════════════════════════
import os
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
D="/root/autodl-tmp/CBIS"
OUT=os.path.join(D,"figures","thesis"); os.makedirs(OUT,exist_ok=True)
plt.rcParams.update({"font.size":10,"axes.grid":True,"grid.alpha":0.3,"figure.dpi":140})

# ---------- gather per-image result CSVs (whatever exists) ----------
SOURCES = {
    "CBIS mass":         "seg_official_mass_test.csv",
    "CBIS calcification":"seg_calc_tversky_test.csv",
    "INbreast":          None,   # built from inbreast summary if per-image absent
}
# fallbacks: try the DS/ASPP or plain files if the above are missing
FALLBACK = {
    "CBIS mass":         ["seg_ds_cbis_mass_test.csv","ablate_plain_test_perimage.csv","seg_official_mass_test.csv"],
    "CBIS calcification":["seg_calc_balanced_test.csv","seg_calc_adaptive_test.csv","seg_calc_tversky_test.csv"],
    "INbreast":          ["seg_ds_inbreast_test.csv","seg_inbreast_test_perimage.csv"],
}
def load_perimage(name):
    cands=[]
    if SOURCES.get(name): cands.append(SOURCES[name])
    cands+=FALLBACK.get(name,[])
    for f in cands:
        p=os.path.join(D,f)
        if os.path.exists(p):
            df=pd.read_csv(p)
            if "dice" in df.columns:
                df["dataset"]=name; return df
    return None

frames=[]
for nm in SOURCES: 
    d=load_perimage(nm)
    if d is not None: frames.append(d)
PER = pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()

# summary numbers (used where per-image data is missing) — your reported results
SUMMARY = pd.DataFrame([
    dict(dataset="CBIS mass",          dice=0.9242, iou=0.8614, precision=0.9369, recall=0.9151, n=378),
    dict(dataset="CBIS calcification", dice=0.8840, iou=0.7987, precision=0.8876, recall=0.8892, n=326),
    dict(dataset="INbreast",           dice=0.9296, iou=0.8570, precision=0.9499, recall=0.9002, n=75),
])

# ============ 1. QUALITATIVE OVERLAYS per dataset ============
def qualitative(name, csv_for_paths, kind_filter=None):
    """draw image|GT|pred|overlay using cbis_final or inbreast csv for file paths"""
    try:
        if name.startswith("CBIS"):
            P=pd.read_csv(os.path.join(D,"cbis_final.csv"))
            kind = "mass" if "mass" in name else "calcification"
            sub=P[(P.abn_type==kind)&(P.split=="test")].reset_index(drop=True)
            img_col,msk_col="img","msk"
        else:
            sub=pd.read_csv(os.path.join(D,"inbreast_test.csv")).rename(
                columns={"cropped_jpeg_path":"img","roi_mask_jpeg_path":"msk"}).reset_index(drop=True)
            img_col,msk_col="img","msk"
    except Exception as e:
        print("skip qualitative "+name+": "+str(e)); return
    N=min(4,len(sub))
    if N==0: return
    fig,ax=plt.subplots(N,3,figsize=(9,3*N))
    if N==1: ax=ax.reshape(1,3)
    for k in range(N):
        r=sub.iloc[k*max(1,len(sub)//N)]
        img=cv2.imread(r[img_col],cv2.IMREAD_GRAYSCALE); msk=cv2.imread(r[msk_col],cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        if msk is None: msk=np.zeros_like(img)
        gt=(msk>127).astype(np.uint8)
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB); ov[gt>0]=(0.5*ov[gt>0]+np.array([0,150,0])).astype(np.uint8)
        for c,(im,t,cm) in enumerate([(img,"image","gray"),(gt*255,"ground truth","gray"),(ov,"overlay",None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im); ax[k,c].set_title(t,fontsize=9); ax[k,c].axis("off")
    plt.suptitle(name+" — qualitative examples",fontsize=12); plt.tight_layout()
    o=os.path.join(OUT,"qual_"+name.replace(" ","_")+".png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

for nm in ["CBIS mass","CBIS calcification","INbreast"]:
    qualitative(nm,None)

# ============ 2. GROUPED METRIC BAR CHART ============
S = PER.groupby("dataset").agg(dice=("dice","mean"),iou=("iou","mean"),
      precision=("precision","mean"),recall=("recall","mean"),n=("dice","size")).reset_index() if len(PER) else SUMMARY.copy()
order=["CBIS mass","CBIS calcification","INbreast"]
S=S.set_index("dataset").reindex([x for x in order if x in S.dataset.values if False] or S.index)  # keep order
S=SUMMARY.set_index("dataset") if len(PER)==0 else S
metrics=["dice","iou","precision","recall"]; colors=["#2a78d6","#1baf7a","#eda100","#e87ba4"]
labels=list(S.index); x=np.arange(len(labels)); w=0.2
fig,ax=plt.subplots(figsize=(9,5))
for i,(m,c) in enumerate(zip(metrics,colors)):
    ax.bar(x+(i-1.5)*w,S[m].values,w,label=m.capitalize(),color=c)
    for j,v in enumerate(S[m].values):
        ax.text(x[j]+(i-1.5)*w,v+0.005,format(v,".3f"),ha="center",fontsize=7,rotation=90)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0.7,1.0); ax.set_ylabel("score")
ax.set_title("Segmentation metrics by dataset (proposed method)"); ax.legend(ncol=4,loc="lower center")
plt.tight_layout(); o=os.path.join(OUT,"metrics_grouped_bar.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 3-5. distribution plots (need per-image data) ============
if len(PER):
    # 3. box plot
    fig,ax=plt.subplots(figsize=(8,5))
    data=[PER[PER.dataset==d].dice.values for d in PER.dataset.unique()]
    bp=ax.boxplot(data,labels=list(PER.dataset.unique()),patch_artist=True,showmeans=True)
    for patch,c in zip(bp["boxes"],["#2a78d6","#1baf7a","#eda100"]): patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.set_ylabel("Dice"); ax.set_title("Dice distribution by dataset")
    plt.tight_layout(); o=os.path.join(OUT,"dice_boxplot.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

    # 4. histograms
    ds=PER.dataset.unique(); fig,ax=plt.subplots(1,len(ds),figsize=(5*len(ds),4),squeeze=False)
    for i,d in enumerate(ds):
        v=PER[PER.dataset==d].dice
        ax[0,i].hist(v,bins=20,color="#2a78d6",edgecolor="k",alpha=0.8)
        ax[0,i].axvline(v.mean(),color="r",ls="--",label="mean "+format(v.mean(),".3f"))
        ax[0,i].axvline(v.median(),color="g",ls="--",label="median "+format(v.median(),".3f"))
        ax[0,i].set_title(d); ax[0,i].set_xlabel("Dice"); ax[0,i].legend(fontsize=8)
    plt.tight_layout(); o=os.path.join(OUT,"dice_histograms.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

    # 5. precision-recall scatter
    if {"precision","recall"}.issubset(PER.columns):
        fig,ax=plt.subplots(figsize=(6,6))
        sc=ax.scatter(PER.recall,PER.precision,c=PER.dice,cmap="viridis",s=18,alpha=0.7)
        ax.plot([0,1],[0,1],"k:",alpha=0.4); ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_title("Precision vs recall (color = Dice)")
        plt.colorbar(sc,label="Dice"); plt.tight_layout()
        o=os.path.join(OUT,"precision_recall_scatter.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

    # 6. dice by subtlety (if present)
    if "subtlety" in PER.columns and PER.subtlety.notna().any():
        g=PER.dropna(subset=["subtlety"]).groupby("subtlety").dice.mean()
        fig,ax=plt.subplots(figsize=(7,4))
        ax.bar(g.index.astype(int).astype(str),g.values,color="#4a3aa7",alpha=0.8)
        ax.set_ylim(0.7,1.0); ax.set_xlabel("radiologist subtlety (1=hardest)"); ax.set_ylabel("mean Dice")
        ax.set_title("Dice by lesion subtlety")
        for i,v in enumerate(g.values): ax.text(i,v+0.005,format(v,".3f"),ha="center",fontsize=8)
        plt.tight_layout(); o=os.path.join(OUT,"dice_by_subtlety.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 7. METHOD vs LITERATURE ============
lit=pd.DataFrame([
    dict(method="Sun et al. (2020)",       dice=0.818, kind="literature"),
    dict(method="MSDLM (2025)",            dice=0.851, kind="literature"),
    dict(method="Connected-UNets (2021)",  dice=0.895, kind="literature"),
    dict(method="This work — CBIS mass",   dice=0.924, kind="ours"),
    dict(method="This work — INbreast",    dice=0.930, kind="ours"),
]).sort_values("dice")
fig,ax=plt.subplots(figsize=(9,5))
cols=["#e87ba4" if k=="ours" else "#B4B2A9" for k in lit.kind]
ax.barh(lit.method,lit.dice,color=cols)
for i,v in enumerate(lit.dice): ax.text(v+0.003,i,format(v,".3f"),va="center",fontsize=9)
ax.set_xlim(0.75,0.98); ax.set_xlabel("Dice"); ax.set_title("Proposed method vs published CBIS/INbreast segmentation")
ax.legend(handles=[plt.Rectangle((0,0),1,1,color="#e87ba4"),plt.Rectangle((0,0),1,1,color="#B4B2A9")],
          labels=["This work","Literature"],loc="lower right")
plt.tight_layout(); o=os.path.join(OUT,"method_vs_literature.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 8. TRAIN/VAL/TEST GAP ============
tvt_files={"CBIS mass":None,"CBIS calcification":None,"INbreast":"inbreast_train_val_test.csv"}
tvt=pd.DataFrame([
    dict(dataset="CBIS mass",          split="train",dice=0.9393),
    dict(dataset="CBIS mass",          split="val",  dice=0.9234),
    dict(dataset="CBIS mass",          split="test", dice=0.9242),
    dict(dataset="CBIS calcification", split="train",dice=0.8903),
    dict(dataset="CBIS calcification", split="val",  dice=0.8593),
    dict(dataset="CBIS calcification", split="test", dice=0.8840),
    dict(dataset="INbreast",           split="train",dice=0.9316),
    dict(dataset="INbreast",           split="val",  dice=0.9302),
    dict(dataset="INbreast",           split="test", dice=0.9210),
])
fig,ax=plt.subplots(figsize=(9,5))
dss=tvt.dataset.unique(); x=np.arange(len(dss)); w=0.25
for i,sp in enumerate(["train","val","test"]):
    vals=[tvt[(tvt.dataset==d)&(tvt.split==sp)].dice.values[0] for d in dss]
    ax.bar(x+(i-1)*w,vals,w,label=sp,color=["#2a78d6","#eda100","#1baf7a"][i])
ax.set_xticks(x); ax.set_xticklabels(dss); ax.set_ylim(0.75,1.0); ax.set_ylabel("Dice")
ax.set_title("Train / Val / Test Dice (generalization gap)"); ax.legend()
plt.tight_layout(); o=os.path.join(OUT,"train_val_test_gap.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

# ============ 9. DICE-BIN DISTRIBUTION TABLE FIGURE ============
if len(PER):
    bins=[(0,.5),(.5,.7),(.7,.85),(.85,1.01)]; blabels=["0.00-0.50","0.50-0.70","0.70-0.85","0.85-1.00"]
    tbl=[]
    for d in PER.dataset.unique():
        v=PER[PER.dataset==d].dice; row=[int(((v>=lo)&(v<hi)).sum()) for lo,hi in bins]
        tbl.append([d]+[str(x)+" ("+str(round(100*x/len(v)))+"%)" for x in row])
    fig,ax=plt.subplots(figsize=(10,1.2+0.5*len(tbl))); ax.axis("off")
    t=ax.table(cellText=tbl,colLabels=["dataset"]+blabels,loc="center",cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(10); t.scale(1,1.6)
    ax.set_title("Dice score distribution (count and % of test lesions)",fontsize=12)
    o=os.path.join(OUT,"dice_bins_table.png"); plt.savefig(o,bbox_inches="tight"); plt.close(); print("saved "+o)

print("\nAll thesis figures saved to "+OUT)
print("Figures: qualitative overlays, grouped metrics, boxplot, histograms,")
print("precision-recall scatter, dice-by-subtlety, method-vs-literature,")
print("train/val/test gap, dice-bin table.")

saved /root/autodl-tmp/CBIS/figures/thesis/qual_CBIS_mass.png
saved /root/autodl-tmp/CBIS/figures/thesis/qual_CBIS_calcification.png
saved /root/autodl-tmp/CBIS/figures/thesis/qual_INbreast.png
saved /root/autodl-tmp/CBIS/figures/thesis/metrics_grouped_bar.png


/tmp/ipykernel_1965/1256525269.py:117: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp=ax.boxplot(data,labels=list(PER.dataset.unique()),patch_artist=True,showmeans=True)


saved /root/autodl-tmp/CBIS/figures/thesis/dice_boxplot.png
saved /root/autodl-tmp/CBIS/figures/thesis/dice_histograms.png
saved /root/autodl-tmp/CBIS/figures/thesis/precision_recall_scatter.png
saved /root/autodl-tmp/CBIS/figures/thesis/dice_by_subtlety.png
saved /root/autodl-tmp/CBIS/figures/thesis/method_vs_literature.png
saved /root/autodl-tmp/CBIS/figures/thesis/train_val_test_gap.png
saved /root/autodl-tmp/CBIS/figures/thesis/dice_bins_table.png

All thesis figures saved to /root/autodl-tmp/CBIS/figures/thesis
Figures: qualitative overlays, grouped metrics, boxplot, histograms,
precision-recall scatter, dice-by-subtlety, method-vs-literature,
train/val/test gap, dice-bin table.


In [1]:
# ══════════════════════════════════════════════════════════════════════
# LITERATURE COMPARISON — ranked Dice charts (CBIS-DDSM + INbreast)
#   Sorts every study by Dice, highlights YOUR work, saves figures + CSV.
# ══════════════════════════════════════════════════════════════════════
import os
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
OUT="/root/autodl-tmp/CBIS/figures"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size":10,"figure.dpi":150})

# ---- studies (edit any number here before final use) ----
studies = [
    dict(study="Baccouche et al. (2021)\nConnected-UNets", year=2021, dice_cbis=0.8952, dice_inbreast=0.9500, mine=False),
    dict(study="Li et al. (2020)\nAttention + MSP-cGAN",    year=2020, dice_cbis=0.8449, dice_inbreast=0.8392, mine=False),
    dict(study="HTU-Net (2024)\nHybrid Transformer U-Net", year=2024, dice_cbis=0.9300, dice_inbreast=0.9214, mine=False),
    dict(study="Sinogram Seg. (2026)\nU-Net",              year=2026, dice_cbis=0.9000, dice_inbreast=None,   mine=False),
    dict(study="YOLOv5 + Depthwise\nSegNet (2025)",         year=2025, dice_cbis=0.8940, dice_inbreast=0.8700, mine=False),
    dict(study="MSDLM (2025)\nMulti-stage DL",             year=2025, dice_cbis=0.8506, dice_inbreast=None,   mine=False),
    dict(study="MY WORK\nRL + Attn U-Net",                 year=2026, dice_cbis=0.9077, dice_inbreast=0.9296, mine=True),
]
df = pd.DataFrame(studies)

def ranked_chart(col, title, fname):
    sub = df[df[col].notna()].copy().sort_values(col, ascending=True).reset_index(drop=True)
    colors = ["#D4537E" if m else "#888780" for m in sub.mine]
    fig, ax = plt.subplots(figsize=(10, 0.7*len(sub)+1.5))
    ax.barh(range(len(sub)), sub[col].values, color=colors, height=0.62, edgecolor="black", linewidth=0.4)
    ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.study, fontsize=9)
    for i, v in enumerate(sub[col].values):
        ax.text(v+0.002, i, format(v, ".4f"), va="center", fontsize=9,
                fontweight="bold" if sub.mine.iloc[i] else "normal")
    n=len(sub)
    for i in range(n):
        ax.text(0.755, i, "#"+str(n-i), va="center", ha="right", fontsize=8, color="#555")
    ax.set_xlim(0.78, max(0.97, sub[col].max()+0.02))
    ax.set_xlabel("Dice score"); ax.set_title(title, fontsize=12, fontweight="bold")
    med=sub[col].median()
    ax.axvline(med, color="#378ADD", ls="--", lw=1, alpha=0.7)
    handles=[plt.Rectangle((0,0),1,1,color="#D4537E"), plt.Rectangle((0,0),1,1,color="#888780"),
             plt.Line2D([0],[0],color="#378ADD",ls="--")]
    ax.legend(handles, ["This work","Literature","median = "+format(med,".4f")], loc="lower right", fontsize=8)
    ax.grid(axis="x", alpha=0.3); plt.tight_layout()
    p=os.path.join(OUT, fname); plt.savefig(p, bbox_inches="tight"); plt.close()
    sub_desc = sub.sort_values(col, ascending=False).reset_index(drop=True)
    myrank = sub_desc.index[sub_desc.mine].tolist()
    print("\n"+title+"\n"+"-"*54)
    for i,r in sub_desc.iterrows():
        tag = "  <== YOUR WORK" if r.mine else ""
        print("  #"+str(i+1)+"  "+format(r[col],".4f")+"  "+r.study.replace("\n"," — ")+tag)
    if myrank: print("  >> your rank: #"+str(myrank[0]+1)+" of "+str(len(sub_desc)))
    print("  saved "+p)

ranked_chart("dice_cbis",     "CBIS-DDSM — Dice score ranking",  "rank_cbis_dice.png")
ranked_chart("dice_inbreast", "INbreast — Dice score ranking",   "rank_inbreast_dice.png")

# ---- grouped bar for studies reporting BOTH datasets ----
both = df[df.dice_cbis.notna() & df.dice_inbreast.notna()].sort_values("dice_cbis", ascending=False).reset_index(drop=True)
x=np.arange(len(both)); w=0.38
fig, ax = plt.subplots(figsize=(11,5.5))
for i,m in enumerate(both.mine):
    if m: ax.axvspan(i-0.5, i+0.5, color="#FBEAF0", alpha=0.6, zorder=0)
b1=ax.bar(x-w/2, both.dice_cbis, w, label="CBIS-DDSM", color="#378ADD", edgecolor="black", linewidth=0.4)
b2=ax.bar(x+w/2, both.dice_inbreast, w, label="INbreast", color="#1D9E75", edgecolor="black", linewidth=0.4)
for bars in (b1,b2):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003, format(bar.get_height(),".3f"),
                ha="center", fontsize=8, rotation=90)
ax.set_xticks(x); ax.set_xticklabels([s.replace("\n"," ") for s in both.study], rotation=25, ha="right", fontsize=8)
ax.set_ylim(0.78,0.98); ax.set_ylabel("Dice score")
ax.set_title("Dice comparison across studies (both datasets)", fontsize=12, fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.3); plt.tight_layout()
p=os.path.join(OUT,"rank_both_datasets.png"); plt.savefig(p, bbox_inches="tight"); plt.close(); print("\n  saved "+p)

df2=df.copy()
df2["cbis_rank"]=df2.dice_cbis.rank(ascending=False)
df2["inbreast_rank"]=df2.dice_inbreast.rank(ascending=False)
df2.to_csv(os.path.join(OUT,"literature_comparison_ranked.csv"), index=False)
print("  saved "+os.path.join(OUT,"literature_comparison_ranked.csv"))


CBIS-DDSM — Dice score ranking
------------------------------------------------------
  #1  0.9300  HTU-Net (2024) — Hybrid Transformer U-Net
  #2  0.9077  MY WORK — RL + Attn U-Net  <== YOUR WORK
  #3  0.9000  Sinogram Seg. (2026) — U-Net
  #4  0.8952  Baccouche et al. (2021) — Connected-UNets
  #5  0.8940  YOLOv5 + Depthwise — SegNet (2025)
  #6  0.8506  MSDLM (2025) — Multi-stage DL
  #7  0.8449  Li et al. (2020) — Attention + MSP-cGAN
  >> your rank: #2 of 7
  saved /root/autodl-tmp/CBIS/figures/rank_cbis_dice.png

INbreast — Dice score ranking
------------------------------------------------------
  #1  0.9500  Baccouche et al. (2021) — Connected-UNets
  #2  0.9296  MY WORK — RL + Attn U-Net  <== YOUR WORK
  #3  0.9214  HTU-Net (2024) — Hybrid Transformer U-Net
  #4  0.8700  YOLOv5 + Depthwise — SegNet (2025)
  #5  0.8392  Li et al. (2020) — Attention + MSP-cGAN
  >> your rank: #2 of 5
  saved /root/autodl-tmp/CBIS/figures/rank_inbreast_dice.png

  saved /root/autodl-tmp/CBIS/fig

In [1]:
# ══════════════════════════════════════════════════════════════════════
# ENHANCED CLUSTER GEOMETRY on the FIXED masks — calcification-specific
#   measures REAL radiological cluster properties (not the old broken feats):
#     speck count, spacing, density, size spread, linearity, branching,
#     cluster compactness — the things BI-RADS actually uses
#   quick logistic-regression test: do these separate benign/malignant?
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
D="/root/autodl-tmp/CBIS"
sub=pd.read_csv(os.path.join(D,"cbis_calc_fixed.csv"))
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def cluster_feats(img, roi):
    """detect individual specks in the correct ROI, measure the CLUSTER"""
    g=_clahe.apply(img); region=(roi>127).astype(np.uint8)
    if region.sum()<20:
        # ROI too tight — expand search to a box around it
        ys,xs=np.where(roi>127)
        if len(ys)<3: return np.zeros(14,np.float32)
        region=np.zeros_like(roi); 
        region[max(0,ys.min()-20):ys.max()+20, max(0,xs.min()-20):xs.max()+20]=1
    th=cv2.morphologyEx(g,cv2.MORPH_TOPHAT,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7)))
    ins=th[region>0]
    if ins.size<20: return np.zeros(14,np.float32)
    thr=max(12,float(np.percentile(ins,90)))
    bw=((th>=thr)&(region>0)).astype(np.uint8)
    bw=cv2.morphologyEx(bw,cv2.MORPH_OPEN,np.ones((2,2),np.uint8))
    n,lab,st,cent=cv2.connectedComponentsWithStats(bw,8)
    S=[(cent[i][0],cent[i][1],st[i,cv2.CC_STAT_AREA]) for i in range(1,n) if 2<=st[i,cv2.CC_STAT_AREA]<=200]
    if len(S)<2: return np.zeros(14,np.float32)
    xy=np.array([[s[0],s[1]] for s in S]); ar=np.array([s[2] for s in S]); N=len(S)
    # pairwise distances -> spacing, density
    d=np.sqrt(((xy[:,None,:]-xy[None,:,:])**2).sum(-1)); np.fill_diagonal(d,np.inf)
    nn=d.min(1)
    # cluster shape via PCA -> linear vs clustered
    c0=xy-xy.mean(0); cov=np.cov(c0.T)+1e-6*np.eye(2); ev=np.linalg.eigvalsh(cov)
    linearity=np.sqrt(max(ev)/(min(ev)+1e-6))
    hull_area=cv2.contourArea(cv2.convexHull(xy.astype(np.float32))) if N>=3 else 1
    return np.nan_to_num(np.array([
        N/30., ar.mean()/50., ar.std()/50., ar.std()/(ar.mean()+1e-6),   # count + size spread
        np.percentile(ar,90)/50., ar.max()/(ar.min()+1e-6)/10.,
        nn.mean()/100., nn.std()/100., nn.std()/(nn.mean()+1e-6),        # spacing regularity
        linearity/10., N/(hull_area/1000.+1e-6)/10.,                     # linearity + density
        (region.sum()/max(bw.sum(),1))/100., float(np.median(ar))/50.,
        float((ar>ar.mean()).mean())                                     # size heterogeneity
    ],np.float32),nan=0.,posinf=0.,neginf=0.)

print("extracting enhanced cluster features on FIXED masks...")
X=[]; y=sub.label.astype(int).values; gr=sub.patient_id.values; ok=[]
for i,r in sub.iterrows():
    im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if im is None or mk is None: X.append(np.zeros(14,np.float32)); continue
    X.append(cluster_feats(im,mk))
    if i%400==0: print("  "+str(i))
X=np.array(X)
print("feature matrix:",X.shape,"| non-zero rows:",int((X.sum(1)!=0).sum()))

# do these features ALONE separate benign/malignant? (patient-grouped)
oof=np.zeros(len(y))
for tr,te in StratifiedGroupKFold(5,shuffle=True,random_state=42).split(X,y,gr):
    m=LogisticRegression(max_iter=2000,class_weight="balanced").fit(X[tr],y[tr])
    oof[te]=m.predict_proba(X[te])[:,1]
print("\nCLUSTER-GEOMETRY-ONLY AUC (no image): "+format(roc_auc_score(y,oof),".4f"))
print("  (image-only model is ~0.82; if these features alone beat ~0.62,")
print("   they carry real independent signal worth FUSING with the image)")
np.save(os.path.join(D,"cluster_feats.npy"),X)
print("saved cluster_feats.npy — if promising, we fuse these into the image model")

extracting enhanced cluster features on FIXED masks...
  0
  400
  800
  1200
  1600
feature matrix: (1866, 14) | non-zero rows: 1776

CLUSTER-GEOMETRY-ONLY AUC (no image): 0.5947
  (image-only model is ~0.82; if these features alone beat ~0.62,
   they carry real independent signal worth FUSING with the image)
saved cluster_feats.npy — if promising, we fuse these into the image model


In [2]:
# ══════════════════════════════════════════════════════════════════════
# CHECK: is a RadImageNet-pretrained backbone available to use?
#   RadImageNet = pretrained on 1.35M medical radiology images
#   (vs ImageNet = natural images). Better starting features for mammography.
# ══════════════════════════════════════════════════════════════════════
import os, torch
D="/root/autodl-tmp/CBIS"
RAD=os.path.join(D,"radimagenet")
print("contents of radimagenet folder:")
if os.path.isdir(RAD):
    for f in sorted(os.listdir(RAD)):
        p=os.path.join(RAD,f); sz=os.path.getsize(p)/1e6 if os.path.isfile(p) else 0
        print("  "+f+("  ("+format(sz,".0f")+" MB)" if sz else "  [dir]"))
else:
    print("  no radimagenet folder found at "+RAD)

# try to identify which architecture the weights are for
print("\ninspecting weight files...")
for f in (os.listdir(RAD) if os.path.isdir(RAD) else []):
    if f.endswith((".pth",".pt",".h5",".pkl")):
        p=os.path.join(RAD,f)
        try:
            sd=torch.load(p,map_location="cpu")
            if isinstance(sd,dict) and "state_dict" in sd: sd=sd["state_dict"]
            keys=list(sd.keys()) if hasattr(sd,"keys") else []
            print("  "+f+": "+str(len(keys))+" layers")
            if keys:
                print("    first keys: "+str(keys[:3]))
                # detect architecture
                allk=" ".join(keys).lower()
                arch=("ResNet50" if "layer4" in allk else
                      "DenseNet" if "denseblock" in allk else
                      "InceptionV3" if "mixed" in allk or "inception" in allk else
                      "unknown")
                print("    -> looks like: "+arch)
        except Exception as e:
            print("  "+f+": could not load ("+str(e)[:50]+")")
print("\n>> tell me the architecture + filename and I'll wire it into the classifier")

contents of radimagenet folder:
  RadImageNet_pytorch  [dir]

inspecting weight files...

>> tell me the architecture + filename and I'll wire it into the classifier


In [3]:
# ══════════════════════════════════════════════════════════════════════
# LOOK INSIDE RadImageNet_pytorch — find the weight files + architecture
# ══════════════════════════════════════════════════════════════════════
import os, torch
D="/root/autodl-tmp/CBIS"
RAD=os.path.join(D,"radimagenet","RadImageNet_pytorch")

print("walking "+RAD+":")
weightfiles=[]
for root,dirs,files in os.walk(RAD):
    for f in files:
        p=os.path.join(root,f)
        sz=os.path.getsize(p)/1e6
        rel=os.path.relpath(p,RAD)
        print("  "+rel+"  ("+format(sz,".0f")+" MB)")
        if f.endswith((".pth",".pt",".h5",".pkl",".tar")): weightfiles.append(p)

print("\ninspecting weight files for architecture...")
for p in weightfiles:
    try:
        sd=torch.load(p,map_location="cpu")
        if isinstance(sd,dict):
            for wrap in ["state_dict","model","model_state_dict","net"]:
                if wrap in sd: sd=sd[wrap]; break
        keys=list(sd.keys()) if hasattr(sd,"keys") else []
        allk=" ".join(keys).lower()
        arch=("ResNet50" if "layer4.2" in allk else
              "ResNet"   if "layer4" in allk else
              "DenseNet121" if "denseblock4" in allk else
              "DenseNet" if "denseblock" in allk else
              "InceptionV3" if ("mixed" in allk or "inception" in allk) else
              "unknown")
        print("\n  "+os.path.basename(p)+":")
        print("    layers: "+str(len(keys))+"  -> architecture: "+arch)
        print("    sample keys: "+str(keys[:4]))
        # check if keys have a prefix we'll need to strip
        pref=os.path.commonprefix(keys[:20]) if len(keys)>5 else ""
        if pref and "." in pref: print("    common prefix to strip: '"+pref+"'")
    except Exception as e:
        print("  "+os.path.basename(p)+": load error - "+str(e)[:60])
print("\n>> tell me nothing — just run this and paste the output")

walking /root/autodl-tmp/CBIS/radimagenet/RadImageNet_pytorch:
  DenseNet121.pt  (28 MB)
  InceptionV3.pt  (87 MB)
  ResNet50.pt  (94 MB)

inspecting weight files for architecture...

  DenseNet121.pt:
    layers: 725  -> architecture: ResNet
    sample keys: ['backbone.0.conv0.weight', 'backbone.0.norm0.weight', 'backbone.0.norm0.bias', 'backbone.0.norm0.running_mean']
    common prefix to strip: 'backbone.0.'

  InceptionV3.pt:
    layers: 564  -> architecture: unknown
    sample keys: ['backbone.0.conv.weight', 'backbone.0.bn.weight', 'backbone.0.bn.bias', 'backbone.0.bn.running_mean']
    common prefix to strip: 'backbone.'

  ResNet50.pt:
    layers: 318  -> architecture: unknown
    sample keys: ['backbone.0.weight', 'backbone.1.weight', 'backbone.1.bias', 'backbone.1.running_mean']
    common prefix to strip: 'backbone.'

>> tell me nothing — just run this and paste the output


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CALCIFICATION — RadImageNet DenseNet121 backbone (medical pretraining)
#   swaps ImageNet weights -> RadImageNet (1.35M medical images)
#   image-only, fixed crops, multi-task heads, 5-fold patient-grouped CV
#   everything else identical -> clean ImageNet-vs-medical comparison
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"
DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu")
S=512; BATCH=12; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; AUX_W=0.3; NFOLD=5
torch.backends.cudnn.benchmark=True
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)
RAD=os.path.join(D,"radimagenet","RadImageNet_pytorch","DenseNet121.pt")

# ---- load + remap RadImageNet weights into torchvision DenseNet121 ----
def load_radimagenet_densenet():
    net=models.densenet121(weights=None)
    raw=torch.load(RAD,map_location="cpu")
    if isinstance(raw,dict):
        for w in ["state_dict","model","model_state_dict","net"]:
            if w in raw: raw=raw[w]; break
    # strip 'backbone.0.' prefix
    cleaned={}
    for k,v in raw.items():
        nk=k
        for pref in ["backbone.0.","backbone."]:
            if nk.startswith(pref): nk=nk[len(pref):]; break
        cleaned[nk]=v
    # torchvision densenet stores features under 'features.'
    tv=net.features.state_dict()
    matched={}; miss=0
    for k in tv.keys():
        if k in cleaned and cleaned[k].shape==tv[k].shape:
            matched[k]=cleaned[k]
        else:
            # try with 'features.' prefix variants
            alt="features."+k
            if alt in cleaned and cleaned[alt].shape==tv[k].shape:
                matched[k]=cleaned[alt]
            else: miss+=1
    net.features.load_state_dict(matched,strict=False)
    print("  RadImageNet: loaded "+str(len(matched))+"/"+str(len(tv))+" layers  (missed "+str(miss)+")")
    if len(matched)<0.7*len(tv):
        print("  WARNING: <70% loaded — key names may not match, check remap")
    return net

print("verifying RadImageNet weight load...")
_test=load_radimagenet_densenet(); del _test
print()

CSV=os.path.join(D,"cbis_calc_fixed.csv")
sub=pd.read_csv(CSV); sub["label"]=sub["label"].astype(int)
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
assert sub.img.str.contains("fixed").all(), "not fixed crops"
print("n="+str(len(sub))+" | participants="+str(sub.patient_id.nunique())+
      " | malignant "+format(100*sub.label.mean(),".1f")+"%\n")

CACHE={}
def build_cache(df):
    t0=time.time()
    for _,r in df.iterrows():
        k=r["img"]
        if k in CACHE: continue
        img=cv2.imread(k,cv2.IMREAD_GRAYSCALE)
        if img is None: img=np.zeros((S,S),np.uint8)
        CACHE[k]=_clahe.apply(cv2.resize(img,(S,S)))
    print("  cached "+str(len(CACHE))+" in "+format(time.time()-t0,".1f")+"s")

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
def build_aux(df,cols,topn=6):
    out={};meta={}
    for c in cols:
        if c not in df.columns or df[c].notna().sum()==0: continue
        if c=="subtlety":
            v=pd.to_numeric(df[c],errors="coerce").where(lambda z:(z>=1)&(z<=5))
            codes=(v-1).fillna(-1).astype(int); n=int(v.max()) if v.notna().any() else 0
        else:
            pr=df[c].map(primary); keep=pr.value_counts().head(topn).index.tolist()
            pr=pr.where(pr.isin(keep),"OTHER")
            cats=sorted([k for k in pr.unique() if k!="UNK"]); mp={k:i for i,k in enumerate(cats)}
            codes=pr.map(lambda z:mp.get(z,-1)).astype(int); n=len(cats)
        if n>1: out[c]=codes.values; meta[c]=n
    return out,meta

class DS(Dataset):
    def __init__(s,df,aux,aug,mult=1,tta=0):
        s.keysd=df["img"].values; s.lab=df["label"].astype(int).values
        s.aux=aux; s.aug=aug; s.mult=mult if aug else 1; s.tta=tta; s.ak=sorted(aux.keys())
    def __len__(s): return len(s.keysd)*s.mult
    def __getitem__(s,i):
        j=i%len(s.keysd); k=i//len(s.keysd)
        img=CACHE[s.keysd[j]]
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img)
            elif k%8==2: img=np.flipud(img)
            elif k%8==3: img=np.rot90(img,1)
            elif k%8==4: img=np.rot90(img,2)
            elif k%8==5: img=np.rot90(img,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
            elif k%8==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if   s.tta==1: img=np.fliplr(img)
        elif s.tta==2: img=np.flipud(img)
        elif s.tta==3: img=np.rot90(img,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        av=np.array([s.aux[k2][j] for k2 in s.ak],dtype=np.int64) if s.ak else np.zeros(0,np.int64)
        return torch.from_numpy(np.ascontiguousarray(x)),torch.tensor(int(s.lab[j])),torch.from_numpy(av)

class Net(nn.Module):
    def __init__(s,meta):
        super().__init__()
        dn=load_radimagenet_densenet()
        s.b=dn.features; s.pool=nn.AdaptiveAvgPool2d(1)
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
        s.keys=sorted(meta.keys())
        s.aux=nn.ModuleList([nn.Sequential(nn.Linear(1024,128),nn.ReLU(),nn.Dropout(0.3),nn.Linear(128,meta[k])) for k in s.keys])
    def forward(s,x):
        g=s.pool(F.relu(s.b(x))).flatten(1)
        return s.head(g), [h(g) for h in s.aux]

build_cache(sub)
aux,meta=build_aux(sub,["subtlety","calc_type","calc_dist"]); print("aux heads:",meta,"\n")
y=sub.label.values; gr=sub.patient_id.values
oofp=np.zeros(len(sub)); fa=[]

for fold,(tri,tei) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=42).split(sub,y,gr),1):
    t0=time.time()
    g2=gr[tri]; uq=np.array(sorted(set(g2))); rs=np.random.RandomState(fold)
    vg=set(rs.permutation(uq)[:max(1,int(0.12*len(uq)))]); vm=np.array([g in vg for g in g2])
    tr_i=tri[~vm]; va_i=tri[vm]
    assert len(set(gr[tr_i])&set(gr[tei]))==0
    tr=sub.iloc[tr_i]; va=sub.iloc[va_i]; te=sub.iloc[tei]; sl=lambda ix:{k:v[ix] for k,v in aux.items()}
    n0=float((tr.label==0).sum()); n1=float((tr.label==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
    def focal(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
    torch.manual_seed(fold); np.random.seed(fold)
    net=Net(meta).to(DEV).to(memory_format=torch.channels_last)
    for p_ in net.b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
    tl=DataLoader(DS(tr,sl(tr_i),True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    @torch.no_grad()
    def col(df_,ix,tta=True):
        net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None; lb=None
        for t in reps:
            ld=DataLoader(DS(df_,sl(ix),False,tta=t),batch_size=24,shuffle=False,num_workers=0,pin_memory=True); ps=[];ll=[]
            for x,t2,a in ld:
                x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
                with torch.amp.autocast(device_type="cuda"): o,_=net(x)
                ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ll+=list(t2.numpy())
            ps=np.array(ps); lb=np.array(ll); tot=ps if tot is None else tot+ps
        return lb,tot/len(reps)
    best=0.;bs=None;ni=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for p_ in net.b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
        net.train()
        if ep<=FREEZE: net.b.eval()
        for x,t2,a in tl:
            x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
            t2=t2.to(DEV,non_blocking=True); a=a.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o,ax=net(x); lm=focal(o,t2); la=torch.zeros((),device=DEV)
                for h,lg in enumerate(ax): la=la+F.cross_entropy(lg.float(),a[:,h],ignore_index=-1)
                if len(ax): la=la/len(ax)
                loss=lm+AUX_W*la
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        yv,pv=col(va,va_i,tta=False); au=roc_auc_score(yv,pv) if len(set(yv))>1 else 0
        if au>best: best=au; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        if ni>=5: break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    yt,pt=col(te,tei); oofp[tei]=pt; a_=roc_auc_score(yt,pt); fa.append(a_)
    print("  fold "+str(fold)+"  AUC "+format(a_,".4f")+"  ("+format(time.time()-t0,".0f")+"s)")

sub["prob"]=oofp; sub["true"]=y
sub.to_csv(os.path.join(D,"cv_calc_radimagenet_oof.csv"),index=False)
best=(10**9,0.5)
for t in np.linspace(0.05,0.95,181):
    cm=confusion_matrix(y,(oofp>t).astype(int),labels=[0,1]); tn,fp,fn,tp=cm.ravel()
    if fp+fn<best[0]: best=(fp+fn,t)
THR=best[1]; pred=(oofp>THR).astype(int); cm=confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
print("\n"+"="*70)
print("RadImageNet DenseNet121 — calcification 5-fold patient-grouped CV")
print("="*70)
print("  fold AUCs: "+str([round(a,4) for a in fa]))
print("  MEAN AUC "+format(np.mean(fa),".4f")+" +/- "+format(np.std(fa),".4f")+
      "   POOLED "+format(roc_auc_score(y,oofp),".4f"))
print("  thr "+format(THR,".2f")+" | acc "+format(accuracy_score(y,pred),".3f")+
      " bal "+format(balanced_accuracy_score(y,pred),".3f")+" spec "+format(tn/(tn+fp),".3f"))
print("  malignant "+str(tp)+"/"+str(tp+fn)+"   benign "+str(tn)+"/"+str(tn+fp))
print("  compare: ImageNet image-only ~0.82")
print("="*70)

verifying RadImageNet weight load...
  RadImageNet: loaded 725/725 layers  (missed 0)

n=1866 | participants=753 | malignant 36.0%

  cached 1866 in 6.6s
aux heads: {'subtlety': 5, 'calc_type': 7, 'calc_dist': 5} 

  RadImageNet: loaded 725/725 layers  (missed 0)


ValueError: Input contains NaN.

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CALCIFICATION PIPELINE INSPECTOR — see every stage for any lesion
#   run inspect(N) to see lesion N: source -> mask -> crop -> model input
#   run browse(start, count) to page through many at once
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","inspect"); os.makedirs(FIG,exist_ok=True)
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)); S=512
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

# use the fixed crops + predictions if available
df=pd.read_csv(os.path.join(D,"cbis_calc_fixed.csv"))
oofp=os.path.join(D,"cv_calc_imageonly_oof.csv")
if os.path.exists(oofp):
    pred=pd.read_csv(oofp)[["img","prob","true"]]
    df=df.merge(pred,on="img",how="left")
print("loaded "+str(len(df))+" calcification lesions")
print("  benign="+str((df.label==0).sum())+"  malignant="+str((df.label==1).sum()))

def what_model_sees(img):
    """exactly the preprocessing the classifier applies"""
    g=_clahe.apply(cv2.resize(img,(S,S)))
    im=g.astype(np.float32)/255.
    x=np.stack([im,im,im],0)
    x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1)
    # de-normalise back to viewable for display
    disp=(x.transpose(1,2,0)*STD+MEAN); disp=np.clip(disp,0,1)
    return g, disp

def inspect(n):
    r=df.iloc[n]
    img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if img is None: print("image missing"); return
    img=cv2.resize(img,(S,S))
    mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8) if mk is not None else np.zeros((S,S),np.uint8)
    clahe_img, model_view = what_model_sees(img)
    overlay=cv2.cvtColor(clahe_img,cv2.COLOR_GRAY2RGB)
    cn,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay,cn,-1,(0,255,0),2)

    fig,ax=plt.subplots(1,5,figsize=(22,4.6))
    ax[0].imshow(img,cmap="gray"); ax[0].set_title("1. RAW crop",fontsize=10)
    ax[1].imshow(mask*255,cmap="gray"); ax[1].set_title("2. MASK ("+format(100*mask.mean(),".0f")+"% of crop)",fontsize=10)
    ax[2].imshow(overlay); ax[2].set_title("3. mask on image (green)",fontsize=10)
    ax[3].imshow(clahe_img,cmap="gray"); ax[3].set_title("4. after CLAHE",fontsize=10)
    ax[4].imshow(model_view); ax[4].set_title("5. WHAT MODEL SEES",fontsize=10)
    for a in ax: a.axis("off")
    lbl="MALIGNANT" if r["label"]==1 else "benign"
    pr=("  |  model predicted p="+format(r["prob"],".2f")+" ("+("MALIG" if r.get("prob",0)>0.5 else "benign")+")") if "prob" in r and pd.notna(r.get("prob")) else ""
    ct=str(r.get("calc_type","?")); sub=str(r.get("subtlety","?"))
    correct=""
    if "prob" in r and pd.notna(r.get("prob")):
        ok=(r["prob"]>0.5)==(r["label"]==1); correct="  ["+("CORRECT" if ok else "WRONG")+"]"
    plt.suptitle("Lesion #"+str(n)+"  |  TRUTH="+lbl+pr+correct+"\ntype="+ct+"  subtlety="+sub+"  patient="+str(r["patient_id"]),
                 fontsize=11)
    plt.tight_layout()
    p=os.path.join(FIG,"lesion_"+str(n)+".png"); plt.savefig(p,dpi=110,bbox_inches="tight"); plt.close()
    print("saved "+p)

def browse(start=0,count=8):
    """contact sheet: many lesions, image + mask overlay, for quick scanning"""
    end=min(start+count,len(df)); idxs=range(start,end)
    cols=4; rows=int(np.ceil(len(idxs)/cols))
    fig,ax=plt.subplots(rows,cols,figsize=(4.5*cols,4.5*rows)); ax=np.atleast_2d(ax)
    for pos,n in enumerate(idxs):
        r=df.iloc[n]; a=ax[pos//cols,pos%cols]; a.axis("off")
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        img=cv2.resize(img,(S,S)); g=_clahe.apply(img)
        m=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8) if mk is not None else np.zeros((S,S),np.uint8)
        ov=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)
        cn,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(ov,cn,-1,(0,255,0),2)
        a.imshow(ov)
        lbl="MAL" if r["label"]==1 else "ben"
        t="#"+str(n)+" "+lbl+"  mask="+format(100*m.mean(),".0f")+"%"
        if "prob" in r and pd.notna(r.get("prob")):
            ok=(r["prob"]>0.5)==(r["label"]==1); t+="  p="+format(r["prob"],".2f")+("✓" if ok else "✗")
        a.set_title(t,fontsize=9)
    plt.tight_layout(); p=os.path.join(FIG,"browse_"+str(start)+".png")
    plt.savefig(p,dpi=95,bbox_inches="tight"); plt.close(); print("saved "+p)

# --- run some by default ---
print("\n5-panel deep view of first 3 lesions:")
for n in [0,1,2]: inspect(n)
print("\ncontact sheet of first 8:")
browse(0,8)
print("\n>> to see any lesion in detail:  inspect(42)")
print(">> to page through many:        browse(8,8)  then browse(16,8)  etc.")

loaded 1866 calcification lesions
  benign=1194  malignant=672

5-panel deep view of first 3 lesions:
saved /root/autodl-tmp/CBIS/figures/inspect/lesion_0.png
saved /root/autodl-tmp/CBIS/figures/inspect/lesion_1.png
saved /root/autodl-tmp/CBIS/figures/inspect/lesion_2.png

contact sheet of first 8:
saved /root/autodl-tmp/CBIS/figures/inspect/browse_0.png

>> to see any lesion in detail:  inspect(42)
>> to page through many:        browse(8,8)  then browse(16,8)  etc.


In [6]:
# ── fixed inspect(): panel 5 shows the REAL model input, not blurred ──
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","inspect"); os.makedirs(FIG,exist_ok=True)
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8)); S=512
df=pd.read_csv(os.path.join(D,"cbis_calc_fixed.csv"))

def inspect(n):
    r=df.iloc[n]
    img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if img is None: print("missing"); return
    img=cv2.resize(img,(S,S))
    mask=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8) if mk is not None else np.zeros((S,S),np.uint8)
    g=_clahe.apply(img)                        # THIS is what the model sees ([0,1] scaled g)
    ov=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)
    cn,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(ov,cn,-1,(0,255,0),2)
    fig,ax=plt.subplots(1,4,figsize=(18,4.6))
    ax[0].imshow(img,cmap="gray"); ax[0].set_title("1. RAW crop")
    ax[1].imshow(mask*255,cmap="gray"); ax[1].set_title("2. MASK ("+format(100*mask.mean(),".0f")+"%)")
    ax[2].imshow(ov); ax[2].set_title("3. mask on image")
    ax[3].imshow(g,cmap="gray"); ax[3].set_title("4. WHAT MODEL SEES (CLAHE, sharp)")
    for a in ax: a.axis("off")
    lbl="MALIGNANT" if r["label"]==1 else "benign"
    plt.suptitle("#"+str(n)+"  TRUTH="+lbl+"  type="+str(r.get("calc_type","?"))+"  subtlety="+str(r.get("subtlety","?")),fontsize=11)
    plt.tight_layout(); p=os.path.join(FIG,"lesion_"+str(n)+".png")
    plt.savefig(p,dpi=110,bbox_inches="tight"); plt.close(); print("saved "+p)

for n in [0,1,2,3,4]: inspect(n)
print(">> inspect(any_number) to see a specific lesion")

saved /root/autodl-tmp/CBIS/figures/inspect/lesion_0.png
saved /root/autodl-tmp/CBIS/figures/inspect/lesion_1.png
saved /root/autodl-tmp/CBIS/figures/inspect/lesion_2.png
saved /root/autodl-tmp/CBIS/figures/inspect/lesion_3.png
saved /root/autodl-tmp/CBIS/figures/inspect/lesion_4.png
>> inspect(any_number) to see a specific lesion


In [7]:
# ══════════════════════════════════════════════════════════════════════
# CHECK: is the mask ONE piece or MANY? (dots vs connected line)
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","inspect")
S=512; _clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
df=pd.read_csv(os.path.join(D,"cbis_calc_fixed.csv"))

def check_mask(n):
    r=df.iloc[n]
    img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if img is None or mk is None: print("missing"); return
    img=cv2.resize(img,(S,S)); g=_clahe.apply(img)
    m=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
    # how many SEPARATE pieces does the mask have?
    n_comp,lab,st,_=cv2.connectedComponentsWithStats(m,8)
    pieces=n_comp-1
    sizes=sorted([st[i,cv2.CC_STAT_AREA] for i in range(1,n_comp)],reverse=True)
    print("lesion #"+str(n)+":")
    print("  mask has "+str(pieces)+" separate piece(s)")
    print("  piece sizes (px): "+str(sizes[:10]))
    if pieces>1:
        biggest=sizes[0]; total=sum(sizes)
        print("  biggest piece = "+format(100*biggest/total,".0f")+"% of mask")
        if biggest/total>0.85:
            print("  >> ONE main piece + tiny specks. The 'dots' are small satellites,")
            print("     the real lesion is the big piece. Mostly cosmetic.")
        else:
            print("  >> genuinely FRAGMENTED into multiple real pieces.")
    else:
        print("  >> ONE connected piece. Any 'dots' you see are the contour")
        print("     tracing holes/bumps — cosmetic, not real fragmentation.")

    # visual: raw mask filled (not outlined) so you see its TRUE shape
    fig,ax=plt.subplots(1,3,figsize=(15,5))
    ax[0].imshow(g,cmap="gray"); ax[0].set_title("image"); ax[0].axis("off")
    ax[1].imshow(m,cmap="gray"); ax[1].set_title("mask FILLED (true shape) — "+str(pieces)+" piece(s)"); ax[1].axis("off")
    # color each separate piece differently
    color=np.zeros((S,S,3),np.uint8); rng=np.random.RandomState(0)
    for i in range(1,n_comp):
        color[lab==i]=rng.randint(80,255,3)
    ov=cv2.addWeighted(cv2.cvtColor(g,cv2.COLOR_GRAY2RGB),0.6,color,0.4,0)
    ax[2].imshow(ov); ax[2].set_title("each piece = different color"); ax[2].axis("off")
    plt.suptitle("lesion #"+str(n)+"  ("+str(pieces)+" mask pieces)",fontsize=11)
    plt.tight_layout(); p=os.path.join(FIG,"maskcheck_"+str(n)+".png")
    plt.savefig(p,dpi=110,bbox_inches="tight"); plt.close(); print("  saved "+p)

# check a few
for n in [0,1,2,3,4,5]: check_mask(n); print()
print(">> check_mask(N) on any lesion where you saw wrong dots")

lesion #0:
  mask has 544 separate piece(s)
  piece sizes (px): [np.int32(95), np.int32(63), np.int32(48), np.int32(47), np.int32(45), np.int32(43), np.int32(42), np.int32(42), np.int32(41), np.int32(39)]
  biggest piece = 1% of mask
  >> genuinely FRAGMENTED into multiple real pieces.
  saved /root/autodl-tmp/CBIS/figures/inspect/maskcheck_0.png

lesion #1:
  mask has 542 separate piece(s)
  piece sizes (px): [np.int32(82), np.int32(75), np.int32(72), np.int32(67), np.int32(56), np.int32(56), np.int32(55), np.int32(51), np.int32(50), np.int32(49)]
  biggest piece = 1% of mask
  >> genuinely FRAGMENTED into multiple real pieces.
  saved /root/autodl-tmp/CBIS/figures/inspect/maskcheck_1.png

lesion #2:
  mask has 78 separate piece(s)
  piece sizes (px): [np.int32(109), np.int32(100), np.int32(78), np.int32(70), np.int32(49), np.int32(45), np.int32(45), np.int32(43), np.int32(43), np.int32(42)]
  biggest piece = 6% of mask
  >> genuinely FRAGMENTED into multiple real pieces.
  saved /roo

In [8]:
# ══════════════════════════════════════════════════════════════════════
# RESTORE the correct 24% ROI masks (undo the over-tightening)
#   the tightening cell saved backups as *_roi.png — put them back
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2, shutil
D="/root/autodl-tmp/CBIS"
OUT=os.path.join(D,"crops_fixed_calc")
df=pd.read_csv(os.path.join(D,"cbis_calc_fixed.csv"))

restored=0; no_backup=0
for _,r in df.iterrows():
    roi_backup=r["msk"].replace("_msk.png","_roi.png")   # the loose 24% mask backup
    if os.path.exists(roi_backup):
        shutil.copy(roi_backup, r["msk"])                # restore it over the fragmented one
        restored+=1
    else:
        no_backup+=1
print("restored "+str(restored)+" masks from _roi backups | "+str(no_backup)+" had no backup")

# verify: check piece count now
def pieces(mp):
    m=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
    if m is None: return -1,0
    b=(m>127).astype(np.uint8)
    n,_,st,_=cv2.connectedComponentsWithStats(b,8)
    return n-1, b.mean()
pc=[]; cov=[]
for _,r in df.sample(min(80,len(df)),random_state=0).iterrows():
    p,c=pieces(r["msk"])
    if p>=0: pc.append(p); cov.append(c)
print("\nafter restore:")
print("  median mask pieces: "+str(int(np.median(pc)))+"  (was 500+, want 1-5)")
print("  median coverage: "+format(100*np.median(cov),".0f")+"%  (want ~24%, not 2%)")
if np.median(pc)<=10:
    print("  >> FIXED: masks are back to clean connected regions")
else:
    print("  >> backups not found or also fragmented — tell me and I'll regenerate from source")

restored 1866 masks from _roi backups | 0 had no backup

after restore:
  median mask pieces: 1  (was 500+, want 1-5)
  median coverage: 24%  (want ~24%, not 2%)
  >> FIXED: masks are back to clean connected regions


In [9]:
import os, pandas as pd, cv2, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","inspect")
S=512; _clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
df=pd.read_csv(os.path.join(D,"cbis_calc_fixed.csv"))

fig,ax=plt.subplots(4,2,figsize=(9,16))
for row,n in enumerate([0,1,2,3]):
    r=df.iloc[n]
    img=cv2.resize(cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE),(S,S))
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    m=(cv2.resize(mk,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
    g=_clahe.apply(img)
    ov=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)
    cn,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(ov,cn,-1,(0,255,0),2)
    lbl="MALIGNANT" if r["label"]==1 else "benign"
    ax[row,0].imshow(g,cmap="gray"); ax[row,0].set_title("#"+str(n)+" image ("+lbl+")"); ax[row,0].axis("off")
    ax[row,1].imshow(ov); ax[row,1].set_title("mask (one clean region)"); ax[row,1].axis("off")
plt.tight_layout(); p=os.path.join(FIG,"final_check.png")
plt.savefig(p,dpi=110,bbox_inches="tight"); plt.close(); print("saved "+p+" — open it")

saved /root/autodl-tmp/CBIS/figures/inspect/final_check.png — open it


In [10]:
# ══════════════════════════════════════════════════════════════════════
# MAXIMUM ACCURACY from your EXISTING calcification model
#   sweeps every threshold, finds the one that maximizes ACCURACY (not AUC)
#   also reports precision/recall so you see the tradeoff honestly
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import (confusion_matrix, accuracy_score, roc_auc_score,
                             balanced_accuracy_score, f1_score)
D="/root/autodl-tmp/CBIS"

# use whichever OOF file you have (image-only or fixed)
for f in ["cv_calc_imageonly_oof.csv","cv_calc_fixed_oof.csv","cv_calc_radimagenet_oof.csv"]:
    p=os.path.join(D,f)
    if os.path.exists(p): df=pd.read_csv(p); print("using "+f); break

y=df.true.values; prob=df.prob.values
print("class balance: benign "+str((y==0).sum())+"  malignant "+str((y==1).sum())+
      "  ("+format(100*(y==0).mean(),".0f")+"% benign)\n")

best_acc=(0,0.5)
print("  thr    ACCURACY  balanced  sens   spec   AUC")
for t in np.linspace(0.05,0.95,181):
    pred=(prob>t).astype(int)
    acc=accuracy_score(y,pred)
    if acc>best_acc[0]: best_acc=(acc,t)
for t in [0.30,0.40,0.50,best_acc[1],0.70,0.80]:
    pred=(prob>t).astype(int)
    cm=confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
    star=" <- MAX ACCURACY" if abs(t-best_acc[1])<0.01 else ""
    print("  "+format(t,".2f")+"   "+format(accuracy_score(y,pred),".4f")+"    "+
          format(balanced_accuracy_score(y,pred),".3f")+"    "+
          format(tp/max(tp+fn,1),".3f")+"  "+format(tn/max(tn+fp,1),".3f")+"  "+
          format(roc_auc_score(y,prob),".3f")+star)

print("\n  MAXIMUM ACCURACY = "+format(100*best_acc[0],".1f")+"% at threshold "+format(best_acc[1],".2f"))
pred=(prob>best_acc[1]).astype(int); cm=confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
print("  at that point: FP="+str(fp)+" FN="+str(fn)+"  (total errors "+str(fp+fn)+" of "+str(len(y))+")")
print("  sensitivity "+format(tp/max(tp+fn,1),".3f")+"  specificity "+format(tn/max(tn+fp,1),".3f"))

using cv_calc_fixed_oof.csv
class balance: benign 1194  malignant 672  (64% benign)

  thr    ACCURACY  balanced  sens   spec   AUC
  0.30   0.5091    0.615    0.994  0.236  0.788
  0.40   0.5606    0.651    0.976  0.327  0.788
  0.50   0.6259    0.690    0.920  0.461  0.788
  0.64   0.7186    0.651    0.408  0.894  0.788 <- MAX ACCURACY
  0.70   0.6940    0.586    0.199  0.972  0.788
  0.80   0.6565    0.523    0.046  1.000  0.788

  MAXIMUM ACCURACY = 71.9% at threshold 0.64
  at that point: FP=127 FN=398  (total errors 525 of 1866)
  sensitivity 0.408  specificity 0.894


In [11]:
# ══════════════════════════════════════════════════════════════════════
# WHY IS SENSITIVITY LOW? test imbalance strategies + calibration
#   compares: current | heavy malignant oversample | class-weight only
#             | balanced-accuracy threshold | equal-error threshold
#   goal: find operating point + training that catches more cancers
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, balanced_accuracy_score
D="/root/autodl-tmp/CBIS"
df=pd.read_csv(os.path.join(D,"cv_calc_fixed_oof.csv"))
y=df.true.values; prob=df.prob.values

print("The real problem: at ANY threshold, can we get BOTH decent sensitivity AND accuracy?\n")
print("  thr   acc    sens   spec   |  cancers_caught  benign_caught")
print("  "+"-"*62)
# find threshold for several clinical goals
goals={}
# equal error rate (sens == spec)
diffs=[]
for t in np.linspace(0.05,0.95,181):
    pred=(prob>t).astype(int); cm=confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
    sens=tp/max(tp+fn,1); spec=tn/max(tn+fp,1)
    diffs.append((abs(sens-spec),t))
goals["equal error (sens=spec)"]=min(diffs)[1]
goals["max balanced acc"]=max([(balanced_accuracy_score(y,(prob>t).astype(int)),t) for t in np.linspace(0.05,0.95,181)])[1]
goals["sensitivity >= 0.80"]=max([t for t in np.linspace(0.05,0.95,181) if (lambda c: c[3]/max(c[3]+c[2],1))(confusion_matrix(y,(prob>t).astype(int),labels=[0,1]).ravel())>=0.80]+[0.05])
goals["max accuracy"]=max([(accuracy_score(y,(prob>t).astype(int)),t) for t in np.linspace(0.05,0.95,181)])[1]

for name,t in goals.items():
    pred=(prob>t).astype(int); cm=confusion_matrix(y,pred,labels=[0,1]); tn,fp,fn,tp=cm.ravel()
    print("  "+format(t,".2f")+"  "+format(accuracy_score(y,pred),".3f")+"  "+
          format(tp/max(tp+fn,1),".3f")+"  "+format(tn/max(tn+fp,1),".3f")+
          "   "+str(tp)+"/"+str(tp+fn)+"        "+str(tn)+"/"+str(tn+fp)+"   <- "+name)

print("\n  AUC "+format(roc_auc_score(y,prob),".3f")+" is fixed — the threshold only")
print("  slides errors between FP and FN. To catch MORE cancer AND keep accuracy,")
print("  we need a BETTER model (higher AUC), not a different threshold.")
print("\n  KEY QUESTION: is 71.9% accuracy acceptable if it misses 59% of cancers? NO.")
print("  A useful model needs ~0.80+ sensitivity. At that point accuracy is ~"+
      format(100*accuracy_score(y,(prob>goals["sensitivity >= 0.80"]).astype(int)),".0f")+"%.")

The real problem: at ANY threshold, can we get BOTH decent sensitivity AND accuracy?

  thr   acc    sens   spec   |  cancers_caught  benign_caught
  --------------------------------------------------------------
  0.58  0.703  0.690  0.709   464/672        847/1194   <- equal error (sens=spec)
  0.56  0.693  0.781  0.644   525/672        769/1194   <- max balanced acc
  0.55  0.681  0.805  0.611   541/672        729/1194   <- sensitivity >= 0.80
  0.64  0.719  0.408  0.894   274/672        1067/1194   <- max accuracy

  AUC 0.788 is fixed — the threshold only
  slides errors between FP and FN. To catch MORE cancer AND keep accuracy,
  we need a BETTER model (higher AUC), not a different threshold.

  KEY QUESTION: is 71.9% accuracy acceptable if it misses 59% of cancers? NO.
  A useful model needs ~0.80+ sensitivity. At that point accuracy is ~68%.


In [1]:
# ══════════════════════════════════════════════════════════════════════
# THRESHOLD TABLE + honest threshold selection (leave-one-fold-out)
#   threshold chosen WITHOUT seeing the fold it's evaluated on
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
D="/root/autodl-tmp/CBIS"

def table(path, name):
    oof=pd.read_csv(path)
    ycol="true" if "true" in oof.columns else "label"
    y=oof[ycol].astype(int).values; p=oof["prob"].values
    print("\n"+"="*72); print(name+"   (n="+str(len(y))+")"); print("="*72)

    # --- full threshold sweep table (for the thesis) ---
    print("  thr    sens    spec    F1     acc     FP    FN")
    for t in [0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]:
        pr=(p>t).astype(int); tn,fp,fn,tp=confusion_matrix(y,pr,labels=[0,1]).ravel()
        print("  "+format(t,".2f")+"  "+format(tp/max(tp+fn,1),".3f")+"  "+
              format(tn/max(tn+fp,1),".3f")+"  "+format(f1_score(y,pr),".3f")+"  "+
              format(accuracy_score(y,pr),".3f")+"  "+str(fp).rjust(4)+"  "+str(fn).rjust(4))
    print("  AUC (threshold-free): "+format(roc_auc_score(y,p),".4f"))

    # --- honest threshold: chosen on other folds, applied to held-out fold ---
    oof["assessment"]=pd.to_numeric(oof["assessment"],errors="coerce")
    gr=oof.patient_id.values
    strat=oof[ycol].astype(str)+"_"+oof.assessment.isin([3,4]).astype(int).astype(str)
    folds=list(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(oof,strat,gr))
    preds=np.zeros(len(y),dtype=int); chosen=[]
    for tr_i,te_i in folds:
        # pick threshold maximising F1 on the OTHER folds only
        best=max([(f1_score(y[tr_i],(p[tr_i]>t).astype(int)),t) for t in np.linspace(.05,.95,181)])
        thr=best[1]; chosen.append(thr)
        preds[te_i]=(p[te_i]>thr).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,preds,labels=[0,1]).ravel()
    print("\n  HONEST (threshold selected on held-out folds, mean thr "+format(np.mean(chosen),".2f")+"):")
    print("    sens "+format(tp/max(tp+fn,1),".3f")+"  spec "+format(tn/max(tn+fp,1),".3f")+
          "  F1 "+format(f1_score(y,preds),".3f")+"  acc "+format(accuracy_score(y,preds),".3f"))
    print("    TN "+str(tn)+"  FP "+str(fp)+"  FN "+str(fn)+"  TP "+str(tp))

table(os.path.join(D,"cv_mass_fixed_oof.csv"),"MASS")
table(os.path.join(D,"cv_calc_fixed_oof.csv"),"CALCIFICATION")


MASS   (n=1696)
  thr    sens    spec    F1     acc     FP    FN
  0.35  0.973  0.205  0.672  0.560   725    21
  0.40  0.935  0.389  0.707  0.642   557    51
  0.45  0.861  0.566  0.728  0.702   396   109
  0.50  0.768  0.752  0.747  0.759   226   182
  0.55  0.639  0.860  0.709  0.758   128   283
  0.60  0.469  0.938  0.609  0.721    57   416
  0.65  0.347  0.967  0.501  0.680    30   512
  0.70  0.231  0.981  0.369  0.634    17   603
  AUC (threshold-free): 0.8308

  HONEST (threshold selected on held-out folds, mean thr 0.50):
    sens 0.767  spec 0.737  F1 0.740  acc 0.751
    TN 672  FP 240  FN 183  TP 601

CALCIFICATION   (n=1866)
  thr    sens    spec    F1     acc     FP    FN
  0.35  0.993  0.281  0.607  0.537   859     5
  0.40  0.976  0.327  0.615  0.561   804    16
  0.45  0.954  0.385  0.626  0.590   734    31
  0.50  0.920  0.461  0.639  0.626   644    54
  0.55  0.820  0.591  0.644  0.674   488   121
  0.60  0.603  0.772  0.600  0.711   272   267
  0.65  0.354  0.923  

In [2]:
# ══════════════════════════════════════════════════════════════════════
# SELECTIVE PREDICTION — model defers uncertain cases to radiologist
#   labels UNCHANGED. we only vary how much the model refuses to answer.
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
D="/root/autodl-tmp/CBIS"

def selective(path, name, thr):
    oof=pd.read_csv(path)
    ycol="true" if "true" in oof.columns else "label"
    y=oof[ycol].astype(int).values; p=oof["prob"].values
    oof["assessment"]=pd.to_numeric(oof["assessment"],errors="coerce")
    conf=np.abs(p-thr)                      # distance from decision boundary = confidence
    order=np.argsort(-conf)                 # most confident first
    print("\n"+"="*70); print(name+"  (decision threshold "+format(thr,".2f")+")"); print("="*70)
    print("  coverage  n_predicted  accuracy  sens   spec   deferred  %B4_deferred")
    for cov in [1.0,0.9,0.8,0.7,0.6,0.5,0.4,0.3]:
        k=int(len(y)*cov); idx=order[:k]; rest=order[k:]
        yy=y[idx]; pp=(p[idx]>thr).astype(int)
        tn,fp,fn,tp=confusion_matrix(yy,pp,labels=[0,1]).ravel()
        b4=(oof.assessment.values[rest]==4).mean()*100 if len(rest)>0 else 0
        print("   "+format(100*cov,".0f").rjust(4)+"%      "+str(k).rjust(5)+"      "+
              format(100*accuracy_score(yy,pp),".1f").rjust(5)+"%   "+
              format(tp/max(tp+fn,1),".2f")+"   "+format(tn/max(tn+fp,1),".2f")+"    "+
              str(len(rest)).rjust(5)+"      "+format(b4,".0f")+"%")
    print("  (%B4_deferred = what fraction of the deferred cases are BI-RADS 4)")

selective(os.path.join(D,"cv_mass_fixed_oof.csv"),"MASS",0.50)
selective(os.path.join(D,"cv_calc_fixed_oof.csv"),"CALCIFICATION",0.56)


MASS  (decision threshold 0.50)
  coverage  n_predicted  accuracy  sens   spec   deferred  %B4_deferred
    100%       1696       75.9%   0.77   0.75        0      0%
     90%       1526       78.4%   0.80   0.77      170      45%
     80%       1356       80.5%   0.82   0.79      340      47%
     70%       1187       82.6%   0.83   0.82      509      50%
     60%       1017       84.0%   0.85   0.83      679      50%
     50%        848       86.4%   0.87   0.86      848      48%
     40%        678       88.8%   0.90   0.87     1018      48%
     30%        508       90.0%   0.93   0.86     1188      46%
  (%B4_deferred = what fraction of the deferred cases are BI-RADS 4)

CALCIFICATION  (decision threshold 0.56)
  coverage  n_predicted  accuracy  sens   spec   deferred  %B4_deferred
    100%       1866       68.6%   0.79   0.63        0      0%
     90%       1679       70.5%   0.82   0.64      187      80%
     80%       1492       72.3%   0.84   0.66      374      77%
     70%  

In [3]:
import os, numpy as np, pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
D="/root/autodl-tmp/CBIS"

GOAL = "sens>=0.85"   # options: "sens>=0.85", "sens>=0.90", "balanced", "max_f1", "max_acc"

oof=pd.read_csv(os.path.join(D,"cv_mass_fixed_oof.csv"))
ycol="true" if "true" in oof.columns else "label"
y=oof[ycol].astype(int).values; p=oof["prob"].values
oof["assessment"]=pd.to_numeric(oof["assessment"],errors="coerce")
gr=oof.patient_id.values
strat=oof[ycol].astype(str)+"_"+oof.assessment.isin([3,4]).astype(int).astype(str)

def pick(yy,pp,goal):
    ts=np.linspace(.05,.95,181)
    def sens(t):
        c=confusion_matrix(yy,(pp>t).astype(int),labels=[0,1]).ravel(); return c[3]/max(c[3]+c[2],1)
    if goal.startswith("sens>="):
        target=float(goal.split(">=")[1]); ok=[t for t in ts if sens(t)>=target]
        return max(ok) if ok else ts[0]
    if goal=="balanced": return max([(balanced_accuracy_score(yy,(pp>t).astype(int)),t) for t in ts])[1]
    if goal=="max_f1":   return max([(f1_score(yy,(pp>t).astype(int)),t) for t in ts])[1]
    return max([(accuracy_score(yy,(pp>t).astype(int)),t) for t in ts])[1]

preds=np.zeros(len(y),int); chosen=[]
for tr_i,te_i in StratifiedGroupKFold(5,shuffle=True,random_state=42).split(oof,strat,gr):
    thr=pick(y[tr_i],p[tr_i],GOAL); chosen.append(thr)
    preds[te_i]=(p[te_i]>thr).astype(int)

tn,fp,fn,tp=confusion_matrix(y,preds,labels=[0,1]).ravel()
print("MASS  goal="+GOAL+"  mean threshold "+format(np.mean(chosen),".2f"))
print("  AUC (unchanged, threshold-free): "+format(roc_auc_score(y,p),".4f"))
print("  sens "+format(tp/max(tp+fn,1),".3f")+"  spec "+format(tn/max(tn+fp,1),".3f")+
      "  F1 "+format(f1_score(y,preds),".3f")+"  acc "+format(accuracy_score(y,preds),".3f"))
print("  TN "+str(tn)+"  FP "+str(fp)+"  FN "+str(fn)+"  TP "+str(tp))
print("  (baseline at balanced threshold: sens 0.767 spec 0.737 acc 0.751, FN 183)")

MASS  goal=sens>=0.85  mean threshold 0.46
  AUC (unchanged, threshold-free): 0.8308
  sens 0.852  spec 0.598  F1 0.734  acc 0.715
  TN 545  FP 367  FN 116  TP 668
  (baseline at balanced threshold: sens 0.767 spec 0.737 acc 0.751, FN 183)


In [4]:
# ══════════════════════════════════════════════════════════════════════
# CONFIDENCE INTERVALS + DeLong TEST — no training, seconds to run
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from scipy import stats
D="/root/autodl-tmp/CBIS"

def load(f):
    p=os.path.join(D,f)
    if not os.path.exists(p): return None,None
    o=pd.read_csv(p); yc="true" if "true" in o.columns else "label"
    return o[yc].astype(int).values, o["prob"].values

def boot_ci(y,p,n=2000,seed=0):
    rs=np.random.RandomState(seed); aucs=[]
    idx=np.arange(len(y))
    for _ in range(n):
        s=rs.choice(idx,len(idx),replace=True)
        if len(set(y[s]))<2: continue
        aucs.append(roc_auc_score(y[s],p[s]))
    return np.percentile(aucs,2.5), np.percentile(aucs,97.5)

# --- DeLong (Sun & Xu fast implementation) ---
def _midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N,float); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N,float); T2[J]=T; return T2

def delong(y,p1,p2):
    pos=p1[y==1]; neg=p1[y==0]; pos2=p2[y==1]; neg2=p2[y==0]
    m,n=len(pos),len(neg); preds=np.vstack([np.r_[pos,neg],np.r_[pos2,neg2]])
    k=2; tx=np.empty([k,m]); ty=np.empty([k,n]); tz=np.empty([k,m+n])
    for r in range(k):
        tx[r]=_midrank(preds[r,:m]); ty[r]=_midrank(preds[r,m:]); tz[r]=_midrank(preds[r,:])
    auc=tz[:,:m].sum(axis=1)/m/n - (m+1.)/2./n
    v01=(tz[:,:m]-tx)/n; v10=1.-(tz[:,m:]-ty)/m
    S=np.cov(v01)/m + np.cov(v10)/n
    S=np.atleast_2d(S); l=np.array([[1,-1]])
    var=l.dot(S).dot(l.T)[0,0]
    if var<=0: return auc[0],auc[1],1.0
    z=(auc[0]-auc[1])/np.sqrt(var)
    return auc[0],auc[1],2*(1-stats.norm.cdf(abs(z)))

print("="*66); print("95% CONFIDENCE INTERVALS (2000 bootstrap resamples)"); print("="*66)
for name,f in [("MASS guided","cv_mass_fixed_oof.csv"),
               ("MASS dual-pathway","cv_mass_dualpath_oof.csv"),
               ("CALC guided","cv_calc_fixed_oof.csv"),
               ("CALC image-only","cv_calc_imageonly_oof.csv"),
               ("CALC hard-focused","cv_calc_hardfocus_oof.csv")]:
    y,p=load(f)
    if y is None: print("  "+name.ljust(22)+"(file not found)"); continue
    lo,hi=boot_ci(y,p)
    print("  "+name.ljust(22)+"AUC "+format(roc_auc_score(y,p),".4f")+
          "   95% CI ["+format(lo,".4f")+", "+format(hi,".4f")+"]")

print("\n"+"="*66); print("DeLong TEST — is the difference real?"); print("="*66)
pairs=[("CALC guided vs image-only","cv_calc_fixed_oof.csv","cv_calc_imageonly_oof.csv"),
       ("CALC guided vs hard-focused","cv_calc_fixed_oof.csv","cv_calc_hardfocus_oof.csv"),
       ("MASS guided vs dual-pathway","cv_mass_fixed_oof.csv","cv_mass_dualpath_oof.csv")]
for label,f1,f2 in pairs:
    y1,p1=load(f1); y2,p2=load(f2)
    if y1 is None or y2 is None: print("  "+label.ljust(30)+"(file missing)"); continue
    if len(y1)!=len(y2) or not np.array_equal(y1,y2):
        print("  "+label.ljust(30)+"(different case sets - cannot compare)"); continue
    a1,a2,pv=delong(y1,p1,p2)
    verdict="SIGNIFICANT" if pv<0.05 else "not significant"
    print("  "+label.ljust(30)+format(a1,".4f")+" vs "+format(a2,".4f")+
          "   p="+format(pv,".4f")+"   "+verdict)

95% CONFIDENCE INTERVALS (2000 bootstrap resamples)
  MASS guided           AUC 0.8308   95% CI [0.8104, 0.8502]
  MASS dual-pathway     AUC 0.7882   95% CI [0.7666, 0.8098]
  CALC guided           AUC 0.7882   95% CI [0.7688, 0.8084]
  CALC image-only       AUC 0.7807   95% CI [0.7612, 0.8005]
  CALC hard-focused     AUC 0.7799   95% CI [0.7600, 0.8005]

DeLong TEST — is the difference real?
  CALC guided vs image-only     0.7882 vs 0.7807   p=0.2719   not significant
  CALC guided vs hard-focused   0.7882 vs 0.7799   p=0.3454   not significant
  MASS guided vs dual-pathway   0.8308 vs 0.7882   p=0.0000   SIGNIFICANT


In [5]:
# ══════════════════════════════════════════════════════════════════════
# MASS IMAGE-ONLY BASELINE — plain DenseNet-121, no guidance, no aux heads
#   identical folds/seed/augmentation/optimizer to the guided run
#   the ONLY difference is your proposed modules are removed
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"]="4"; os.environ["HF_HUB_OFFLINE"]="1"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")
S=512; BATCH=12; MULT=8; EPOCHS=20; LR_HEAD=1e-3; LR_FT=1e-5; FREEZE=3; GAMMA=2.0; NFOLD=5
torch.backends.cudnn.benchmark=True; torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
MEAN=np.array([0.485,0.456,0.406],np.float32); STD=np.array([0.229,0.224,0.225],np.float32)

sub=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); sub["label"]=sub["label"].astype(int)
sub["assessment"]=pd.to_numeric(sub["assessment"],errors="coerce")
sub=sub.dropna(subset=["img","label"]).reset_index(drop=True)
print("MASS baseline (no guidance, no aux heads): n="+str(len(sub))+"\n")

CACHE={}
for _,r in sub.iterrows():
    k=r["img"]
    if k in CACHE: continue
    img=cv2.imread(k,cv2.IMREAD_GRAYSCALE)
    CACHE[k]=_clahe.apply(cv2.resize(img if img is not None else np.zeros((S,S),np.uint8),(S,S)))
print("cached "+str(len(CACHE)))

class DS(Dataset):
    def __init__(s,idx,aug,mult=1,tta=0):
        s.idx=np.array(idx); s.aug=aug; s.mult=mult if aug else 1; s.tta=tta
    def __len__(s): return len(s.idx)*s.mult
    def __getitem__(s,i):
        j=s.idx[i%len(s.idx)]; k=i//len(s.idx); r=sub.iloc[j]
        img=CACHE[r["img"]].copy()
        if s.aug and k>0:
            if   k%8==1: img=np.fliplr(img)
            elif k%8==2: img=np.flipud(img)
            elif k%8==3: img=np.rot90(img,1)
            elif k%8==4: img=np.rot90(img,2)
            elif k%8==5: img=np.rot90(img,3)
            elif k%8==6:
                M=cv2.getRotationMatrix2D((S/2,S/2),np.random.uniform(-20,20),np.random.uniform(.9,1.1))
                img=cv2.warpAffine(img,M,(S,S),borderMode=cv2.BORDER_REFLECT)
            elif k%8==7: img=np.clip(img.astype(np.float32)*np.random.uniform(.85,1.15),0,255).astype(np.uint8)
        if s.tta==1: img=np.fliplr(img)
        elif s.tta==2: img=np.flipud(img)
        elif s.tta==3: img=np.rot90(img,2)
        im=np.ascontiguousarray(img).astype(np.float32)/255.
        x=np.stack([im,im,im],0); x=((x.transpose(1,2,0)-MEAN)/STD).transpose(2,0,1).astype(np.float32)
        return torch.from_numpy(np.ascontiguousarray(x)), torch.tensor(int(r["label"]))

class Plain(nn.Module):
    def __init__(s):
        super().__init__()
        try: dn=models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn=models.densenet121(weights=None)
        s.b=dn.features; s.pool=nn.AdaptiveAvgPool2d(1)
        s.head=nn.Sequential(nn.Linear(1024,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,2))
    def forward(s,x):
        return s.head(s.pool(F.relu(s.b(x))).flatten(1))

y=sub.label.values; gr=sub.patient_id.values
strat=sub["label"].astype(str)+"_"+sub.assessment.isin([3,4]).astype(int).astype(str)
oofp=np.zeros(len(sub)); fa=[]
for fold,(tri,tei) in enumerate(StratifiedGroupKFold(NFOLD,shuffle=True,random_state=42).split(sub,strat,gr),1):
    t0=time.time()
    g2=gr[tri]; uq=np.array(sorted(set(g2))); rs=np.random.RandomState(fold)
    vg=set(rs.permutation(uq)[:max(1,int(0.12*len(uq)))]); vm=np.array([g in vg for g in g2])
    tr_i=tri[~vm]; va_i=tri[vm]; assert len(set(gr[tr_i])&set(gr[tei]))==0
    n0=float((y[tr_i]==0).sum()); n1=float((y[tr_i]==1).sum()); al=torch.tensor([n1/(n0+n1),n0/(n0+n1)],device=DEV)
    def focal(lo,t):
        ce=F.cross_entropy(lo.float(),t,weight=al,reduction="none"); pt=torch.exp(-ce); return ((1-pt)**GAMMA*ce).mean()
    torch.manual_seed(fold); np.random.seed(fold)
    net=Plain().to(DEV).to(memory_format=torch.channels_last)
    for p_ in net.b.parameters(): p_.requires_grad=False
    sc=torch.amp.GradScaler(); opt=torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],lr=LR_HEAD,weight_decay=1e-3)
    tl=DataLoader(DS(tr_i,True,MULT),batch_size=BATCH,shuffle=True,num_workers=0,pin_memory=True)
    @torch.no_grad()
    def col(idx,tta=True):
        net.eval(); reps=[0,1,2,3] if tta else [0]; tot=None; lb=None
        for t in reps:
            ld=DataLoader(DS(idx,False,tta=t),batch_size=20,shuffle=False,num_workers=0,pin_memory=True); ps=[];ll=[]
            for x,t2 in ld:
                x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last)
                with torch.amp.autocast(device_type="cuda"): o=net(x)
                ps+=list(torch.softmax(o.float(),1)[:,1].cpu().numpy()); ll+=list(t2.numpy())
            ps=np.array(ps); lb=np.array(ll); tot=ps if tot is None else tot+ps
        return lb,tot/len(reps)
    best=0.;bs=None;ni=0
    for ep in range(1,EPOCHS+1):
        if ep==FREEZE+1:
            for p_ in net.b.parameters(): p_.requires_grad=True
            opt=torch.optim.AdamW(net.parameters(),lr=LR_FT,weight_decay=1e-3)
        net.train()
        if ep<=FREEZE: net.b.eval()
        for x,t2 in tl:
            x=x.to(DEV,non_blocking=True).to(memory_format=torch.channels_last); t2=t2.to(DEV,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"): o=net(x); loss=focal(o,t2)
            if not torch.isfinite(loss): continue
            sc.scale(loss).backward(); sc.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(),5.0); sc.step(opt); sc.update()
        yv,pv=col(va_i,tta=False); au=roc_auc_score(yv,pv) if len(set(yv))>1 else 0
        if au>best: best=au; bs={k:v.cpu().clone() for k,v in net.state_dict().items()}; ni=0
        else: ni+=1
        if ni>=5: break
    net.load_state_dict({k:v.to(DEV) for k,v in bs.items()})
    yt,pt=col(tei); oofp[tei]=pt; fa.append(roc_auc_score(yt,pt))
    print("  fold "+str(fold)+"  AUC "+format(fa[-1],".4f")+"  ("+format(time.time()-t0,".0f")+"s)")

out=sub.copy(); out["prob"]=oofp; out["true"]=y
out.to_csv(os.path.join(D,"cv_mass_imageonly_oof.csv"),index=False)
print("\n"+"="*58)
print("MASS IMAGE-ONLY BASELINE")
print("="*58)
print("  MEAN AUC "+format(np.mean(fa),".4f")+" +/- "+format(np.std(fa),".4f")+
      "   POOLED "+format(roc_auc_score(y,oofp),".4f"))
print("  your GUIDED model: 0.8308")
print("  difference: "+format(roc_auc_score(y,oofp)-0.8308,"+.4f")+"  (negative = guided is better)")
print("  saved cv_mass_imageonly_oof.csv -> rerun the DeLong cell to test significance")
print("="*58)

MASS baseline (no guidance, no aux heads): n=1696

cached 1696
  fold 1  AUC 0.7751  (1648s)
  fold 2  AUC 0.6810  (510s)
  fold 3  AUC 0.8036  (1812s)
  fold 4  AUC 0.8408  (1798s)
  fold 5  AUC 0.7261  (447s)

MASS IMAGE-ONLY BASELINE
  MEAN AUC 0.7653 +/- 0.0564   POOLED 0.7603
  your GUIDED model: 0.8308
  difference: -0.0705  (negative = guided is better)
  saved cv_mass_imageonly_oof.csv -> rerun the DeLong cell to test significance


In [1]:
# ══════════════════════════════════════════════════════════════════════
# FILE AUDIT — what format are we actually reading?
# ══════════════════════════════════════════════════════════════════════
import os, glob, numpy as np, cv2, pandas as pd
D="/root/autodl-tmp/CBIS"

print("="*70); print("1. WHAT IMAGE FORMATS EXIST ON DISK"); print("="*70)
for pat,label in [("**/*.dcm","DICOM (16-bit, original)"),
                  ("**/*.jpg","JPEG (8-bit)"),
                  ("**/*.jpeg","JPEG (8-bit)"),
                  ("**/*.png","PNG")]:
    n=len(glob.glob(os.path.join(D,pat),recursive=True))
    print("  "+label.ljust(28)+str(n)+" files")

print("\n"+"="*70); print("2. BIT DEPTH OF SOURCE IMAGES"); print("="*70)
src=glob.glob(os.path.join(D,"jpeg","**","*.jpg"),recursive=True)[:5]
for f in src:
    im=cv2.imread(f,cv2.IMREAD_UNCHANGED)
    if im is None: continue
    print("  "+os.path.basename(f)[:40].ljust(42)+"dtype "+str(im.dtype)+
          "  range "+str(int(im.min()))+"-"+str(int(im.max()))+
          "  unique levels "+str(len(np.unique(im))))

print("\n"+"="*70); print("3. YOUR CROPS — how many intensity levels survive?"); print("="*70)
for name,csv in [("mass","cbis_mass_fixed.csv"),("calc","cbis_calc_fixed.csv")]:
    d=pd.read_csv(os.path.join(D,csv))
    lv=[]; rng=[]
    for _,r in d.sample(min(60,len(d)),random_state=0).iterrows():
        im=cv2.imread(r["img"],cv2.IMREAD_UNCHANGED)
        if im is None: continue
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        if mk is not None and (mk>127).sum()>0:
            inside=im[(cv2.resize(mk,im.shape[::-1],interpolation=cv2.INTER_NEAREST)>127)]
            lv.append(len(np.unique(inside))); rng.append(int(inside.max())-int(inside.min()))
    print("  "+name.ljust(6)+"dtype "+str(im.dtype)+
          "   distinct levels INSIDE lesion: median "+str(int(np.median(lv)))+
          "   intensity range: median "+str(int(np.median(rng))))
print("  (8-bit gives max 256 levels. 16-bit DICOM would give up to 65536.)")

print("\n"+"="*70); print("4. CSV INTEGRITY"); print("="*70)
for f in ["cbis_mass_fixed.csv","cbis_calc_fixed.csv"]:
    d=pd.read_csv(os.path.join(D,f))
    miss_img=sum(0 if os.path.exists(p) else 1 for p in d["img"])
    miss_msk=sum(0 if os.path.exists(p) else 1 for p in d["msk"])
    print("  "+f)
    print("     rows "+str(len(d))+"   patients "+str(d.patient_id.nunique())+
          "   label 0/1 "+str((d.label==0).sum())+"/"+str((d.label==1).sum()))
    print("     missing image files: "+str(miss_img)+"   missing mask files: "+str(miss_msk))
    print("     duplicate rows: "+str(d.duplicated(subset=["img"]).sum()))

print("\n"+"="*70); print("5. IS THE ORIGINAL DICOM AVAILABLE?"); print("="*70)
dcm=glob.glob(os.path.join(D,"**","*.dcm"),recursive=True)
if dcm:
    print("  YES — "+str(len(dcm))+" DICOM files found.")
    try:
        import pydicom
        ds=pydicom.dcmread(dcm[0])
        a=ds.pixel_array
        print("  sample: dtype "+str(a.dtype)+"  range "+str(int(a.min()))+"-"+str(int(a.max()))+
              "  BitsStored "+str(getattr(ds,'BitsStored','?')))
        print("  >> regenerating crops from DICOM would preserve full bit depth")
    except ImportError:
        print("  (pip install pydicom --break-system-packages to inspect)")
else:
    print("  NO DICOM on disk — you have only the 8-bit JPEG mirror.")
    print("  >> the original 16-bit data is on TCIA; re-downloading is ~160GB")

1. WHAT IMAGE FORMATS EXIST ON DISK
  DICOM (16-bit, original)    0 files
  JPEG (8-bit)                10237 files
  JPEG (8-bit)                0 files
  PNG                         106137 files

2. BIT DEPTH OF SOURCE IMAGES
  1-263.jpg                                 dtype uint8  range 0-255  unique levels 230
  2-241.jpg                                 dtype uint8  range 0-255  unique levels 15
  1-126.jpg                                 dtype uint8  range 0-255  unique levels 256
  1-231.jpg                                 dtype uint8  range 0-255  unique levels 256
  1-111.jpg                                 dtype uint8  range 0-255  unique levels 256

3. YOUR CROPS — how many intensity levels survive?
  mass  dtype uint8   distinct levels INSIDE lesion: median 72   intensity range: median 72
  calc  dtype uint8   distinct levels INSIDE lesion: median 80   intensity range: median 85
  (8-bit gives max 256 levels. 16-bit DICOM would give up to 65536.)

4. CSV INTEGRITY
  cbis_mas

In [2]:
import os, glob, pandas as pd
D="/root/autodl-tmp/CBIS"
print("CONTENTS OF CBIS/csv/:")
for f in sorted(os.listdir(os.path.join(D,"csv"))):
    p=os.path.join(D,"csv",f)
    print("  "+f.ljust(46)+format(os.path.getsize(p)/1e6,".1f")+" MB")

di=os.path.join(D,"csv","dicom_info.csv")
if os.path.exists(di):
    d=pd.read_csv(di)
    print("\ndicom_info.csv: "+str(len(d))+" rows")
    print("columns:", list(d.columns)[:20])
    for c in ["BitsStored","BitsAllocated","Rows","Columns","SeriesDescription","file_path"]:
        if c in d.columns:
            print("  "+c+": "+str(d[c].dropna().unique()[:6]))
print("\nany .dcm anywhere:", len(glob.glob(os.path.join(D,"**","*.dcm"),recursive=True)))

CONTENTS OF CBIS/csv/:
  calc_case_description_test_set.csv            0.2 MB
  calc_case_description_train_set.csv           0.9 MB
  dicom_info.csv                                6.4 MB
  mass_case_description_test_set.csv            0.2 MB
  mass_case_description_train_set.csv           0.8 MB
  meta.csv                                      1.2 MB

dicom_info.csv: 10237 rows
columns: ['file_path', 'image_path', 'AccessionNumber', 'BitsAllocated', 'BitsStored', 'BodyPartExamined', 'Columns', 'ContentDate', 'ContentTime', 'ConversionType', 'HighBit', 'InstanceNumber', 'LargestImagePixelValue', 'Laterality', 'Modality', 'PatientBirthDate', 'PatientID', 'PatientName', 'PatientOrientation', 'PatientSex']
  BitsStored: [16  8]
  BitsAllocated: [16  8]
  Rows: [ 289 6256 4126   97 4560 3931]
  Columns: [ 351 3526 1546   97 3104 1981]
  SeriesDescription: ['cropped images' 'full mammogram images' 'ROI mask images']
  file_path: ['CBIS-DDSM/dicom/1.3.6.1.4.1.9590.100.1.2.12930872681285196400

In [3]:
import os, pandas as pd
D="/root/autodl-tmp/CBIS"
d=pd.read_csv(os.path.join(D,"csv","dicom_info.csv"))
print("bit depth by image type:")
print(pd.crosstab(d["SeriesDescription"], d["BitsStored"]))
full=d[d.SeriesDescription=="full mammogram images"]
n16=(full.BitsStored==16).sum()
print("\nfull mammograms (source for your crops): "+str(len(full)))
print("  originally 16-bit: "+str(n16)+"  ("+format(100*n16/max(len(full),1),".0f")+"%)")
if "LargestImagePixelValue" in d.columns:
    v=pd.to_numeric(full["LargestImagePixelValue"],errors="coerce").dropna()
    if len(v): print("  original max pixel value: median "+format(v.median(),".0f")+"  (8-bit JPEG caps at 255)")

bit depth by image type:
BitsStored               8     16
SeriesDescription                
ROI mask images        3247     0
cropped images            0  3567
full mammogram images     0  2857

full mammograms (source for your crops): 2857
  originally 16-bit: 2857  (100%)
  original max pixel value: median 65535  (8-bit JPEG caps at 255)


In [1]:
# ══════════════════════════════════════════════════════════════════════
# FIND THE CORRECT FORWARD PASS — tries wiring variants, reports Dice
# ══════════════════════════════════════════════════════════════════════
import os, itertools, torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda")
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

def blk(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class Att(nn.Module):
    def __init__(s,Fg,Fx,Fi):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(Fg,Fi,1),nn.BatchNorm2d(Fi))
        s.Wx=nn.Sequential(nn.Conv2d(Fx,Fi,1),nn.BatchNorm2d(Fi))
        s.psi=nn.Sequential(nn.Conv2d(Fi,1,1),nn.BatchNorm2d(1))
    def forward(s,g,x): return x*torch.sigmoid(s.psi(F.relu(s.Wg(g)+s.Wx(x))))

class U(nn.Module):
    def __init__(s,cat_skip_first=True,att_swap=False,duel=True):
        super().__init__()
        s.cf=cat_skip_first; s.sw=att_swap; s.duel=duel
        s.e1=blk(1,32); s.e2=blk(32,64); s.e3=blk(64,128); s.e4=blk(128,256); s.bn=blk(256,512)
        s.u4=nn.ConvTranspose2d(512,256,2,2); s.a4=Att(256,256,128); s.d4=blk(512,256)
        s.u3=nn.ConvTranspose2d(256,128,2,2); s.a3=Att(128,128,64);  s.d3=blk(256,128)
        s.u2=nn.ConvTranspose2d(128,64,2,2);  s.a2=Att(64,64,32);    s.d2=blk(128,64)
        s.u1=nn.ConvTranspose2d(64,32,2,2);   s.a1=Att(32,32,16);    s.d1=blk(64,32)
        s.v=nn.Conv2d(32,1,1); s.adv=nn.Conv2d(32,2,1)
    def _m(s,att,d):
        return torch.cat([att,d],1) if s.cf else torch.cat([d,att],1)
    def forward(s,x):
        p=F.max_pool2d
        c1=s.e1(x); c2=s.e2(p(c1,2)); c3=s.e3(p(c2,2)); c4=s.e4(p(c3,2)); b=s.bn(p(c4,2))
        A=lambda g,c,gate: gate(c,g) if s.sw else gate(g,c)
        d=s.u4(b); d=s.d4(s._m(A(d,c4,s.a4),d))
        d=s.u3(d); d=s.d3(s._m(A(d,c3,s.a3),d))
        d=s.u2(d); d=s.d2(s._m(A(d,c2,s.a2),d))
        d=s.u1(d); d=s.d1(s._m(A(d,c1,s.a1),d))
        a=s.adv(d)
        return (s.v(d)+a-a.mean(1,keepdim=True)) if s.duel else a

sub=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")).head(16)
sd=torch.load(os.path.join(D,"seg_cbis_mass.pth"),map_location="cpu")
if isinstance(sd,dict) and "state_dict" in sd: sd=sd["state_dict"]

def dice_at(size, cat_first, swap, duel, norm):
    ims=[];gts=[]
    for _,r in sub.iterrows():
        im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); im=cv2.resize(im,(size,size))
        g=_clahe.apply(im) if norm.startswith("clahe") else im
        v=g.astype(np.float32)/255. if norm.endswith("255") else (g.astype(np.float32)-g.mean())/(g.std()+1e-6)
        ims.append(v)
        mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        gts.append((cv2.resize(mk,(size,size),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8))
    net=U(cat_first,swap,duel).to(DEV)
    mi,un=net.load_state_dict(sd,strict=False)
    if len(mi)>0: return None,len(mi)
    net.eval()
    x=torch.from_numpy(np.stack(ims)[:,None]).float().to(DEV)
    with torch.no_grad(): lo=net(x).float()
    best=0; bestinfo=""
    for ch,tag in [(1,"softmax_ch1"),(0,"softmax_ch0")]:
        pr=torch.softmax(lo,1)[:,ch].cpu().numpy()
        d=[]
        for i in range(len(pr)):
            m=(pr[i]>0.5).astype(np.uint8); t=m.sum()+gts[i].sum()
            d.append(2*(m&gts[i]).sum()/t if t>0 else 0)
        if np.mean(d)>best: best=np.mean(d); bestinfo=tag+" cov "+format(100*np.mean([(pr[i]>0.5).mean() for i in range(len(pr))]),".0f")+"%"
    return (best,bestinfo),0

print("searching wiring variants (GT coverage ~23%, target Dice ~0.92)")
print("="*84)
print(f"{'size':<7}{'cat order':<16}{'att args':<12}{'head':<9}{'norm':<14}{'Dice':<8}{'detail'}")
print("-"*84)
rows=[]
for size in [512,256]:
    for cat_first in [True,False]:
        for swap in [False,True]:
            for duel in [True,False]:
                for norm in ["clahe/255","raw/255","clahe/z","raw/z"]:
                    r,nmiss=dice_at(size,cat_first,swap,duel,norm)
                    if r is None: continue
                    d,info=r
                    rows.append((d,size,cat_first,swap,duel,norm,info))
rows.sort(reverse=True)
for d,size,cf,sw,du,nm,info in rows[:12]:
    print(f"{size:<7}{'skip,dec' if cf else 'dec,skip':<16}{'(c,g)' if sw else '(g,c)':<12}"
          f"{'duel' if du else 'adv':<9}{nm:<14}{format(d,'.4f'):<8}{info}")
print("="*84)
if rows and rows[0][0]>0.7:
    print("FOUND: use size="+str(rows[0][1])+", cat="+("skip,dec" if rows[0][2] else "dec,skip")+
          ", att="+("(c,g)" if rows[0][3] else "(g,c)")+", head="+("duel" if rows[0][4] else "adv")+
          ", norm="+rows[0][5])
else:
    print("No variant reached Dice>0.7 — paste your original segmentation model class definition")
    print("(the `class ...(nn.Module)` with e1/d1/v/adv) and I'll match it exactly.")

searching wiring variants (GT coverage ~23%, target Dice ~0.92)
size   cat order       att args    head     norm          Dice    detail
------------------------------------------------------------------------------------
256    dec,skip        (g,c)       duel     clahe/255     0.7819  softmax_ch1 cov 35%
256    dec,skip        (g,c)       adv      clahe/255     0.7819  softmax_ch1 cov 35%
256    skip,dec        (c,g)       duel     raw/255       0.7667  softmax_ch1 cov 20%
256    skip,dec        (c,g)       adv      raw/255       0.7667  softmax_ch1 cov 20%
256    dec,skip        (c,g)       duel     clahe/255     0.7649  softmax_ch1 cov 36%
256    dec,skip        (c,g)       adv      clahe/255     0.7649  softmax_ch1 cov 36%
256    dec,skip        (c,g)       duel     clahe/z       0.7466  softmax_ch1 cov 26%
256    dec,skip        (c,g)       adv      clahe/z       0.7466  softmax_ch1 cov 26%
256    dec,skip        (g,c)       duel     raw/z         0.7389  softmax_ch1 cov 31%
256 

In [2]:
# ══════════════════════════════════════════════════════════════════════
# Which crop folders exist, and which does the segmentation model fit?
# ══════════════════════════════════════════════════════════════════════
import os, glob, torch, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda"); SZ=256
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

print("CROP FOLDERS ON DISK:")
for d in sorted(glob.glob(os.path.join(D,"crops*"))):
    n=len(glob.glob(os.path.join(d,"*_img.png")))
    print("  "+os.path.basename(d).ljust(28)+str(n)+" images")
print("\nCSV FILES:")
for f in sorted(glob.glob(os.path.join(D,"*.csv"))):
    print("  "+os.path.basename(f))

# test the seg model on each candidate crop folder
net=U(cat_skip_first=False, att_swap=False, duel=True).to(DEV)
sd=torch.load(os.path.join(D,"seg_cbis_mass.pth"),map_location="cpu")
if isinstance(sd,dict) and "state_dict" in sd: sd=sd["state_dict"]
net.load_state_dict(sd,strict=True); net.eval()

def test_folder(folder, n=60):
    imgs=sorted(glob.glob(os.path.join(folder,"*_img.png")))[:n]
    if not imgs: return None
    ds=[]
    for p in imgs:
        mp=p.replace("_img.png","_msk.png")
        if not os.path.exists(mp): continue
        im=cv2.imread(p,cv2.IMREAD_GRAYSCALE); mk=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
        if im is None or mk is None: continue
        x0=_clahe.apply(cv2.resize(im,(SZ,SZ))).astype(np.float32)/255.
        gt=(cv2.resize(mk,(SZ,SZ),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        with torch.no_grad():
            t=torch.from_numpy(x0[None,None]).float().to(DEV)
            pr=torch.softmax(net(t).float(),1)[0,1].cpu().numpy()
        m=(pr>0.5).astype(np.uint8); tot=m.sum()+gt.sum()
        ds.append(2*(m&gt).sum()/tot if tot>0 else 1.0)
    return (np.mean(ds), len(ds)) if ds else None

print("\n"+"="*58)
print("SEGMENTATION MODEL TESTED ON EACH CROP FOLDER")
print("="*58)
for d in sorted(glob.glob(os.path.join(D,"crops*"))):
    r=test_folder(d)
    if r: print("  "+os.path.basename(d).ljust(28)+"Dice "+format(r[0],".4f")+"   (n="+str(r[1])+")")
print("="*58)
print("the folder with the highest Dice is what the model was trained on")

CROP FOLDERS ON DISK:
  crops_clean512_calc         1866 images
  crops_clean512_calc_tightmask0 images
  crops_fixed_calc            1866 images
  crops_fixed_mass            1696 images
  crops_hires_calc            1866 images
  crops_tight512_calc         1866 images
  crops_v3                    3242 images
  crops_v4                    3242 images
  crops_v5                    3164 images
  crops_v6                    3566 images

CSV FILES:
  ATT_test_metrics.csv
  ATT_test_perimage.csv
  CV_calc_ALL_false_negatives.csv
  CV_calc_ALL_false_positives.csv
  FINAL_cv_calcification.csv
  FINAL_cv_mass.csv
  FINAL_operating_points.csv
  FINAL_test_metrics.csv
  FINAL_test_perimage.csv
  THESIS_final_metrics.csv
  THESIS_final_perimage.csv
  THESIS_perimage_with_metadata.csv
  ablation_plain_attnunet.csv
  ablation_plain_test_perimage.csv
  ablation_prep_aug.csv
  al_labeled_pool.csv
  al_unlabeled_pool.csv
  align_report.csv
  all_dataset_seg_summary.csv
  all_results_per_image.csv
 

In [3]:
import os, pandas as pd
D="/root/autodl-tmp/CBIS"
for f in ["seg_official_mass_test.csv","seg_official_calcification_test.csv",
          "seg_official_combined_test.csv","seg_cbis_mass_test.csv","seg_cbis_calc_test.csv",
          "test_results_attn_dueling.csv","THESIS_final_metrics.csv","FINAL_test_metrics.csv",
          "all_dataset_seg_summary.csv"]:
    p=os.path.join(D,f)
    if not os.path.exists(p): continue
    d=pd.read_csv(p)
    print("="*66); print(f+"   rows="+str(len(d))); print("  cols:",list(d.columns)[:12])
    num=[c for c in d.columns if d[c].dtype.kind in "fc"]
    if len(d)<=8: print(d.to_string(index=False))
    else:
        for c in num[:6]:
            print("    "+c.ljust(16)+"mean "+format(d[c].mean(),".4f")+"  median "+format(d[c].median(),".4f"))

seg_official_mass_test.csv   rows=378
  cols: ['abn', 'patient', 'dice', 'iou', 'precision', 'recall', 'subtlety', 'pathology']
    dice            mean 0.9225  median 0.9309
    iou             mean 0.8584  median 0.8707
    precision       mean 0.9028  median 0.9151
    recall          mean 0.9463  median 0.9589
seg_official_calcification_test.csv   rows=326
  cols: ['abn', 'patient', 'dice', 'iou', 'precision', 'recall', 'subtlety', 'pathology']
    dice            mean 0.8795  median 0.9052
    iou             mean 0.7938  median 0.8269
    precision       mean 0.8310  median 0.8653
    recall          mean 0.9460  median 0.9558
seg_official_combined_test.csv   rows=704
  cols: ['abn', 'patient', 'dice', 'iou', 'precision', 'recall', 'subtlety', 'pathology']
    dice            mean 0.9026  median 0.9213
    iou             mean 0.8285  median 0.8541
    precision       mean 0.8695  median 0.8955
    recall          mean 0.9461  median 0.9569
seg_cbis_mass_test.csv   rows=378
  col

In [4]:
import os, glob, pandas as pd
D="/root/autodl-tmp/CBIS"

def stems(folder):
    return set(os.path.basename(p).replace("_img.png","") for p in glob.glob(os.path.join(folder,"*_img.png")))

v6=stems(os.path.join(D,"crops_v6"))
fm=stems(os.path.join(D,"crops_fixed_mass"))
fc=stems(os.path.join(D,"crops_fixed_calc"))
print("crops_v6            "+str(len(v6)))
print("crops_fixed_mass    "+str(len(fm))+"   overlap with v6: "+str(len(fm&v6)))
print("crops_fixed_calc    "+str(len(fc))+"   overlap with v6: "+str(len(fc&v6)))

for f in ["seg_test_split.csv","seg_train_split.csv"]:
    p=os.path.join(D,f)
    if not os.path.exists(p): continue
    d=pd.read_csv(p)
    print("\n"+f+"   rows="+str(len(d))+"   cols:",list(d.columns)[:8])
    if "img" in d.columns:
        print("   points to:", d["img"].astype(str).str.split("/").str[-2].value_counts().head(3).to_dict())
    if "patient_id" in d.columns:
        segp=set(d.patient_id.astype(str))
        for name,csv in [("mass","cbis_mass_fixed.csv"),("calc","cbis_calc_fixed.csv")]:
            c=pd.read_csv(os.path.join(D,csv))
            cls=set(c.patient_id.astype(str))
            print("   patients shared with "+name+" classification: "+str(len(segp&cls))+" / "+str(len(segp)))

crops_v6            3566
crops_fixed_mass    1696   overlap with v6: 1696
crops_fixed_calc    1866   overlap with v6: 1866

seg_test_split.csv   rows=379   cols: ['patient_id', 'full_case_id', 'cropped_case_id', 'roi_case_id', 'pathology', 'label_name', 'abn_type', 'split']
   patients shared with mass classification: 201 / 202
   patients shared with calc classification: 22 / 202

seg_train_split.csv   rows=2433   cols: ['patient_id', 'full_case_id', 'cropped_case_id', 'roi_case_id', 'pathology', 'label_name', 'abn_type', 'split']
   patients shared with mass classification: 673 / 1192
   patients shared with calc classification: 592 / 1192
